<a href="https://colab.research.google.com/github/SergeiVKalinin/camm_hackathon/blob/k4my4r/docs/day_20_10092026/Day_20_llm_decision_making_Sergei.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color = "red">**Deciphering the Decision Logic of an AI Agent**</font>

- Can we infer what an AI agent values by observing the sequence of decisions it makes?

- In this challenge, GPT acts as an autonomous microscopy agent that repeatedly selects one measurement location from a set of candidate AFM image patches.

- We record its decisions and then attempt to reverse-engineer the hidden logic behind those choices using visual, statistical, and data-driven methods.

<font color = "blue" size = 6>**Goal**</font>: develop a method that best explains GPT’s behavior and predicts its future decisions on unseen candidate sets.

<a href="https://colab.research.google.com/github/kbarakati/camm_hackathon/blob/k4my4r/docs/day_20_10092026/llm_decision_making.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports and reproducibility

In [ ]:
! pip install -q igor2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from igor2 import binarywave
import pickle

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

import random
np.random.seed(42)


Download data

In [ ]:
! gdown --folder "https://drive.google.com/drive/folders/1Gy7PeI4EhmlNE5hLxvP9p9A4u026QQ_X?usp=sharing"

## load data

In [ ]:
fname = "/content/day_20_data/scan_main_0000.ibw"

ibw = binarywave.load(fname)

wave = ibw["wave"]

data = np.array(
    wave["wData"]
)

img_data = np.moveaxis(data,-1, 0)

channels = [
    "Height",
    "Amplitude1",
    "Amplitude2",
    "Phase1",
    "Phase2",
    "Frequency"
]

print("Channels:", channels)
print("Shape:", img_data.shape)
print("Mode: DART Mode")

In [ ]:
trace_files = [
    "/content/day_20_data/gpt_decision_trace_0.pkl",
    "/content/day_20_data/gpt_decision_trace_1.pkl",
    "/content/day_20_data/gpt_decision_trace_2.pkl",

    "/content/day_20_data/simulated_decision_trace_0.pkl",
    "/content/day_20_data/simulated_decision_trace_1.pkl",
    "/content/day_20_data/simulated_decision_trace_2.pkl",
    "/content/day_20_data/simulated_decision_trace_3.pkl",
    "/content/day_20_data/simulated_decision_trace_4.pkl",
    "/content/day_20_data/simulated_decision_trace_5.pkl",
    "/content/day_20_data/simulated_decision_trace_6.pkl",
    "/content/day_20_data/simulated_decision_trace_7.pkl",
]


traces = {}

for i, fname in enumerate(trace_files):

    with open(fname, "rb") as f:
        traces[i] = pickle.load(f)

Plot the AFM image

In [ ]:
channel_index = 2 #--> we want amplitude

img = img_data[channel_index].T
img = (img - img.min()) / (img.max() - img.min())

plt.figure(figsize=(6, 6))

plt.imshow(
    img,
    cmap="gray"
)

plt.axis("off")
plt.show()

Each trace contains multiple decision steps.

For every step:

- `patches` contains all candidate image patches available to the agent.
- `selected_candidate` gives the candidate selected by the agent.
- `selected_patch` contains the selected image patch.
- `position` gives the selected location in the original AFM image.
- `score` may be available for some traces.

In [ ]:
TRACE_ID = 4
STEP = 1

trace = traces[TRACE_ID]

step_data = trace[STEP - 1]

print("Selected candidate:",
      step_data["selected_candidate"])

print("Position:",
      step_data["position"])

print("Number of candidates:",
      len(step_data["patches"]))

if "score" in step_data:
    print("Score:",
          step_data["score"])

VISUALIZE ONE DECISION STEP

In [ ]:
patches = step_data["patches"]
selected = step_data["selected_candidate"]

n = len(patches)

ncols = 5
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(10, 2 * nrows)
)

axes = np.array(axes).reshape(-1)

for i, ax in enumerate(axes):

    if i >= n:
        ax.axis("off")
        continue

    ax.imshow(
        patches[i],
        cmap="gray"
    )

    if i == selected:

        ax.set_title(
            f"Candidate {i} ✓",
            color="red",
            weight="bold"
        )

        for spine in ax.spines.values():
            spine.set_edgecolor("red")
            spine.set_linewidth(3)

    else:

        ax.set_title(
            f"Candidate {i}"
        )

    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(
    f"Trace {TRACE_ID} — Step {STEP}"
)

plt.tight_layout()
plt.show()

TRACE MAP ON ORIGINAL IMAGE

In [ ]:
trace = traces[TRACE_ID]

steps = sorted(trace.keys())

xs = [
    trace[step]["position"][0]
    for step in steps
]

ys = [
    trace[step]["position"][1]
    for step in steps
]


fig, ax = plt.subplots(
    figsize=(7, 7)
)

ax.imshow(
    img,
    cmap="gray"
)


# ------------------------------------------------------------
# Plot all selected locations
# ------------------------------------------------------------

ax.scatter(
    xs,
    ys,
    s=55,
    color="cyan",
    edgecolors="black"
)


# ------------------------------------------------------------
# Label each point by decision step
# ------------------------------------------------------------

for step, x, y in zip(
    steps,
    xs,
    ys
):

    ax.text(
        x + 5,
        y + 5,
        str(step + 1),
        fontsize=9,
        color="yellow",
        weight="bold"
    )


ax.axis("off")

plt.show()

DECISION MAPS FOR 10 TRACES

In [ ]:
TRACE_IDS = list(traces.keys())[:10]

fig, axes = plt.subplots(
    2,
    5,
    figsize=(18, 7)
)

axes = axes.flatten()


for ax, trace_id in zip(axes, TRACE_IDS):

    trace = traces[trace_id]

    steps = sorted(trace.keys())

    xs = [
        trace[step]["position"][0]
        for step in steps
    ]

    ys = [
        trace[step]["position"][1]
        for step in steps
    ]


    # Original AFM image
    ax.imshow(
        img,
        cmap="gray"
    )


    # Selected locations
    ax.scatter(
        xs,
        ys,
        s=35,
        color="r",
        edgecolors="black"
    )

    ax.set_title(
        f"Trace {trace_id}",
        fontsize=12,
        weight="bold"
    )

    ax.axis("off")


plt.tight_layout()
plt.show()

# What Makes a Patch Get Selected?

Each decision step contains 20 candidate patches, but only one was selected.
The goal is to discover visual or quantitative characteristics that may explain the hidden objective.

### DEFINE YOUR HYPOTHESIZED REWARD

In [ ]:
def reward(patch):

    p = patch.astype(float)

    # --------------------------------------------------------
    # YOUR HYPOTHESIS HERE
    # --------------------------------------------------------

    score = np.mean(p)   # example only — modify this

    return score


### SCORE ALL PATCHES IN ONE STEP

In [ ]:
TRACE_ID = 6
STEP = 1

trace = traces[TRACE_ID]
step_data = trace[STEP - 1]

patches = step_data["patches"]
actual_selected = step_data["selected_candidate"]


# Score all 20 candidates
scores = np.array([
    reward(patch)
    for patch in patches
])


# Select highest-scoring candidate
predicted_selected = int(
    np.argmax(scores)
)

print(
    f"Actual selection:    Candidate {actual_selected}"
)

print(
    f"Predicted selection: Candidate {predicted_selected}"
)


### VISUALIZE PREDICTED vs ACTUAL CLEARLY

In [ ]:
fig, axes = plt.subplots(
    4,
    5,
    figsize=(10, 8)
)

for i, ax in enumerate(axes.flat):

    ax.imshow(
        patches[i],
        cmap="gray"
    )

    ax.set_title(
        f"Candidate {i}\nScore = {scores[i]:.2f}",
        fontsize=9
    )

    # --------------------------------------------------------
    # Highlight selections
    # --------------------------------------------------------

    if i == actual_selected and i == predicted_selected:

        for spine in ax.spines.values():
            spine.set_edgecolor("blue")
            spine.set_linewidth(5)

        ax.set_title(
            f"Candidate {i}\n"
            f"Score = {scores[i]:.2f}\n"
            f"ACTUAL + PREDICTED",
            fontsize=9,
            color="blue",
            weight="bold"
        )

    elif i == actual_selected:

        for spine in ax.spines.values():
            spine.set_edgecolor("green")
            spine.set_linewidth(5)

        ax.set_title(
            f"Candidate {i}\n"
            f"Score = {scores[i]:.2f}\n"
            f"ACTUAL",
            fontsize=9,
            color="green",
            weight="bold"
        )

    elif i == predicted_selected:

        for spine in ax.spines.values():
            spine.set_edgecolor("red")
            spine.set_linewidth(5)

        ax.set_title(
            f"Candidate {i}\n"
            f"Score = {scores[i]:.2f}\n"
            f"PREDICTED",
            fontsize=9,
            color="red",
            weight="bold"
        )

    ax.set_xticks([])
    ax.set_yticks([])


plt.tight_layout()
plt.show()

### BENCHMARK HYPOTHESIZED REWARD ACROSS ALL STEPS

In [ ]:
results = []

for step in sorted(trace.keys()):

    step_data = trace[step]

    patches = step_data["patches"]
    actual_selected = step_data["selected_candidate"]

    # Score all candidates in this step
    scores = np.array([
        reward(patch)
        for patch in patches
    ])

    # Predicted choice = highest score
    predicted_selected = int(
        np.argmax(scores)
    )

    results.append({
        "step": step + 1,
        "actual": actual_selected,
        "predicted": predicted_selected,
        "correct": int(
            predicted_selected == actual_selected
        )
    })


results = pd.DataFrame(results)

accuracy = results["correct"].mean()

print(f"Trace {TRACE_ID}")
print(f"Accuracy: {accuracy:.1%}")
print(f"Correct: {results['correct'].sum()} / {len(results)}")
print(f"Random baseline: {1 / len(trace[0]['patches']):.1%}")

results

# Sergei's solution


# Inferring a Global Linear Reward

## 1. Scientific Objective

The goal of this analysis is to reconstruct the decision-making logic of a human or AI agent from recorded image-selection trajectories.

In each decision, the agent is presented with a set of candidate microscopy image patches and selects one of them. The resulting dataset contains both the candidate images and the identity of the selected patch.

We assume that the decision-maker operates according to an unknown **reward function** that assigns a numerical preference to each candidate image. Images with higher rewards are more likely to be selected.

Our objective is to infer this reward function from the observed choices.

This is an example of **inverse preference learning**: rather than specifying a reward and predicting a decision, we observe decisions and reconstruct a reward that could have produced them.

The analysis has three objectives:

1. Identify measurable image characteristics that influence selections.
2. Construct an interpretable linear reward function from these characteristics.
3. Determine whether the inferred reward can predict decisions that were excluded from training.

The longer-term goal is to use the inferred reward as an objective for autonomous microscopy optimization, including Deep Kernel Learning (DKL) workflows.

---

## 2. Structure of the Experimental Data

The dataset consists of recorded decision trajectories from human or AI-agent image selections.

Each trajectory is a sequence of decision steps. At each step, the agent is presented with a collection of candidate image patches and selects one candidate.

For decision step $t$, define the candidate set:

$$
\mathcal{C}_t =
\left\{
I_{t,1},I_{t,2},\ldots,I_{t,K_t}
\right\},
$$

where $K_t$ is the number of candidate patches.

The observed selection is represented by the index

$$
a_t \in \{1,2,\ldots,K_t\}.
$$

Thus, the selected image is

$$
I_t^*=I_{t,a_t}.
$$

The available data for one decision can be represented as

$$
D_t =
\left(
\mathcal{C}_t,a_t
\right).
$$

A complete trajectory is

$$
\mathcal{T} =
\left\{
D_1,D_2,\ldots,D_T
\right\}.
$$

In the current notebook, the demonstrated traces contain 20 candidate patches per decision. The dataset includes GPT-generated decision trajectories and simulated decision trajectories.

The critical feature of these data is that we know not only which image was selected but also which alternatives were rejected.

Consequently, every decision provides relative preference information:

$$
I_{t,a_t}\succ I_{t,j},
\qquad j\neq a_t.
$$

Here, $\succ$ denotes an observed preference implied by the selection, rather than a guaranteed deterministic ordering of the underlying rewards.

Unlike ordinary supervised regression, the dataset does not necessarily contain a numerical reward assigned by the decision-maker to every image.

Instead, the reward must be reconstructed from the selections.

---

## 3. Representing Images Through Numerical Descriptors

A linear reward function requires a numerical representation of each image.

For each candidate image $I$, we construct a descriptor vector

$$
\mathbf{f}(I)=
\begin{bmatrix}
f_1(I)\\
f_2(I)\\
\vdots\\
f_M(I)
\end{bmatrix}.
$$

In this implementation, $M=16$ descriptors.

The descriptors are designed to capture complementary aspects of image appearance, including intensity distribution, spatial gradients, texture, and correlations.

### 3.1. Intensity descriptors

The first group characterizes the distribution of pixel intensities.

For an image containing $N$ pixels with intensities $I_p$, the mean intensity is

$$
\mu_I=\frac{1}{N}\sum_{p=1}^{N}I_p.
$$

The standard deviation is

$$
\sigma_I=
\sqrt{
\frac{1}{N}
\sum_{p=1}^{N}(I_p-\mu_I)^2
}.
$$

Additional intensity descriptors include:

- Fifth percentile of the intensity distribution.
- Ninety-fifth percentile.
- Difference between the 95th and 5th percentiles.
- Skewness of the intensity distribution.
- Kurtosis of the intensity distribution.

These descriptors capture brightness, contrast, asymmetry, and the presence of extreme intensity values.

They may be relevant if the decision-maker favors high-contrast structures, bright regions, or images containing localized features.

### 3.2. Spatial-gradient descriptors

The second group characterizes variations in the image intensity.

The image gradient is

$$
\nabla I=
\left(
\frac{\partial I}{\partial x},
\frac{\partial I}{\partial y}
\right).
$$

Its magnitude is

$$
G(x,y)=
\sqrt{
\left(\frac{\partial I}{\partial x}\right)^2+
\left(\frac{\partial I}{\partial y}\right)^2
}.
$$

The implemented descriptors include:

**Mean gradient magnitude**

$$
f_{\mathrm{grad}}=
\langle G(x,y)\rangle.
$$

**Gradient standard deviation**

$$
f_{\mathrm{grad,std}}=
\operatorname{std}(G).
$$

**Gradient energy**

$$
f_{\mathrm{grad,energy}}=
\langle G^2(x,y)\rangle.
$$

**Mean absolute Laplacian**

$$
f_{\mathrm{lap}}=
\left\langle
|\nabla^2 I|
\right\rangle.
$$

The numerical implementation calculates Sobel gradients and the Laplacian on intensity-normalized patches.

These descriptors reflect edges, boundaries, local intensity variations, and high-spatial-frequency structures.

They may distinguish smooth regions from regions containing domain walls, particles, defects, or other spatial heterogeneities.

However, gradients can also be enhanced by noise. Therefore, high gradient magnitude should not automatically be interpreted as high image quality or greater scientific interest.

### 3.3. Texture and spatial correlation descriptors

The remaining descriptors characterize spatial organization.

**Intensity entropy**

An intensity histogram with probabilities $p_b$ is used to calculate

$$
H=-\sum_b p_b\log p_b.
$$

The implementation uses 32 histogram bins and normalizes the result by $\log 32$.

Entropy describes the complexity of the intensity distribution, although it does not directly measure scientific information content.

**High-frequency Fourier power**

The image is Fourier-transformed:

$$
\tilde I(k_x,k_y)=
\mathcal{F}[I(x,y)].
$$

The high-frequency power fraction is

$$
P_{\mathrm{HF}}=
\frac{
\sum_{|\mathbf{k}|>k_c}|\tilde I(\mathbf{k})|^2
}{
\sum_{\mathbf{k}}|\tilde I(\mathbf{k})|^2
}.
$$

The code uses a normalized spatial-frequency threshold of $k_c=0.25$ cycles per pixel.

This descriptor quantifies the fraction of image power associated with relatively fine spatial structures.

**Edge anisotropy**

$$
A_{\mathrm{edge}}=
\langle |G_x|\rangle-
\langle |G_y|\rangle.
$$

This quantity captures directional differences in the image gradients.

**Horizontal and vertical correlations**

For normalized image intensity $\hat I$, the implemented nearest-neighbor correlations are

$$
C_x=
\left\langle
\hat I(x,y)\hat I(x+1,y)
\right\rangle
$$

and

$$
C_y=
\left\langle
\hat I(x,y)\hat I(x,y+1)
\right\rangle.
$$

These descriptors characterize local spatial continuity and the presence of correlated textures.

### 3.4. Complete descriptor vector

The final image representation contains 16 quantities:

| # | Descriptor | Interpretation |
|---|---|---|
| 1 | Mean intensity | Average signal |
| 2 | Intensity standard deviation | Overall intensity variation |
| 3 | 5th percentile | Lower intensity range |
| 4 | 95th percentile | Upper intensity range |
| 5 | Contrast range | Robust image contrast |
| 6 | Skewness | Intensity asymmetry |
| 7 | Kurtosis | Distribution tails |
| 8 | Mean gradient | Average spatial variation |
| 9 | Gradient standard deviation | Heterogeneity of gradients |
| 10 | Gradient energy | Strength of spatial variations |
| 11 | Absolute Laplacian | Local curvature and fine structure |
| 12 | Entropy | Intensity-distribution complexity |
| 13 | High-frequency power | Fine spatial structure |
| 14 | Edge anisotropy | Directional texture |
| 15 | Horizontal correlation | Spatial continuity |
| 16 | Vertical correlation | Spatial continuity |

These descriptors provide an initial, interpretable basis for reconstructing the decision-making rule.

They are not assumed to represent the complete scientific utility of an image. The purpose is to test how much of the observed behavior can be explained by relatively simple image characteristics.

---

## 4. Descriptor Standardization

The image descriptors have different physical scales and numerical ranges.

For example, mean intensity, entropy, and gradient energy cannot be compared directly using their unnormalized values.

Therefore, each descriptor is standardized:

$$
z_k(I)=
\frac{f_k(I)-\mu_k}{\sigma_k},
$$

where $\mu_k$ and $\sigma_k$ are the mean and standard deviation of descriptor $k$.

Importantly, these statistics are calculated **only from the training images**.

The same transformation is then applied to the validation images.

This prevents information from the hidden validation dataset from influencing the training procedure.

The standardized descriptor vector is

$$
\mathbf{z}(I)=
[z_1(I),z_2(I),\ldots,z_{16}(I)]^T.
$$

After standardization, the magnitude of a linear reward coefficient provides a more meaningful measure of that descriptor's contribution to the model.

However, correlated descriptors can share predictive importance, so individual coefficients should not automatically be interpreted as independent causal contributions.

---

## 5. Linear Reward Function

We assume that image preference can be approximated using a linear function of the standardized descriptors:

$$
\boxed{
R_{\mathbf w}(I)=
\mathbf w^T\mathbf z(I)
=
\sum_{k=1}^{16}w_kz_k(I)
}
$$

where

$$
\mathbf w=
[w_1,w_2,\ldots,w_{16}]^T
$$

is the vector of unknown reward coefficients.

Each coefficient describes the contribution of a particular image descriptor to the inferred reward.

For example:

- A positive coefficient for intensity contrast indicates that higher-contrast images tend to be preferred, conditional on the other descriptors.
- A positive coefficient for spatial gradients suggests a preference for images containing stronger local variations.
- A negative coefficient for high-frequency power could suggest avoidance of fine-scale noise or texture.
- A positive coefficient for spatial correlation could indicate a preference for coherent structures.

These interpretations must be assessed in the context of feature correlations and the actual images.

The absolute numerical scale of the reward is not known in advance.

We therefore learn a relative reward function whose values are meaningful primarily for comparing candidate images.

---

## 6. Probabilistic Model of the Selection Process

The decision-maker does not necessarily select the image with the highest reward deterministically.

Human choices can be inconsistent, and agent decisions may involve stochasticity or exploration.

We therefore describe selection using a probabilistic model.

For a candidate image $I_{t,i}$, define

$$
R_{t,i}=
\mathbf w^T\mathbf z(I_{t,i}).
$$

The probability that candidate $i$ is selected is modeled using the softmax function:

$$
\boxed{
P(a_t=i\mid\mathcal C_t,\mathbf w)=
\frac{\exp(R_{t,i})}
{\sum_{j=1}^{K_t}\exp(R_{t,j})}
}
$$

This is a multinomial logit, or conditional choice, model.

Images with higher reward receive higher selection probabilities.

For two candidates, the probability that candidate $i$ is chosen over candidate $j$ is

$$
P(i\succ j)=
\frac{1}
{1+\exp[-(R_i-R_j)]}.
$$

Thus, the probability of selecting an image depends on its reward relative to the alternatives.

A reward of 2 is not intrinsically good or bad. Its significance depends on whether the competing images have rewards of 0, 2, or 5.

The model therefore learns relative preferences rather than absolute quality scores.

---

## 7. Learning the Reward Coefficients

For every training decision, the observed selected candidate is known.

The likelihood of the complete collection of selections is

$$
\mathcal L(\mathbf w)=
\prod_{t\in\mathcal D_{\mathrm{train}}}
P(a_t\mid\mathcal C_t,\mathbf w).
$$

Taking the negative logarithm gives the optimization objective:

$$
-\log\mathcal L(\mathbf w)=
-\sum_{t\in\mathcal D_{\mathrm{train}}}
\log P(a_t\mid\mathcal C_t,\mathbf w).
$$

Substituting the softmax probabilities:

$$
-\log\mathcal L(\mathbf w)=
\sum_t
\left[
\log
\left(
\sum_{j=1}^{K_t}\exp(R_{t,j})
\right)
-R_{t,a_t}
\right].
$$

To reduce overfitting, we introduce an L2 regularization term:

$$
\boxed{
J(\mathbf w)=
\frac{1}{T_{\mathrm{train}}}
\sum_{t\in\mathcal D_{\mathrm{train}}}
\left[
\log\sum_{j=1}^{K_t}\exp(R_{t,j})
-R_{t,a_t}
\right]
+
\frac{\lambda}{2}\|\mathbf w\|_2^2
}
$$

In the current implementation,

$$
\lambda=0.1.
$$

This is an initial regularization choice rather than a parameter established by validation.

The optimal coefficients are found by minimizing

$$
\boxed{
\mathbf w^*=
\arg\min_{\mathbf w}J(\mathbf w).
}
$$

The optimization uses the L-BFGS-B algorithm.

For the softmax linear model with L2 regularization, the objective is convex in the coefficient vector, providing a well-defined optimization problem.

The gradient is calculated analytically.

For one decision, it is

$$
\nabla_{\mathbf w}J_t=
\sum_{j=1}^{K_t}
P(a_t=j\mid\mathcal C_t,\mathbf w)
\mathbf z(I_{t,j})
-
\mathbf z(I_{t,a_t}).
$$

This expression has an intuitive interpretation.

The optimization compares the descriptor vector of the image actually selected with the probability-weighted average descriptor vector of all candidates.

The coefficients are updated so that the selected image becomes more probable.

---

## 8. Hidden-Trace Validation: 80% Training and 20% Testing

The key question is whether the inferred reward reproduces decisions that were not used to construct it.

For each trajectory containing $T$ decisions, we divide the data into:

**Training dataset**

$$
\mathcal D_{\mathrm{train}}=
\{D_1,\ldots,D_{\lfloor0.8T\rfloor}\}.
$$

**Hidden validation dataset**

$$
\mathcal D_{\mathrm{test}}=
\{D_{\lfloor0.8T\rfloor+1},\ldots,D_T\}.
$$

The final 20% of each trajectory is excluded from model fitting.

Importantly, entire decisions are held out.

All candidate images belonging to one validation decision remain together in the hidden set.

This ensures that the model cannot learn directly from the actual selected image or the rejected candidates in that decision.

The reward coefficients are learned using only the earlier 80% of decisions.

The fitted reward function is then applied to every candidate image in both the training and validation datasets.

For every candidate in a hidden decision, we calculate

$$
\hat R_{t,i}=
{\mathbf w^*}^T
\mathbf z(I_{t,i}).
$$

The model predicts the selected candidate as

$$
\boxed{
\hat a_t=
\arg\max_i\hat R_{t,i}.
}
$$

We then compare $\hat a_t$ with the actual recorded selection $a_t$.

The training procedure does not access the hidden selection indices until validation.

This is a temporal validation scheme: earlier decisions are used to predict later ones.

It is more representative of sequential deployment than randomly withholding patches, although it does not eliminate possible dependence among nearby decisions.

---

## 9. Validation Metrics

Several complementary metrics are calculated.

### 9.1. Top-1 selection accuracy

The most direct measure is the fraction of held-out decisions in which the model predicts the actual selected image:

$$
\boxed{
\mathrm{Accuracy}=
\frac{1}{T_{\mathrm{test}}}
\sum_{t\in\mathcal D_{\mathrm{test}}}
\mathbb{1}(\hat a_t=a_t).
}
$$

For a decision with 20 candidate images, uniform random selection has an expected accuracy of

$$
P_{\mathrm{random}}=\frac{1}{20}=0.05.
$$

Thus, random selection provides a 5% reference level.

Accuracy substantially exceeding this level indicates that the inferred reward contains information about the observed choices.

However, high selection accuracy does not by itself establish that the learned model has recovered the decision-maker's actual internal reward.

### 9.2. Rank of the selected image

Exact agreement is a relatively stringent criterion.

For example, the actual selected image might receive the second-highest inferred reward rather than the highest reward.

Such a result is more informative than a selection ranked last.

We therefore calculate the rank of the actual selected image:

$$
r_t=
1+
\sum_{j=1}^{K_t}
\mathbb{1}
\left(
\hat R_{t,j}>
\hat R_{t,a_t}
\right).
$$

A rank of 1 means the actual selected image has the highest inferred reward, allowing tied rewards to share rank 1.

The mean rank is

$$
\overline r=
\frac{1}{T_{\mathrm{test}}}
\sum_t r_t.
$$

Lower mean rank indicates better agreement.

### 9.3. Mean reciprocal rank

We also calculate

$$
\boxed{
\mathrm{MRR}=
\frac{1}{T_{\mathrm{test}}}
\sum_t\frac{1}{r_t}.
}
$$

This metric gives greater weight to selections ranked near the top.

A perfect model has

$$
\mathrm{MRR}=1.
$$

### 9.4. Negative log-likelihood

The predicted probability assigned to the actual selection is

$$
p_t=
P(a_t\mid\mathcal C_t,\mathbf w^*).
$$

The average negative log-likelihood is

$$
\boxed{
\mathrm{NLL}=
-\frac{1}{T_{\mathrm{test}}}
\sum_t\log p_t.
}
$$

Lower values indicate that the model assigns higher probabilities to actual selections.

For uniform random selection among 20 candidates,

$$
\mathrm{NLL}_{\mathrm{random}}=\log 20
\approx 2.996.
$$

Unlike top-1 accuracy, negative log-likelihood evaluates the confidence of the predictions.

A model that assigns very low probability to the observed choice is penalized more strongly than one that makes an uncertain prediction.

### 9.5. Relative reward of the selected image

To compare rewards within decisions, we calculate the reward relative to the average candidate reward:

$$
\Delta R_{t,i}=
\hat R_{t,i}-
\frac{1}{K_t}
\sum_{j=1}^{K_t}\hat R_{t,j}.
$$

For the actual selection,

$$
\Delta R_t^*=
\hat R_{t,a_t}
-
\overline R_t.
$$

A positive value indicates that the selected image received above-average inferred reward.

This quantity is particularly useful for comparing decisions because it eliminates arbitrary additive offsets in the rewards within each candidate set.

---

## 10. Visualization of the Inferred Reward

The analysis generates several complementary visualizations.

### 10.1. Reward coefficient plot

The learned coefficients $w_k$ are plotted as a horizontal bar chart.

Positive coefficients indicate descriptors associated with greater inferred preference.

Negative coefficients indicate descriptors associated with lower inferred preference.

The relative magnitudes characterize the structure of the linear decision model.

This visualization addresses the scientific question:

**What measurable image properties appear to guide the decision-maker?**

### 10.2. Rank distribution for hidden decisions

A histogram shows the rank assigned to the actual selected image in each held-out decision.

A strong model should concentrate the distribution near rank 1.

A weak model will produce a broader distribution.

This provides a more detailed characterization of preference reconstruction than selection accuracy alone.

### 10.3. Candidate-patch reward visualization

For an individual hidden decision, all candidate image patches are displayed together.

Each image is annotated with its inferred reward:

$$
\hat R_{t,i}=
\sum_k w_k^*z_k(I_{t,i}).
$$

The actual selected patch and the predicted selected patch are highlighted.

This visualization allows direct comparison of image morphology and inferred reward.

It can reveal cases in which the learned model correctly reproduces the decision, as well as cases where it favors different structures.

The disagreement cases are particularly valuable because they identify potentially missing reward descriptors.

### 10.4. Actual versus predicted candidate index

The code also plots actual and predicted candidate indices.

Points on the diagonal correspond to correct selections.

This plot is a simple diagnostic of exact agreement. However, candidate indices are categorical identifiers, not continuous physical coordinates.

Therefore, distance from the diagonal does not measure how different two images are.

The rank-based metrics and patch-level visualization provide more meaningful assessments.

---

## 11. Interpretation of the Reconstructed Reward

The inferred reward should be interpreted as a model of **observed selection preferences**.

It is not necessarily identical to the internal reward or reasoning process used by the human or AI agent.

For example, if a selected image has high contrast, several possible explanations exist:

1. The decision-maker explicitly preferred high contrast.
2. High contrast correlated with the presence of scientifically interesting features.
3. High contrast correlated with an unmeasured experimental variable.
4. The decision-maker selected the image for an unrelated reason that happens to correlate with contrast.

A linear preference model identifies statistical associations between image descriptors and choices.

It does not automatically distinguish these explanations.

The learned reward is also conditioned on the candidate sets represented in the training data.

If the candidate-generation procedure changes substantially, the inferred reward may not generalize to the new candidate distribution.

### What can and cannot be validated

The present data contain observed selections but do not establish a numerical ground-truth reward for every candidate image.

Therefore, the analysis can test whether the inferred rewards reproduce observed decisions.

It cannot directly calculate a meaningful reward-value RMSE against the decision-maker's true latent reward without additional reward annotations or information about the original scoring rule.

This distinction is important: **predicting behavior and recovering the underlying reward are related but different objectives.**

---

## 12. Shared Reward Across Multiple Decision Trajectories

The current implementation combines decision data from all available trajectories and learns one shared coefficient vector:

$$
\mathbf w_{\mathrm{shared}}.
$$

This assumes that the observed decisions can be described, at least approximately, by a common preference function.

The assumption can be tested by calculating validation statistics separately for each trajectory.

Differences in performance may indicate that:

- Some trajectories contain more consistent decisions.
- The relevant image features differ between trajectories.
- Human and AI agents use different reward functions.
- The decision strategies depend on the stage of the experiment.
- A linear reward is insufficient for some types of decisions.

A natural extension is to fit separate models:

$$
R_{\mathrm{GPT}}(I)=
\mathbf w_{\mathrm{GPT}}^T\mathbf z(I)
$$

and

$$
R_{\mathrm{sim}}(I)=
\mathbf w_{\mathrm{sim}}^T\mathbf z(I).
$$

Comparing the coefficient vectors would help identify common and distinct selection preferences.

---

## 13. Limitations of the Linear Model

The linear model is intentionally simple.

It provides an interpretable baseline against which more sophisticated approaches can be compared.

However, several limitations should be recognized.

**First, image preferences may be nonlinear.**

For example, increasing contrast may improve preference up to a certain threshold, beyond which the effect saturates.

**Second, descriptors may interact.**

An image with strong gradients and low noise may be desirable, while strong gradients associated with noise may be undesirable.

A purely additive reward cannot fully capture this interaction.

**Third, decisions may depend on experimental history.**

A human or agent might select an image because it is novel relative to previous measurements, rather than because of any intrinsic property of the image.

The current model does not explicitly include decision history, uncertainty, or exploration objectives.

**Fourth, the descriptor space may be incomplete.**

Scientific interest can depend on specific domain boundaries, defects, particles, phase transitions, or other structures that are not adequately characterized by global image statistics.

Finally, repeated or strongly correlated images across the training and validation periods may make the predictive task easier without establishing generalization to new experiments.

These limitations motivate more advanced reward-learning approaches.

---

## 14. Connection to Deep Kernel Learning Optimization

The ultimate objective is to use the reconstructed reward to guide autonomous microscopy experiments.

Suppose the experimental parameter vector is

$$
\mathbf x=
(x_1,x_2,\ldots,x_d).
$$

The parameter vector might represent spatial coordinates, measurement conditions, or other controllable experimental variables.

An experiment performed at $\mathbf x$ generates an image patch:

$$
I=I(\mathbf x).
$$

The learned reward function assigns a value to this image:

$$
y(\mathbf x)=
R_{\mathbf w^*}(I(\mathbf x)).
$$

This converts the decision-preference problem into a scalar optimization problem.

We can then construct a DKL model:

$$
y(\mathbf x)
\sim
\mathcal{GP}
\left(
m(\mathbf x),
k_\psi(
f_\phi(\mathbf x),
f_\phi(\mathbf x')
)
\right),
$$

where $f_\phi$ is a learned representation of experimental parameters and $k_\psi$ is the Gaussian-process kernel.

The DKL model provides a predicted reward mean

$$
\mu_R(\mathbf x)
$$

and predictive uncertainty

$$
\sigma_R(\mathbf x).
$$

An acquisition function can use these quantities to determine the next experiment.

For example, an upper-confidence-bound strategy is

$$
\boxed{
A(\mathbf x)=
\mu_R(\mathbf x)+
\beta\sigma_R(\mathbf x).
}
$$

The next experimental condition is selected by

$$
\boxed{
\mathbf x_{t+1}
=
\arg\max_{\mathbf x}A(\mathbf x).
}
$$

This establishes a pathway from observed human or agent preferences to autonomous experimental optimization.

The first implementation can treat the learned reward as a fixed scalar label.

A more rigorous extension should also propagate uncertainty associated with reward inference, rather than assuming the inferred rewards are exact observations.

---

## 15. Summary

This analysis establishes a baseline methodology for reconstructing scientific decision-making from recorded microscopy image selections.

The workflow consists of the following stages:

1. Represent each candidate image using 16 interpretable numerical descriptors.
2. Standardize the descriptors using training data only.
3. Construct a linear reward function from the standardized descriptors.
4. Infer the reward coefficients by maximizing the likelihood of observed selections.
5. Exclude the final 20% of each decision trajectory from training.
6. Predict rewards for all candidate images in the hidden decisions.
7. Compare predicted selections with actual selections using accuracy, rank, reciprocal rank, and negative log-likelihood.
8. Analyze reward coefficients and disagreements to identify the characteristics associated with decision-making.
9. Use the learned reward as a potential objective for subsequent DKL optimization.

The central hypothesis is that at least part of the decision-making logic of a human or AI agent can be represented by a low-dimensional, interpretable function of image characteristics.

The experiment tests this hypothesis quantitatively.

If the inferred linear reward predicts previously unseen choices, it provides evidence that the selection behavior contains a systematic component captured by the descriptors.

If the model fails, the disagreement cases provide a starting point for identifying missing image characteristics, nonlinear dependencies, or history-dependent decision mechanisms.

In either case, the analysis provides a foundation for developing autonomous experimental systems that learn scientific objectives from observed decisions rather than requiring explicitly specified reward functions.


In [ ]:

# ================================================================
# ATHENA HACKATHON DAY 20
# INVERSE LINEAR REWARD LEARNING FROM IMAGE SELECTION TRACES
# 80% training / 20% hidden validation
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import sobel, laplace
from scipy.stats import skew, kurtosis
from scipy.special import logsumexp, softmax
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------
# 1. USE TRACES ALREADY LOADED IN THE ORIGINAL NOTEBOOK
# ---------------------------------------------------------------

assert "traces" in globals(), (
    "Run the original notebook's trace-loading cell first."
)

SEED = 42
np.random.seed(SEED)

# Original notebook stores traces in a dictionary.
TRACE_IDS = list(traces.keys())

print("Available traces:", TRACE_IDS)


# ---------------------------------------------------------------
# 2. EXTRACT INTERPRETABLE IMAGE DESCRIPTORS
# ---------------------------------------------------------------

FEATURE_NAMES = [
    "mean_intensity",
    "intensity_std",
    "percentile_05",
    "percentile_95",
    "contrast_range",
    "skewness",
    "kurtosis",
    "mean_gradient",
    "gradient_std",
    "gradient_energy",
    "laplacian_abs",
    "entropy",
    "high_frequency_power",
    "edge_anisotropy",
    "horizontal_correlation",
    "vertical_correlation",
]


def patch_features(patch):

    a = np.asarray(patch, dtype=float).squeeze()

    if a.ndim != 2:
        raise ValueError(f"Expected 2D patch; got {a.shape}")

    valid = np.isfinite(a)

    if not valid.any():
        return np.zeros(len(FEATURE_NAMES))

    v = a[valid]
    a = np.where(valid, a, np.median(v))

    mu = a.mean()
    sd = a.std()

    norm = (a - mu) / (sd + 1e-9)

    gx = sobel(norm, axis=1) / 8
    gy = sobel(norm, axis=0) / 8

    grad = np.hypot(gx, gy)

    power = np.abs(
        np.fft.fftshift(np.fft.fft2(norm))
    ) ** 2

    fy = np.fft.fftshift(
        np.fft.fftfreq(a.shape[0])
    )[:, None]

    fx = np.fft.fftshift(
        np.fft.fftfreq(a.shape[1])
    )[None, :]

    radius = np.sqrt(fx**2 + fy**2)

    high_power = (
        power[radius > 0.25].sum()
        / (power.sum() + 1e-9)
    )

    hist, _ = np.histogram(
        norm, bins=32, range=(-4, 4)
    )

    probs = hist[hist > 0] / max(hist.sum(), 1)

    entropy = (
        -np.sum(probs * np.log(probs)) / np.log(32)
    )

    horizontal_corr = (
        np.mean(norm[:, 1:] * norm[:, :-1])
        if a.shape[1] > 1 else 0
    )

    vertical_corr = (
        np.mean(norm[1:, :] * norm[:-1, :])
        if a.shape[0] > 1 else 0
    )

    return np.array([
        mu,
        sd,
        np.percentile(a, 5),
        np.percentile(a, 95),
        np.percentile(a, 95) - np.percentile(a, 5),
        skew(v) if sd > 1e-9 else 0,
        kurtosis(v) if sd > 1e-9 else 0,
        grad.mean(),
        grad.std(),
        np.mean(grad**2),
        np.mean(np.abs(laplace(norm))),
        entropy,
        high_power,
        np.mean(np.abs(gx)) - np.mean(np.abs(gy)),
        horizontal_corr,
        vertical_corr
    ], dtype=float)


# ---------------------------------------------------------------
# 3. BUILD DECISION DATASET
# ---------------------------------------------------------------

records = []

for trace_id in TRACE_IDS:

    trace = traces[trace_id]

    steps = (
        sorted(trace.keys())
        if isinstance(trace, dict)
        else range(len(trace))
    )

    for step in steps:

        entry = trace[step]

        patches = entry["patches"]
        selected = int(entry["selected_candidate"])

        X = np.vstack([
            patch_features(p)
            for p in patches
        ])

        records.append({
            "trace_id": trace_id,
            "step": step,
            "patches": patches,
            "X": X,
            "selected": selected,
            "n_candidates": len(patches)
        })

print("\nDataset:")
print("Decisions:", len(records))
print("Candidate patches:", sum(r["n_candidates"] for r in records))


# ---------------------------------------------------------------
# 4. HIDE FINAL 20% OF EACH TRACE
# ---------------------------------------------------------------

train_idx = []
test_idx = []

for trace_id in TRACE_IDS:

    ids = [
        i for i, r in enumerate(records)
        if r["trace_id"] == trace_id
    ]

    ids = sorted(ids, key=lambda i: records[i]["step"])

    cutoff = int(np.floor(0.8 * len(ids)))

    train_idx.extend(ids[:cutoff])
    test_idx.extend(ids[cutoff:])

train_idx = np.asarray(train_idx, dtype=int)
test_idx = np.asarray(test_idx, dtype=int)

print("\nTrain decisions:", len(train_idx))
print("Hidden decisions:", len(test_idx))


# ---------------------------------------------------------------
# 5. STANDARDIZE DESCRIPTORS USING TRAINING DATA ONLY
# ---------------------------------------------------------------

scaler = StandardScaler()

X_train_all = np.vstack([
    records[i]["X"] for i in train_idx
])

scaler.fit(X_train_all)

for r in records:

    r["Z"] = np.nan_to_num(
        scaler.transform(r["X"]),
        nan=0,
        posinf=0,
        neginf=0
    )


# ---------------------------------------------------------------
# 6. FIT LINEAR REWARD USING CHOICE LIKELIHOOD
#
# R(I) = w dot z(I)
#
# P(select i) = exp(R_i) / sum_j exp(R_j)
# ---------------------------------------------------------------

ALPHA = 0.1  # L2 regularization

def objective(w):

    loss = 0.0
    gradient = np.zeros_like(w)

    for idx in train_idx:

        rec = records[idx]

        Z = rec["Z"]
        y = rec["selected"]

        rewards = Z @ w
        probabilities = softmax(rewards)

        loss += logsumexp(rewards) - rewards[y]

        gradient += Z.T @ probabilities - Z[y]

    loss /= len(train_idx)
    gradient /= len(train_idx)

    loss += 0.5 * ALPHA * np.dot(w, w)
    gradient += ALPHA * w

    return loss, gradient


result = minimize(
    objective,
    np.zeros(len(FEATURE_NAMES)),
    jac=True,
    method="L-BFGS-B",
    options={"maxiter": 500}
)

weights = result.x

print("\nOptimization converged:", result.success)

print("\nLearned reward coefficients:")

coef_df = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "weight": weights
})

print(
    coef_df.reindex(
        coef_df.weight.abs().sort_values(
            ascending=False
        ).index
    ).to_string(index=False)
)


# ---------------------------------------------------------------
# 7. CALCULATE REWARDS FOR ALL PATCHES
# ---------------------------------------------------------------

decision_rows = []
patch_rows = []

test_set = set(test_idx.tolist())

for idx, rec in enumerate(records):

    Z = rec["Z"]

    rewards = Z @ weights
    probabilities = softmax(rewards)

    actual = rec["selected"]
    predicted = int(np.argmax(rewards))

    rank = 1 + np.sum(rewards > rewards[actual])

    split = "test" if idx in test_set else "train"

    rec["rewards"] = rewards
    rec["probabilities"] = probabilities
    rec["predicted"] = predicted

    decision_rows.append({
        "trace": rec["trace_id"],
        "step": rec["step"],
        "split": split,
        "actual": actual,
        "predicted": predicted,
        "correct": int(actual == predicted),
        "actual_rank": rank,
        "actual_probability": probabilities[actual],
        "selected_reward": rewards[actual],
        "relative_selected_reward":
            rewards[actual] - rewards.mean(),
        "n_candidates": len(rewards)
    })

    for j, reward_value in enumerate(rewards):

        patch_rows.append({
            "trace": rec["trace_id"],
            "step": rec["step"],
            "split": split,
            "candidate": j,
            "selected": int(j == actual),
            "reward": reward_value,
            "relative_reward":
                reward_value - rewards.mean(),
            "selection_probability": probabilities[j]
        })


decision_df = pd.DataFrame(decision_rows)
patch_df = pd.DataFrame(patch_rows)


# ---------------------------------------------------------------
# 8. VALIDATION STATISTICS
# ---------------------------------------------------------------

print("\n========== VALIDATION ==========")

for split in ["train", "test"]:

    df = decision_df[decision_df.split == split]

    accuracy = df.correct.mean()
    mean_rank = df.actual_rank.mean()
    mrr = (1 / df.actual_rank).mean()

    chance = (1 / df.n_candidates).mean()

    log_loss = -np.log(
        np.maximum(df.actual_probability, 1e-15)
    ).mean()

    print(f"\n{split.upper()}")
    print(f"Decisions: {len(df)}")
    print(f"Top-1 accuracy: {accuracy:.3f}")
    print(f"Random accuracy: {chance:.3f}")
    print(f"Mean selected rank: {mean_rank:.3f}")
    print(f"Mean reciprocal rank: {mrr:.3f}")
    print(f"Negative log likelihood: {log_loss:.3f}")


# ---------------------------------------------------------------
# 9. PLOT LEARNED REWARD COEFFICIENTS
# ---------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 6))

order = np.argsort(np.abs(weights))

ax.barh(
    np.array(FEATURE_NAMES)[order],
    weights[order]
)

ax.axvline(0, color="gray", linewidth=1)

ax.set_xlabel("Standardized reward coefficient")
ax.set_title("Inferred Linear Reward Function")

plt.tight_layout()
plt.show()


# ---------------------------------------------------------------
# 10. HELD-OUT RANK DISTRIBUTION
# ---------------------------------------------------------------

test_df = decision_df[
    decision_df.split == "test"
]

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(
    test_df.actual_rank,
    bins=np.arange(
        0.5,
        test_df.n_candidates.max() + 1.5
    )
)

ax.set_xlabel(
    "Rank of actual selected patch (1 = highest reward)"
)

ax.set_ylabel("Number of held-out decisions")
ax.set_title("Hidden 20%: Actual Selection Rank")

plt.tight_layout()
plt.show()


# ---------------------------------------------------------------
# 11. VISUALIZE ALL 20 PATCHES FOR A HIDDEN DECISION
# ---------------------------------------------------------------

EXAMPLE_INDEX = int(test_idx[0])

rec = records[EXAMPLE_INDEX]

patches = rec["patches"]
rewards = rec["rewards"]

actual = rec["selected"]
predicted = rec["predicted"]

n = len(patches)
ncols = 5
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(15, 3 * nrows),
    squeeze=False
)

for j, ax in enumerate(axes.flat):

    if j >= n:
        ax.axis("off")
        continue

    ax.imshow(patches[j], cmap="gray")

    title = f"#{j}: R={rewards[j]:+.2f}"

    if j == actual:
        title += "\nACTUAL"

    if j == predicted:
        title += "\nPREDICTED"

    color = (
        "green" if j == actual == predicted
        else "red" if j == actual
        else "blue" if j == predicted
        else "black"
    )

    ax.set_title(title, color=color, fontsize=10)

    if j in (actual, predicted):
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(color)
            spine.set_linewidth(3)

    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(
    f"Held-out decision: trace {rec['trace_id']}, "
    f"step {rec['step']}\n"
    f"Actual={actual}, Predicted={predicted}",
    fontsize=15
)

plt.tight_layout()
plt.show()


# ---------------------------------------------------------------
# 12. COMPARISON OF ACTUAL VERSUS INFERRED CHOICE
# ---------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(
    test_df.actual,
    test_df.predicted,
    alpha=0.55
)

max_candidate = int(
    max(test_df.actual.max(), test_df.predicted.max())
)

ax.plot(
    [0, max_candidate],
    [0, max_candidate],
    "--",
    color="gray"
)

ax.set_xlabel("Actual selected candidate index")
ax.set_ylabel("Predicted selected candidate index")
ax.set_title("Held-out Decisions: Actual vs Predicted")

plt.tight_layout()
plt.show()


# ---------------------------------------------------------------
# 13. OPTIONAL: PER-TRACE VALIDATION
# ---------------------------------------------------------------

print("\n========== VALIDATION BY TRACE ==========")

trace_metrics = (
    test_df.groupby("trace")
    .agg(
        decisions=("correct", "count"),
        accuracy=("correct", "mean"),
        mean_rank=("actual_rank", "mean"),
        mean_probability=("actual_probability", "mean")
    )
)

print(trace_metrics.to_string())


# ---------------------------------------------------------------
# 14. EXPORT RESULTS
# ---------------------------------------------------------------

from pathlib import Path

OUTPUT_DIR = Path("day20_linear_reward_results")
OUTPUT_DIR.mkdir(exist_ok=True)

coef_df.to_csv(
    OUTPUT_DIR / "reward_coefficients.csv",
    index=False
)

decision_df.to_csv(
    OUTPUT_DIR / "decision_predictions.csv",
    index=False
)

patch_df.to_csv(
    OUTPUT_DIR / "all_patch_rewards.csv",
    index=False
)

trace_metrics.to_csv(
    OUTPUT_DIR / "per_trace_validation.csv"
)

print("\nResults saved in:", OUTPUT_DIR.resolve())

print("\nNOTE:")
print("The learned rewards are relative preference scores.")
print("Actual numerical rewards are not available from")
print("selected_candidate labels alone.")
print("Validation therefore measures reproduction of")
print("held-out decisions, not reward-value RMSE.")


# Trace by trace

Now, reward function is different for each trace. So let's do it trace by trace.

The objective is to reconstruct the reward function underlying a sequence of image-selection decisions made by a human or AI agent.

At each step, the decision-maker is presented with multiple candidate microscopy image patches and selects one of them. We assume that the selected patch is preferred according to an underlying decision-making criterion, which can be approximated by a mathematical reward function.

Rather than defining this reward function explicitly, we seek to **infer it from recorded selections**.

The central questions are:

1. Can the decision-making logic be represented by an interpretable linear combination of image descriptors?
2. Can the inferred reward function reproduce selections that were not used during training?
3. How does the inferred reward assigned to the selected patch compare with the rewards assigned to all rejected candidates?
4. What can the observed selections tell us about the underlying reward function, and what remains fundamentally uncertain?

This is a form of **inverse preference learning**, in which the observed decisions are used to reconstruct a latent utility or reward function.

### Important: Each trace is analyzed independently

Each recorded trajectory was generated using a potentially different decision-making logic.

Therefore, it is inappropriate to combine all traces into a single training dataset without first establishing that they share a common decision function.

In this implementation, we select one trace using:

`TRACE_ID = 0`

and reconstruct a reward function specifically for that trace.

Changing `TRACE_ID` allows the procedure to be repeated independently for other trajectories.

---

## 2. Structure of the Decision Data

A decision trace consists of a sequence of steps indexed by $t$.

At each step, the decision-maker is presented with a collection of candidate image patches:

$$
\mathcal{C}_t =
\left\{
I_{t,1}, I_{t,2}, \ldots, I_{t,K_t}
\right\},
$$

where $K_t$ is the number of candidate patches.

In the demonstrated dataset, there are typically 20 candidates per decision.

The decision-maker selects one patch, identified by an index $a_t$:

$$
I_t^* = I_{t,a_t}.
$$

The available information for each step includes:

- The complete set of candidate image patches.
- The index of the selected candidate.
- The decision-step identifier.
- Other trace-specific metadata, when available.

The notebook stores these data in a structure containing fields such as `patches` and `selected_candidate`.

The observed decision gives a set of relative preferences:

$$
I_{t,a_t} \succ I_{t,j},
\qquad j \neq a_t.
$$

Thus, each decision with 20 candidates provides 19 selected-versus-rejected comparisons.

These comparisons do not establish numerical reward values, but they constrain the possible reward functions.

---

## 3. Representation of Image Patches

To construct an interpretable reward function, each image patch is represented by a vector of numerical descriptors.

For image patch $I$, we define

$$
\mathbf{f}(I) =
\left[
f_1(I),f_2(I),\ldots,f_{16}(I)
\right]^T.
$$

The implementation uses 16 descriptors organized into three categories.

### 3.1. Intensity statistics

The first seven descriptors characterize the intensity distribution:

1. Mean pixel intensity.
2. Standard deviation of pixel intensity.
3. Fifth intensity percentile.
4. Ninety-fifth intensity percentile.
5. Difference between the 95th and 5th percentiles.
6. Skewness.
7. Kurtosis.

For example, the mean intensity is

$$
\mu_I = \frac{1}{N}\sum_{p=1}^{N} I_p,
$$

and the standard deviation is

$$
\sigma_I =
\sqrt{
\frac{1}{N}
\sum_{p=1}^{N}(I_p-\mu_I)^2
}.
$$

These descriptors capture brightness, contrast, asymmetry, and the presence of extreme intensity values.

### 3.2. Gradient and curvature descriptors

The next four descriptors characterize spatial variations:

8. Mean gradient magnitude.
9. Standard deviation of gradient magnitude.
10. Mean squared gradient magnitude.
11. Mean absolute Laplacian.

The gradient magnitude is

$$
G(x,y) =
\sqrt{
\left(\frac{\partial I}{\partial x}\right)^2
+
\left(\frac{\partial I}{\partial y}\right)^2
}.
$$

The corresponding features quantify edges, domain boundaries, morphological changes, and spatial heterogeneity.

Gradients are calculated using Sobel operators on locally standardized images.

### 3.3. Texture and correlation descriptors

The final five descriptors are:

12. Histogram entropy.
13. High-spatial-frequency Fourier power fraction.
14. Edge anisotropy.
15. Horizontal nearest-neighbor correlation.
16. Vertical nearest-neighbor correlation.

Entropy is calculated from a 32-bin intensity histogram:

$$
H = -\frac{1}{\log 32}\sum_b p_b \log p_b.
$$

The high-frequency power fraction is

$$
P_{\mathrm{HF}} =
\frac{
\sum_{|\mathbf{k}|>k_c}|\tilde I(\mathbf{k})|^2
}{
\sum_{\mathbf{k}}|\tilde I(\mathbf{k})|^2
},
$$

where $k_c=0.25$ cycles per pixel in the implementation.

These descriptors characterize texture complexity, fine spatial structure, directional variations, and local correlations.

The descriptor set is intentionally simple and interpretable.

It provides a baseline for testing whether decision-making can be explained using conventional image characteristics.

---

## 4. Standardization of the Descriptors

Because the descriptors have different numerical scales, they are standardized before fitting the reward model.

For descriptor $k$,

$$
z_k(I) =
\frac{f_k(I)-\mu_k}{\sigma_k}.
$$

The mean $\mu_k$ and standard deviation $\sigma_k$ are calculated using **training patches only**.

The same normalization is subsequently applied to the hidden validation patches.

This prevents information from the validation dataset from influencing model construction.

After standardization, each patch is represented by

$$
\mathbf{z}(I) =
[z_1(I),z_2(I),\ldots,z_{16}(I)]^T.
$$

---

## 5. Linear Reward Model

The unknown reward function is approximated as a linear combination of standardized image descriptors:

$$
\boxed{
R_{\mathbf{w}}(I) =
\mathbf{w}^T\mathbf{z}(I)
=
\sum_{k=1}^{16} w_k z_k(I)
}
$$

where $\mathbf{w}$ is the vector of reward coefficients.

Each coefficient represents the contribution of one descriptor to the inferred preference.

For example:

- A positive contrast coefficient suggests preference for higher-contrast images.
- A positive gradient coefficient suggests preference for spatially varying structures.
- A negative high-frequency-power coefficient could indicate avoidance of fine-scale noise.
- A positive spatial-correlation coefficient suggests preference for coherent image structures.

These interpretations are conditional on the other descriptors and do not establish causality.

The objective of training is to determine the coefficients that best explain the recorded selections.

---

## 6. Probabilistic Model of the Decision Process

We assume that the probability of selecting a candidate increases with its reward.

For candidate $i$ at decision step $t$:

$$
R_{t,i} =
\mathbf{w}^T\mathbf{z}(I_{t,i}).
$$

The selection probability is modeled using a softmax function:

$$
\boxed{
P(a_t=i \mid \mathcal{C}_t,\mathbf{w}) =
\frac{
\exp(R_{t,i})
}{
\sum_{j=1}^{K_t}\exp(R_{t,j})
}
}
$$

This formulation allows for stochastic decisions.

The candidate with the highest reward has the highest predicted selection probability, but alternatives can still be selected.

The probability depends on **relative rewards within the candidate set**, not on their absolute values.

For two candidates:

$$
P(i \succ j) =
\frac{1}{
1+\exp[-(R_i-R_j)]
}.
$$

This equation shows that larger reward differences correspond to stronger predicted preferences.

---

## 7. Inferring Reward Coefficients from Observed Selections

The model is fitted by maximizing the likelihood of the recorded selections.

For a training dataset containing $T_{\mathrm{train}}$ decisions, the negative log-likelihood is

$$
\mathcal{L}(\mathbf{w}) =
-\sum_{t=1}^{T_{\mathrm{train}}}
\log P(a_t\mid\mathcal{C}_t,\mathbf{w}).
$$

Substituting the softmax formulation gives

$$
\mathcal{L}(\mathbf{w}) =
\sum_{t=1}^{T_{\mathrm{train}}}
\left[
\log
\left(
\sum_{j=1}^{K_t}\exp(R_{t,j})
\right)
-
R_{t,a_t}
\right].
$$

We introduce L2 regularization to reduce overfitting:

$$
\boxed{
J(\mathbf{w}) =
\frac{\mathcal{L}(\mathbf{w})}{T_{\mathrm{train}}}
+
\frac{\lambda}{2}\|\mathbf{w}\|_2^2
}
$$

The implementation uses

$$
\lambda=0.1.
$$

The optimal coefficients are

$$
\mathbf{w}^* =
\arg\min_{\mathbf{w}} J(\mathbf{w}).
$$

Optimization is performed using L-BFGS-B with an analytical gradient.

The model learns the coefficients that increase the probability of the observed selections relative to the rejected alternatives.

---

## 8. Training and Hidden Validation

Each trace is divided into two consecutive segments.

The first 80% of decisions are used for training:

$$
\mathcal{D}_{\mathrm{train}} =
\{D_1,\ldots,D_{\lfloor0.8T\rfloor}\}.
$$

The final 20% are withheld:

$$
\mathcal{D}_{\mathrm{test}} =
\{D_{\lfloor0.8T\rfloor+1},\ldots,D_T\}.
$$

The hidden decisions are not used to fit the model or standardize the descriptors.

Importantly, entire decisions are held out.

All candidate patches belonging to a hidden decision remain in the test set.

After fitting the model on the training decisions, the inferred reward is calculated for **every candidate patch**, including all patches in the hidden decisions.

This produces a complete reward vector for each decision.

---

## 9. Reconstructed Reward Distribution for Every Decision

For a decision containing $K_t$ patches, the inferred reward vector is

$$
\boxed{
\hat{\mathbf{R}}_t =
\begin{bmatrix}
\hat R_{t,1}\\
\hat R_{t,2}\\
\vdots\\
\hat R_{t,K_t}
\end{bmatrix}
}
$$

where

$$
\hat R_{t,i} =
{\mathbf{w}^*}^T\mathbf{z}(I_{t,i}).
$$

We also calculate the corresponding probabilities:

$$
\hat p_{t,i} =
\frac{\exp(\hat R_{t,i})}
{\sum_{j=1}^{K_t}\exp(\hat R_{t,j})}.
$$

The predicted selected candidate is

$$
\hat a_t = \arg\max_i \hat R_{t,i}.
$$

Unlike simply comparing selected indices, the complete reward vector allows us to examine how the actual choice compares with every alternative.

For example, the actual selected patch may have the second-highest reward, only slightly below the predicted maximum.

Alternatively, it may have a much lower inferred reward than several rejected patches.

These two situations indicate different degrees of disagreement between the reconstructed reward and observed behavior.

---

## 10. Evaluation Metrics

### 10.1. Top-1 accuracy

The simplest measure is whether the highest-reward patch matches the actual selection:

$$
\mathrm{Accuracy} =
\frac{1}{T_{\mathrm{test}}}
\sum_t
\mathbb{1}(\hat a_t=a_t).
$$

For 20 equally likely candidates, the random-selection baseline is 5%.

However, top-1 accuracy provides only a binary assessment.

It does not measure whether an incorrect prediction was nearly correct or strongly inconsistent with the recorded choice.

We therefore use additional reward-based metrics.

### 10.2. Rank of the actually selected patch

The inferred rank of the actual selected patch is

$$
\boxed{
r_t =
1+
\sum_j
\mathbb{1}
\left(
\hat R_{t,j}>\hat R_{t,a_t}
\right)
}
$$

A rank of 1 indicates that the actual selection received the highest inferred reward.

Rank 2 indicates that one alternative received a higher reward.

The mean rank over the hidden decisions summarizes how close actual selections are to the top of the inferred preference ordering.

### 10.3. Pairwise concordance

The most direct reward-ranking measure compares the selected patch against every rejected alternative.

Define

$$
\boxed{
C_t =
\frac{1}{K_t-1}
\sum_{j\neq a_t}
\left[
\mathbb{1}(\hat R_{t,a_t}>\hat R_{t,j})
+
\frac12\mathbb{1}(\hat R_{t,a_t}=\hat R_{t,j})
\right]
}
$$

Interpretation:

- $C_t=1$: the actual selection has a higher inferred reward than every rejected patch.
- $C_t=0.5$: the selected patch wins approximately half of the comparisons, with ties receiving half credit.
- $C_t=0$: every rejected candidate has a higher inferred reward.

The mean concordance over hidden decisions is

$$
\overline C =
\frac{1}{T_{\mathrm{test}}}
\sum_t C_t.
$$

This provides a graded measure of how well the inferred reward ranking agrees with actual decisions.

### 10.4. Reward margin

We calculate the difference between the inferred reward of the actual selection and the highest-reward rejected candidate:

$$
\boxed{
\Delta R_t =
\hat R_{t,a_t}-
\max_{j\neq a_t}\hat R_{t,j}
}
$$

A positive margin indicates that the selected patch has the highest inferred reward.

A negative margin indicates that the model favors another candidate.

The magnitude measures how strongly the linear model prefers one candidate over the other.

However, the numerical size of the margin depends on the inferred reward scale and should not be interpreted as an independently measured physical quantity.

### 10.5. Negative log-likelihood

The probability assigned to the actual selected patch is

$$
p_t =
P(a_t\mid\mathcal{C}_t,\mathbf{w}^*).
$$

The average negative log-likelihood is

$$
\boxed{
\mathrm{NLL} =
-\frac{1}{T_{\mathrm{test}}}
\sum_t\log p_t.
}
$$

Lower values indicate better probabilistic prediction.

This metric penalizes a model when it assigns very low probability to the observed choice.

### 10.6. Relative selected reward

The average inferred reward across the candidate set is

$$
\overline R_t =
\frac{1}{K_t}
\sum_{j=1}^{K_t}\hat R_{t,j}.
$$

The relative reward of the actual selection is

$$
\boxed{
\Delta R_t^{\mathrm{relative}} =
\hat R_{t,a_t}-\overline R_t.
}
$$

A positive value indicates that the actual selection receives an above-average inferred reward.

Relative rewards are useful because additive offsets common to all candidates in a decision do not affect the selection probabilities.

---

## 11. Visualization of the Hidden Decisions

For several held-out decisions, the code produces two complementary visualizations.

### 11.1. Reward bar plot

The first visualization displays the inferred reward for every candidate patch.

The horizontal axis is the candidate index.

The vertical axis is the inferred reward.

The actual selected candidate is highlighted with a green outline, while the predicted candidate is hatched.

This plot directly shows:

- The inferred reward distribution.
- The position of the actual selection in that distribution.
- The model's preferred patch.
- The reward difference between actual and predicted selections.

### 11.2. Image-patch montage

The second visualization displays all candidate image patches.

Each patch is annotated with its inferred reward.

The actual selection is outlined, and the predicted choice is identified in its title.

The purpose is to connect numerical reward values to image morphology.

For example, we may find that high inferred rewards are consistently associated with domain boundaries, high contrast, or spatially complex structures.

Conversely, disagreement cases may reveal that the actual decision-maker favored characteristics absent from the current descriptor set.

---

## 12. Direct Comparison with Ground-Truth Reward Values

The selection labels establish which candidate was selected, but they do not necessarily provide the numerical reward assigned to every candidate.

Therefore, the principal validation is a comparison between observed selections and inferred reward rankings.

However, if the trajectory contains an independently defined ground-truth reward array, the code can additionally compare actual and inferred rewards.

The implementation checks for fields named:

- `candidate_scores`
- `candidate_rewards`

A valid array must contain one finite numerical value for each candidate patch.

If such an array exists, the code computes Spearman rank correlation between the actual and inferred reward vectors for held-out decisions.

For one decision:

$$
\rho_t =
\operatorname{Spearman}
\left(
\mathbf R_t^{\mathrm{true}},
\hat{\mathbf R}_t
\right).
$$

The mean correlation across hidden decisions provides a measure of reward-ranking recovery.

A high correlation indicates that the learned model reproduces much of the original reward ordering, not merely the identity of the selected patch.

A direct numerical reward RMSE would require both the actual reward vector and a justified calibration of reward scales.

**A single optional scalar score is not sufficient to validate rewards for all candidate patches.**

---

## 13. What Can Be Inferred from Selection Data?

Each observed decision constrains the possible reward function.

Under deterministic reward maximization, the selected patch must have reward at least as large as every rejected patch:

$$
R_{\mathbf w}(I_{t,a_t})
\geq
R_{\mathbf w}(I_{t,j}).
$$

For a linear reward function:

$$
\boxed{
\mathbf w^T
\left[
\mathbf z(I_{t,a_t})-
\mathbf z(I_{t,j})
\right]
\geq0.
}
$$

Every comparison therefore imposes a linear inequality on the unknown coefficient vector.

A sequence of decisions constrains the feasible region of possible reward coefficients.

This suggests an important extension: rather than inferring only a single optimal coefficient vector, one could characterize the entire set of coefficient vectors consistent with the observed selections.

That feasible set could then be used to estimate bounds on the rewards assigned to individual patches.

The current implementation does not calculate these identification bounds.

Instead, it fits one regularized probabilistic reward model and reports its inferred reward values.

Thus, the numerical rewards should be interpreted as model-dependent preference scores, rather than uniquely determined physical quantities.

---

## 14. Main Limitations

### 14.1. Reward scale is not directly observable

Selections identify relative preferences rather than absolute reward values.

Different numerical reward functions can reproduce the same ranking.

The softmax likelihood and regularization establish a particular reward scale for the fitted model, but that scale is not independently measured.

### 14.2. Linear models may be insufficient

The actual decision function may involve nonlinear interactions.

For example, the decision-maker may prefer strong contrast only when the image also has low noise.

A linear model cannot generally represent such conditional preferences.

### 14.3. The reward may depend on experimental history

A patch may be scientifically interesting at the beginning of an experiment but redundant after similar regions have been measured.

The current model treats reward as a function of image descriptors alone:

$$
R(I)=R(\mathbf z(I)).
$$

A history-dependent decision model would instead be

$$
R_t(I)=R(\mathbf z(I),\mathcal H_t),
$$

where $\mathcal H_t$ represents previous observations and decisions.

### 14.4. Correlated descriptors can obscure interpretation

Several descriptors may capture similar image characteristics.

For example, intensity standard deviation and contrast range may be strongly correlated.

As a result, the model may distribute reward contributions across correlated features.

Individual coefficients should therefore be interpreted cautiously.

### 14.5. Selection accuracy does not prove reward recovery

Two different reward functions can select the same preferred image.

Consequently, reproducing observed selections does not establish that the original decision-making logic has been uniquely recovered.

Independent ground-truth reward information, additional comparisons, or controlled counterfactual tests would provide stronger validation.

### 14.6. Validation is limited to the available candidate distribution

The model is trained using candidate patches presented in the selected trajectory.

It does not automatically establish reliable predictions for arbitrary patches elsewhere in the full microscopy image.

Applying the model to the entire image requires computing descriptors for additional patches and evaluating whether these patches fall within the feature distribution represented during training.

---

## 15. Output Files

The code exports two CSV files for each analyzed trace.

### Patch-level rewards

`trace_<TRACE_ID>_patch_rewards.csv`

Each row corresponds to an individual candidate patch and contains:

- Decision step.
- Training or testing designation.
- Candidate index.
- Whether the candidate was selected.
- Inferred reward.
- Reward relative to the candidate-set mean.
- Predicted selection probability.

This file contains the complete reconstructed reward distribution for all recorded candidate patches.

### Decision-level validation

`trace_<TRACE_ID>_validation.csv`

Each row corresponds to one decision and contains:

- Actual selected candidate.
- Predicted selected candidate.
- Exact selection agreement.
- Rank of the actual selection.
- Pairwise concordance.
- Reward margin.
- Probability assigned to the actual selection.
- Relative reward of the actual selection.

Together, these files support statistical analysis of reward reconstruction and decision prediction.

---

## 16. Conclusions and Next Steps

The present implementation provides an interpretable baseline for inverse reward learning from microscopy decision traces.

The essential methodological distinction is that **each trace is treated as an independent realization of a particular decision-making logic**.

For each trace, we learn a linear reward function from the first 80% of observed decisions and evaluate it using the hidden final 20%.

Rather than relying exclusively on whether the predicted candidate index matches the recorded choice, we calculate the inferred reward of every candidate image and characterize the agreement of the complete reward ranking with the observed selection.

The analysis therefore provides three levels of information:

1. **Behavioral agreement:** How accurately does the model reproduce selections?
2. **Reward-ranking agreement:** How highly does the model rank the actually selected patches relative to the rejected alternatives?
3. **Descriptor interpretation:** Which image characteristics contribute most strongly to the inferred decision function?

The next methodological extension should move beyond a single best-fit reward vector to estimate the **range of possible reward functions consistent with the observed preferences**.

These bounds could then be propagated to individual image patches, providing minimum and maximum plausible rewards rather than only a point estimate.

Finally, the inferred reward function can be evaluated across overlapping patches covering the original microscopy image, producing spatial maps of predicted preference.

This would connect inverse decision learning with the subsequent DKL optimization workflow, allowing autonomous experiments to use reward criteria inferred from previous human or agent decisions.


In [ ]:
# ATHENA DAY 20: fit ONE trace, validate all candidate-patch rewards
# Run after the original notebook has created `traces`.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import sobel, laplace
from scipy.special import logsumexp, softmax
from scipy.optimize import minimize
from scipy.stats import skew, kurtosis, spearmanr
from sklearn.preprocessing import StandardScaler

TRACE_ID = 0                 # CHANGE: 0..10 in original notebook
TRAIN_FRACTION = 0.80
L2 = 0.10
DISPLAY_TEST_DECISIONS = 3

FEATURE_NAMES = [
    'mean', 'std', 'p05', 'p95', 'p95_minus_p05', 'skew', 'kurtosis',
    'mean_gradient', 'gradient_std', 'gradient_energy', 'abs_laplacian',
    'entropy', 'high_frequency_power', 'edge_anisotropy',
    'horizontal_correlation', 'vertical_correlation'
]

def features(patch):
    a = np.asarray(patch, dtype=float).squeeze()
    if a.ndim != 2:
        raise ValueError(f'Patch is not 2D: {a.shape}')
    good = a[np.isfinite(a)]
    if len(good) == 0:
        return np.zeros(16)
    a = np.where(np.isfinite(a), a, np.median(good))
    sd = a.std()
    b = (a-a.mean())/(sd+1e-9)
    gx, gy = sobel(b, axis=1)/8, sobel(b, axis=0)/8
    g = np.hypot(gx,gy)
    F = np.abs(np.fft.fftshift(np.fft.fft2(b)))**2
    fx, fy = np.fft.fftshift(np.fft.fftfreq(a.shape[1])), np.fft.fftshift(np.fft.fftfreq(a.shape[0]))
    hf = (fx[None,:]**2 + fy[:,None]**2) > 0.25**2
    hist, _ = np.histogram(b, bins=32, range=(-4,4))
    p = hist[hist > 0]/max(1, hist.sum())
    corr_x = np.mean(b[:,1:]*b[:,:-1]) if a.shape[1] > 1 else 0.
    corr_y = np.mean(b[1:,:]*b[:-1,:]) if a.shape[0] > 1 else 0.
    vals = [a.mean(),sd,np.percentile(a,5),np.percentile(a,95),
            np.percentile(a,95)-np.percentile(a,5),
            skew(good) if sd>1e-9 else 0., kurtosis(good) if sd>1e-9 else 0.,
            g.mean(),g.std(),np.mean(g*g),np.mean(np.abs(laplace(b))),
            -np.sum(p*np.log(p))/np.log(32),F[hf].sum()/(F.sum()+1e-9),
            np.mean(np.abs(gx))-np.mean(np.abs(gy)),corr_x,corr_y]
    return np.nan_to_num(np.array(vals, dtype=float))

assert 'traces' in globals(), 'Run the notebook trace-loading cells first.'
trace = traces[TRACE_ID]
step_ids = sorted(trace.keys()) if isinstance(trace,dict) else list(range(len(trace)))
D=[]
for s in step_ids:
    item=trace[s]
    patches=list(item['patches'])
    actual=int(item['selected_candidate'])
    assert 0<=actual<len(patches),f'Invalid selected candidate at {s}'
    D.append(dict(step=s,patches=patches,selected=actual,
                  X=np.vstack([features(p) for p in patches]),raw=item))
n=len(D); ntrain=int(np.floor(n*TRAIN_FRACTION))
assert ntrain>0 and ntrain<n,'Trace too short for train/test split.'
scaler=StandardScaler().fit(np.vstack([d['X'] for d in D[:ntrain]]))
for d in D: d['Z']=np.nan_to_num(scaler.transform(d['X']))

# Each decision contributes ONE multinomial-choice observation.
# Its entire candidate set stays together during splitting.
def loss_grad(w):
    loss=0.; grad=np.zeros_like(w)
    for d in D[:ntrain]:
        Z=d['Z']; scores=Z@w; probs=softmax(scores)
        loss+=logsumexp(scores)-scores[d['selected']]
        grad+=Z.T@probs-Z[d['selected']]
    return (loss/ntrain + 0.5*L2*np.dot(w,w), grad/ntrain+L2*w)
fit=minimize(loss_grad,np.zeros(16),jac=True,method='L-BFGS-B')
w=fit.x
print(f'Trace {TRACE_ID}: {n} decisions; first {ntrain} train, last {n-ntrain} hidden')
print('Optimization:',fit.message)
print(pd.DataFrame({'feature':FEATURE_NAMES,'weight':w}).sort_values('weight',key=lambda x:x.abs(),ascending=False).to_string(index=False))

rows=[]; choices=[]; true_vectors=[]; predicted_vectors=[]
for t,d in enumerate(D):
    r=d['Z']@w; prob=softmax(r); a=d['selected']; pred=int(r.argmax())
    alternatives=np.delete(r,a)
    margin=float(r[a]-alternatives.max())
    concordance=float(np.mean(r[a]>alternatives)+0.5*np.mean(r[a]==alternatives))
    rank=int(1+np.sum(r>r[a]))
    split='train' if t<ntrain else 'test'
    d.update(rewards=r,probabilities=prob,predicted=pred)
    choices.append(dict(step=d['step'],split=split,actual=a,predicted=pred,
        correct=int(a==pred),rank=rank,concordance=concordance,
        margin=margin,chosen_probability=float(prob[a]),
        selected_relative_reward=float(r[a]-r.mean())))
    for j,score in enumerate(r):
        rows.append(dict(step=d['step'],split=split,candidate=j,
            chosen=int(j==a),inferred_reward=float(score),
            centered_reward=float(score-r.mean()),
            inferred_choice_probability=float(prob[j])))
    # A documented full per-candidate ground-truth reward array permits numerical comparison.
    actual_scores=d['raw'].get('candidate_scores',None)
    if actual_scores is None:
        actual_scores=d['raw'].get('candidate_rewards',None)
    if actual_scores is not None:
        v=np.asarray(actual_scores,dtype=float).ravel()
        if len(v)==len(r) and np.all(np.isfinite(v)):
            if split=='test': true_vectors.append(v); predicted_vectors.append(r)

reward_table=pd.DataFrame(rows)
choice_table=pd.DataFrame(choices)
test=choice_table.query('split == "test"')
print('\nHELD-OUT RESULTS (single trace)')
print('Top-1 accuracy:',round(test.correct.mean(),3))
print('Mean selected rank:',round(test['rank'].mean(),3))
print('Mean pairwise concordance:',round(test.concordance.mean(),3))
print('Mean reward margin:',round(test.margin.mean(),3))
print('Mean choice NLL:',round(-np.log(test.chosen_probability.clip(lower=1e-15)).mean(),3))
print('Random top-1 baseline:',round(np.mean([1/len(d['patches']) for d in D[ntrain:]]),3))

if true_vectors:
    correlations=[]
    for true,guess in zip(true_vectors,predicted_vectors):
        rho=spearmanr(true,guess).statistic
        if np.isfinite(rho): correlations.append(rho)
    print('\nTRUE CANDIDATE REWARD VECTORS PRESENT:')
    print('Mean held-out within-decision Spearman rho:',np.mean(correlations) if correlations else 'undefined')
    print('Note: absolute reward RMSE requires a justified shared reward scale/calibration.')
else:
    print('\nNo documented full per-candidate actual reward vectors found.')
    print('Selections identify relative preferences, not ground-truth numerical rewards.')
    print('An optional scalar `score` cannot validate all 20 candidate rewards.')

fig,axs=plt.subplots(1,3,figsize=(17,4))
order=np.argsort(np.abs(w))
axs[0].barh(np.array(FEATURE_NAMES)[order],w[order]); axs[0].set_title('Linear reward coefficients')
axs[1].hist(test['rank'],bins=np.arange(.5,max(len(d['patches']) for d in D)+1.5)); axs[1].set_title('Selected patch rank (held out)'); axs[1].set_xlabel('1 = highest reward')
axs[2].plot(test.step,test.margin,'o-'); axs[2].axhline(0,color='gray',ls='--'); axs[2].set_title('Chosen vs best rejected reward'); axs[2].set_xlabel('Decision step'); axs[2].set_ylabel('Reward margin')
plt.tight_layout();plt.show()

for d in D[ntrain:ntrain+DISPLAY_TEST_DECISIONS]:
    r=d['rewards']; a=d['selected']; pred=d['predicted']; K=len(r)
    fig,ax=plt.subplots(figsize=(12,3.5))
    bars=ax.bar(np.arange(K),r)
    bars[a].set_edgecolor('green');bars[a].set_linewidth(3)
    bars[pred].set_hatch('///')
    ax.set_xlabel('Candidate patch index');ax.set_ylabel('Inferred reward')
    ax.set_title(f'Trace {TRACE_ID}, held-out step {d["step"]}: chosen={a} (green border), predicted={pred} (hatched)')
    plt.tight_layout();plt.show()
    cols=5; rows_n=int(np.ceil(K/cols))
    fig,axes=plt.subplots(rows_n,cols,figsize=(14,2.9*rows_n),squeeze=False)
    for j,axis in enumerate(axes.ravel()):
        if j>=K:axis.axis('off');continue
        axis.imshow(d['patches'][j],cmap='gray')
        axis.set_title(f'{j} | R={r[j]:+.2f}' + (' | ACTUAL' if j==a else '') + (' | PRED' if j==pred else ''),fontsize=8)
        axis.set_xticks([]);axis.set_yticks([])
        if j==a:
            for spine in axis.spines.values():spine.set_visible(True);spine.set_edgecolor('green');spine.set_linewidth(3)
    fig.suptitle(f'Trace {TRACE_ID} — hidden decision {d["step"]}: every candidate scored')
    plt.tight_layout();plt.show()

reward_table.to_csv(f'trace_{TRACE_ID}_patch_rewards.csv',index=False)
choice_table.to_csv(f'trace_{TRACE_ID}_validation.csv',index=False)
print(f'Exported trace_{TRACE_ID}_patch_rewards.csv and trace_{TRACE_ID}_validation.csv')



# Linear Reward Inference and Spatial Reward Mapping

### 1. Scientific objective

In the previous step, we inferred a linear reward function from the decisions recorded in one experimental trajectory.

Each decision consisted of a set of candidate microscopy image patches and a single selected patch. The objective was to identify a linear combination of image descriptors that explained the observed choices.

The resulting model was

$$
R_{\mathbf w}(I)=\mathbf w^T\mathbf z(I),
$$

where $\mathbf z(I)$ contains 16 standardized image descriptors.

The previous implementation estimated one optimal coefficient vector $\mathbf w^*$ and used it to assign rewards to each candidate patch.

However, a single reward vector does not capture the ambiguity inherent in learning rewards from decisions.

The current analysis extends this framework in two directions:

1. Determine the range of reward coefficients consistent with the observed decisions under explicitly defined constraints.
2. Propagate the resulting range of possible rewards to every recorded candidate patch and to a grid of patches covering the original AFM image.

The goal is to reconstruct not only a preferred reward function but also characterize how strongly the available choices constrain it.

---

### 2. Preference inequalities

Consider decision step $t$, with candidate images

$$
\mathcal C_t=\{I_{t,1},\ldots,I_{t,K_t}\}.
$$

The decision-maker selected candidate $a_t$.

Assuming deterministic maximization of an unknown reward, this choice implies

$$
R(I_{t,a_t})\geq R(I_{t,j}),
\qquad j\neq a_t.
$$

For a linear reward,

$$
R(I)=\mathbf w^T\mathbf z(I),
$$

the corresponding inequalities are

$$
\boxed{
\mathbf w^T
\left[
\mathbf z(I_{t,a_t})-
\mathbf z(I_{t,j})
\right]\geq0.
}
$$

Each decision involving 20 candidates therefore provides 19 preference inequalities.

The inequalities constrain the possible coefficient vectors.

If the decision-maker consistently selects images with certain combinations of descriptors, the accumulated constraints restrict the range of reward functions capable of explaining the choices.

However, stochastic decisions and incomplete descriptors can produce contradictory inequalities.

We therefore use a relaxed formulation.

---

### 3. Relaxed preference constraints

Define the difference vector for a selected-versus-rejected comparison:

$$
\mathbf d_{t,j}=
\mathbf z(I_{t,a_t})-\mathbf z(I_{t,j}).
$$

The ideal inequality is

$$
\mathbf d_{t,j}^T\mathbf w\geq0.
$$

We relax it to

$$
\boxed{
\mathbf d_{t,j}^T\mathbf w
\geq -s_{t,j}
}
$$

where $s_{t,j}\geq0$ is an allowed preference violation.

The previous model produced fitted coefficients $\mathbf w^*$.

We calculate the violation under those coefficients and define

$$
s_{t,j}
=
\max\left(
0,-\mathbf d_{t,j}^T\mathbf w^*
\right)+\epsilon.
$$

Here, $\epsilon$ is an additional tolerance.

The default is

$$
\epsilon=0.10.
$$

This construction ensures that the previous fitted model remains feasible.

It also permits nearby coefficient vectors to violate some preferences by limited amounts.

These tolerances are modeling choices, not quantities independently measured from the decision-maker.

Only the training decisions contribute constraints. The hidden final 20% remain excluded.

---

### 4. Bounding the coefficient space

Preference inequalities alone do not establish the numerical scale of the reward.

If one coefficient vector satisfies a set of homogeneous preference inequalities, multiplying it by a positive constant preserves the rankings.

Consequently, the feasible region may be unbounded.

To make minimum and maximum reward calculations well-defined, we impose bounds on each coefficient around the previously fitted value.

For coefficient $k$:

$$
w_k^*-\delta_k
\leq w_k
\leq w_k^*+\delta_k.
$$

The implementation defines

$$
\boxed{
\delta_k=
\max\left(
\delta_{\min},
\alpha|w_k^*|
\right).
}
$$

The initial parameters are

$$
\alpha=0.50,
\qquad
\delta_{\min}=0.25.
$$

Thus, every coefficient can vary around its fitted value, with a minimum allowed variation of 0.25.

The resulting feasible set is

$$
\boxed{
\mathcal W=
\left\{
\mathbf w:
\begin{array}{l}
\mathbf d_{t,j}^T\mathbf w\geq-s_{t,j},\\
w_k^*-\delta_k\leq w_k\leq w_k^*+\delta_k
\end{array}
\right\}.
}
$$

This is a convex polytope: the intersection of linear half-spaces and a bounded box.

**Interpretation:** The polytope represents a family of coefficient vectors allowed by the selected tolerance and coefficient limits.

It is not the uniquely identified set of true rewards, nor is it a Bayesian posterior or frequentist confidence region.

The width of this region depends on the chosen constraints.

---

### 5. Minimum and maximum reward of an individual patch

For a patch with standardized descriptor vector $\mathbf z(I)$, the previously fitted reward is

$$
R_{\mathrm{fit}}(I)
=
{\mathbf w^*}^T\mathbf z(I).
$$

We now seek the smallest and largest rewards permitted by the feasible region:

$$
\boxed{
R_{\min}(I)=
\min_{\mathbf w\in\mathcal W}
\mathbf w^T\mathbf z(I)
}
$$

and

$$
\boxed{
R_{\max}(I)=
\max_{\mathbf w\in\mathcal W}
\mathbf w^T\mathbf z(I).
}
$$

Since the objective and constraints are linear, both problems can be solved by linear programming.

We use the HiGHS optimization algorithm through `scipy.optimize.linprog`.

The resulting three quantities are

$$
R_{\min}(I)
\leq R_{\mathrm{fit}}(I)
\leq R_{\max}(I).
$$

The interval width is

$$
\boxed{
\Delta R(I)=R_{\max}(I)-R_{\min}(I).
}
$$

A narrow interval indicates that the imposed constraints permit relatively little variation in the reward of that patch.

A wide interval indicates greater ambiguity within the allowed coefficient region.

The interval should not be interpreted as a probability interval.

---

### 6. Propagation to all recorded candidate patches

The first application calculates reward intervals for every candidate image in the trace.

For a decision with 20 candidate patches, the model produces

$$
\mathbf R_{\min,t}
=
[R_{\min}(I_{t,1}),\ldots,R_{\min}(I_{t,20})],
$$

$$
\mathbf R_{\mathrm{fit},t}
=
[R_{\mathrm{fit}}(I_{t,1}),\ldots,R_{\mathrm{fit}}(I_{t,20})],
$$

and

$$
\mathbf R_{\max,t}
=
[R_{\max}(I_{t,1}),\ldots,R_{\max}(I_{t,20})].
$$

This is done for both training and hidden decisions.

For the hidden decisions, the constraints remain fixed at values inferred exclusively from training data.

Thus, the validation examines how well the family of feasible reward functions generalizes to decisions excluded from the inference procedure.

#### Visualization 1: Candidate reward intervals

For each selected hidden decision, we plot all candidate rewards.

Each candidate is represented by:

- Its fitted reward.
- Its minimum feasible reward.
- Its maximum feasible reward.

The actual selected candidate is highlighted.

The candidate with the highest fitted reward is also marked.

The plot reveals whether the actual selection has high inferred reward and whether competing candidates possess strongly overlapping reward intervals.

#### Visualization 2: Patch montage

All candidate images are displayed in a grid.

The title of each patch contains

$$
R_{\min}/R_{\mathrm{fit}}/R_{\max}.
$$

This visualization allows the reward ranges to be related directly to structural characteristics of the microscopy images.

It is especially useful for examining failed predictions and identifying descriptors that may be missing from the model.

---

### 7. Robust preference comparisons

The interval formulation allows a more nuanced comparison than selecting the single highest-reward candidate.

Consider the actual selected candidate $a_t$ and a rejected candidate $j$.

If

$$
R_{\min}(I_{t,a_t})
>
R_{\max}(I_{t,j}),
$$

then the actual selection is guaranteed to receive a higher reward than candidate $j$ for every coefficient vector in the feasible region.

This is a sufficient condition for a robust preference.

Conversely, if

$$
R_{\max}(I_{t,a_t})
<
R_{\min}(I_{t,j}),
$$

then every admissible reward function ranks candidate $j$ above the actual selection.

If the intervals overlap, the result is inconclusive using these individual interval bounds.

The implementation reports conservative bounds on the number of rejected candidates that the actual selection can outrank.

These comparisons provide additional information beyond top-1 accuracy and selected-patch rank.

An important mathematical limitation is that individual reward intervals are optimized separately. Their endpoints can correspond to different coefficient vectors.

Therefore, overlapping intervals do not prove that the ranking can reverse under a single admissible coefficient vector.

An exact ranking-feasibility test would directly optimize the pairwise reward difference over the common feasible region.

---

### 8. Extending the reward model to the full AFM image

The original notebook loads the complete AFM image into the variable `img`.

The recorded decision traces contain candidate image patches drawn from the experiment.

Once a reward function has been inferred, it can be applied to additional patches extracted from the original image.

Let $I_{x,y}$ denote a patch centered at spatial position $(x,y)$.

The fitted spatial reward field is

$$
\boxed{
R_{\mathrm{fit}}(x,y)=
{\mathbf w^*}^T\mathbf z(I_{x,y}).
}
$$

The corresponding minimum and maximum reward fields are

$$
R_{\min}(x,y)=
\min_{\mathbf w\in\mathcal W}
\mathbf w^T\mathbf z(I_{x,y}),
$$

$$
R_{\max}(x,y)=
\max_{\mathbf w\in\mathcal W}
\mathbf w^T\mathbf z(I_{x,y}).
$$

The spatial reward-range field is

$$
\boxed{
\Delta R(x,y)=
R_{\max}(x,y)-R_{\min}(x,y).
}
$$

These fields describe how the inferred reward and its model-dependent ambiguity vary across the image.

---

### 9. Spatial sliding-window implementation

The code uses the same patch dimensions as the recorded decision candidates.

It places patches on a regular spatial grid across the full AFM image.

The initial grid spacing is

`SPATIAL_STRIDE = 16`

pixels.

For every valid grid location:

1. Extract the corresponding image patch.
2. Calculate the same 16 descriptors.
3. Standardize them using the original training-data scaler.
4. Calculate the fitted reward.
5. Solve two linear programs to calculate its feasible minimum and maximum rewards.
6. Store the reward interval and spatial coordinates.

Only positions where the full patch fits inside the image are evaluated.

The results are represented as a spatial grid of reward estimates.

Reducing the stride increases spatial resolution but also increases computation time.

For example, decreasing the stride from 16 to 8 pixels can approximately quadruple the number of evaluated patches in a sufficiently large image.

---

### 10. Spatial visualization

The code produces six panels.

#### Panel A: Original AFM image

Displays the original microscopy image for spatial reference.

#### Panel B: Fitted reward field

$$
R_{\mathrm{fit}}(x,y).
$$

Shows the spatial distribution predicted by the previously learned linear model.

Regions with high values are those favored by the inferred reward function.

#### Panel C: Minimum feasible reward

$$
R_{\min}(x,y).
$$

Shows the smallest reward assigned to each patch by any coefficient vector satisfying the specified constraints.

#### Panel D: Maximum feasible reward

$$
R_{\max}(x,y).
$$

Shows the largest allowed reward for each patch.

#### Panel E: Reward-range width

$$
\Delta R(x,y).
$$

Characterizes the sensitivity of the predicted reward to variations in the admissible coefficient vector.

Regions with large values have less tightly constrained rewards under the chosen coefficient bounds.

#### Panel F: Distance from training patches

Measures how different each spatial patch is from the descriptors represented in the training set.

This provides an approximate indication of extrapolation beyond the original decision dataset.

---

### 11. Out-of-distribution analysis

A critical issue arises when the reward function is evaluated at image locations that were never presented to the decision-maker.

The inferred reward may assign high values to patches with descriptor combinations absent from training.

These predictions are extrapolations.

To identify such cases, we calculate the distance from a spatial patch to its nearest training patch in standardized descriptor space.

For spatial descriptor vector $\mathbf z(x,y)$:

$$
\boxed{
d_{\mathrm{OOD}}(x,y)=
\min_{i\in\mathcal D_{\mathrm{train}}}
\|\mathbf z(x,y)-\mathbf z_i\|_2.
}
$$

The implementation compares this distance with a threshold derived from nearest-neighbor distances among the training patches.

Locations above the threshold are flagged as potentially out of distribution.

This is a heuristic diagnostic rather than a calibrated measure of predictive uncertainty.

Its purpose is to warn when the reward model is being applied to patches substantially unlike those used to infer it.

Importantly, the coefficient-based reward intervals do not necessarily become wider in out-of-distribution regions.

Coefficient ambiguity and distributional extrapolation are different sources of uncertainty.

---

### 12. Interpretation and limitations

The new framework provides substantially more information than a single best-fit reward model.

However, several limitations remain.

**Reward scale:** Selection rankings do not uniquely determine numerical rewards. The coefficient bounds introduce a scale and must be treated as part of the modeling assumptions.

**Constraint tolerances:** Relaxation parameters determine which inconsistent or stochastic decisions are tolerated. Different choices produce different feasible regions.

**Reward intervals:** The calculated intervals are extrema over the specified coefficient polytope, not statistical confidence intervals.

**Independent interval extrema:** Minimum and maximum rewards at different spatial locations may correspond to different coefficient vectors. Therefore, the collection of minimum rewards is not necessarily one physically realizable reward map, and the same applies to the maximum rewards.

**Linear approximation:** Nonlinear selection criteria and interactions between image characteristics may not be represented adequately.

**Sequential decision-making:** The reward is assumed to depend only on the current patch. Novelty, previous measurements, and evolving experimental objectives are not explicitly included.

**Extrapolation:** Spatial patches not represented in candidate sets may have unreliable reward estimates despite receiving finite numerical values.

These limitations should be considered when interpreting reconstructed reward maps.

---

### 13. Output data

The implementation exports the following results for the selected trace.

**Recorded candidate rewards**

`recorded_patch_reward_bounds.csv`

Contains the fitted, minimum, and maximum reward for every recorded candidate patch.

**Decision-level validation**

`decision_bounds_validation.csv`

Contains selected and predicted candidate indices, selected-patch rank, reward bounds, and conservative preference comparisons.

**Full-image spatial reward grid**

`whole_image_reward_grid.csv`

Contains spatial coordinates, fitted reward, minimum reward, maximum reward, reward-range width, and out-of-distribution diagnostics.

**NumPy reward arrays**

`whole_image_reward_maps.npz`

Stores the spatial reward maps as arrays for subsequent image analysis or DKL optimization.

---

### 14. Connection to subsequent DKL optimization

The spatial reward reconstruction creates an inferred scalar objective over image locations:

$$
(x,y)\longrightarrow R_{\mathrm{fit}}(x,y).
$$

This can be used to train a Deep Kernel Learning surrogate that predicts reward at unmeasured positions.

The DKL model would represent

$$
R(x,y)\sim
\mathcal{GP}
\left(
m(x,y),
k_\theta(f_\phi(x,y),f_\phi(x',y'))
\right).
$$

A conventional acquisition function could then optimize expected reward while accounting for the DKL model's predictive uncertainty.

However, an important distinction must be maintained between:

1. The uncertainty in the inferred reward function.
2. The uncertainty of the DKL surrogate arising from limited spatial observations.
3. The uncertainty associated with evaluating image patches outside the training distribution.

These quantities have different origins and should not automatically be combined into a single uncertainty estimate.

The current analysis addresses the first through a constrained coefficient region and the third through a descriptor-distance diagnostic.

It does not yet construct the DKL surrogate.

---

### 15. Conclusions

This step extends inverse reward learning from one optimal linear reward function to a family of admissible reward functions.

The principal outputs are:

- A constrained region of linear reward coefficients.
- Minimum, fitted, and maximum rewards for recorded candidate patches.
- Reward-interval visualization for held-out decisions.
- Full-image spatial reward maps.
- An estimate of reward sensitivity to admissible coefficient variations.
- A diagnostic of extrapolation beyond the original training patches.

The resulting methodology allows us to investigate not only which image patches are preferred but also how strongly the available decisions constrain those preferences.

The next scientific task will be to compare the reconstructed spatial reward landscape with the actual agent trajectories, and eventually use it to guide new measurements through Deep Kernel Learning.


In [ ]:
# ATHENA DAY 20 — run AFTER day20_single_trace_reward.py in same notebook kernel.
# One trace at a time: conditional feasible coefficient region, exact LP score
# bounds on recorded patches and a grid over the full AFM image.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog
from scipy.spatial.distance import cdist
from pathlib import Path

# ----- Controls -------------------------------------------------------------
SPATIAL_STRIDE = 32          # pixels between sampled full-image patch centers
COEF_REL_RADIUS = 0.50       # per-coefficient feasible half-width, relative to fit
COEF_MIN_RADIUS = 0.25       # min half-width, in standardized descriptor units
PREF_SLACK_EXTRA = 0.10      # extra permissible pairwise reward violation
DISPLAY_HIDDEN = 2          # held-out decision reward intervals to show
PLOT_SELECTION_POSITIONS = True  # requires coordinates to be image (x,y)
OUTDIR = Path(f'day20_trace_{TRACE_ID}_bounds')
OUTDIR.mkdir(exist_ok=True)

# The prior cell must have provided: D, ntrain, w, scaler, features, img, TRACE_ID.
needed = ['D','ntrain','w','scaler','features','img','TRACE_ID']
missing = [x for x in needed if x not in globals()]
if missing: raise RuntimeError('Run the previous single-trace reward cell first. Missing: '+str(missing))
full_image = np.asarray(img, dtype=float).squeeze()
if full_image.ndim != 2: raise ValueError('`img` must be a 2D AFM channel')
if not (0 < ntrain < len(D)): raise ValueError('Invalid train/test split')
if len(w) != D[0]['Z'].shape[1]: raise ValueError('Fitted weights do not match descriptors')
print(f'Trace {TRACE_ID}: {ntrain} training decisions; {len(D)-ntrain} hidden decisions')

# ----- Construct a bounded preference-consistent region ----------------------
# For each selected vs rejected comparison, require
#      (z_selected - z_rejected) . v >= -slack_ij.
# If the point estimate already violates a comparison, allow its observed
# violation plus PREF_SLACK_EXTRA. Thus fitted coefficients remain feasible.
# Box constraints around fitted weights resolve the otherwise unbounded scale.
A_rows, b_rows = [], []
for d in D[:ntrain]:
    Z = d['Z']; a = d['selected']
    diffs = Z[a][None,:] - np.delete(Z, a, axis=0)
    fitted_margins = diffs @ w
    allowable_slack = np.maximum(0.0, -fitted_margins) + PREF_SLACK_EXTRA
    A_rows.extend(-diffs)
    b_rows.extend(allowable_slack)
A_ub = np.asarray(A_rows, dtype=float)
b_ub = np.asarray(b_rows, dtype=float)
halfwidths = np.maximum(COEF_MIN_RADIUS, COEF_REL_RADIUS*np.abs(w))
coefficient_bounds = list(zip(w-halfwidths, w+halfwidths))
check = linprog(np.zeros(len(w)), A_ub=A_ub, b_ub=b_ub,
                bounds=coefficient_bounds, method='highs')
if not check.success: raise RuntimeError('Infeasible region; increase slack or coefficient radius.')
print('Feasible region constructed:', len(b_ub), 'training preference constraints')
print('WARNING: Bounds depend on COEF_REL_RADIUS, COEF_MIN_RADIUS, and PREF_SLACK_EXTRA.')

# The following LPs return exact min/max within this *defined* feasible set.
def reward_interval(z):
    z = np.asarray(z, dtype=float).ravel()
    low = linprog(z, A_ub=A_ub, b_ub=b_ub, bounds=coefficient_bounds, method='highs')
    high = linprog(-z, A_ub=A_ub, b_ub=b_ub, bounds=coefficient_bounds, method='highs')
    if not low.success or not high.success:
        raise RuntimeError(f'LP did not converge: {low.message}; {high.message}')
    return float(low.fun), float(-high.fun)

# ----- Evaluate all recorded candidate patches: train and held-out ----------
rows, metric_rows = [], []
for d_idx, d in enumerate(D):
    split = 'train' if d_idx < ntrain else 'test'
    Z = d['Z']; a = d['selected']; fit_rewards = Z @ w
    low = np.empty(len(Z)); high = np.empty(len(Z))
    for j, z in enumerate(Z): low[j], high[j] = reward_interval(z)
    d['bound_low'] = low; d['bound_high'] = high
    d['bound_fit'] = fit_rewards
    # No offset-invariant absolute reward scale is observable.
    # Shift intervals by their fitted candidate-set mean (same reference).
    ref = np.mean(fit_rewards)
    for j in range(len(Z)):
        rows.append(dict(trace=TRACE_ID, step=d['step'], split=split,
            candidate=j, selected=int(j==a), reward_fit=float(fit_rewards[j]),
            reward_min=low[j], reward_max=high[j],
            centered_fit=float(fit_rewards[j]-ref),
            centered_min=float(low[j]-ref), centered_max=float(high[j]-ref)))
    others = np.arange(len(Z)) != a
    # Robust preference means every feasible coefficient vector ranks actual
    # above rejected patch; sufficient condition: lower(actual)>upper(other).
    guaranteed_wins = int(np.sum(low[a] > high[others]))
    possible_wins = int(np.sum(high[a] >= low[others]))
    # These bounds are conservative because extrema can require different v.
    metric_rows.append(dict(step=d['step'], split=split,
        actual=a, predicted=int(np.argmax(fit_rewards)),
        selected_rank=1+int(np.sum(fit_rewards>fit_rewards[a])),
        guaranteed_wins_lower_bound=guaranteed_wins,
        possible_wins_upper_bound=possible_wins,
        number_of_rejected=int(others.sum()),
        chosen_reward_min=low[a], chosen_reward_fit=float(fit_rewards[a]),
        chosen_reward_max=high[a]))

patch_bounds = pd.DataFrame(rows)
bounds_validation = pd.DataFrame(metric_rows)
patch_bounds.to_csv(OUTDIR/'recorded_patch_reward_bounds.csv', index=False)
bounds_validation.to_csv(OUTDIR/'decision_bounds_validation.csv', index=False)
print('\nHidden decision summary:')
print(bounds_validation.query("split == 'test'").to_string(index=False))

# ----- Plot coefficient intervals (user-defined, not inferred intervals) -----
feature_labels = globals().get('FEATURE_NAMES', [f'f{i}' for i in range(len(w))])
fig, ax = plt.subplots(figsize=(9,6))
y = np.arange(len(w))
ax.errorbar(w, y, xerr=halfwidths, fmt='o', capsize=3)
ax.set_yticks(y, labels=feature_labels)
ax.axvline(0, ls='--', color='gray')
ax.invert_yaxis()
ax.set_xlabel('Reward coefficient and allowed box range')
ax.set_title(f'Trace {TRACE_ID}: imposed coefficient bounds (not confidence intervals)')
fig.tight_layout(); plt.show()

# ----- Held-out candidate-level reward comparisons --------------------------
for d in D[ntrain:ntrain+DISPLAY_HIDDEN]:
    z = np.arange(len(d['Z'])); a = d['selected']; r = d['bound_fit']
    lo=d['bound_low']; hi=d['bound_high']; predicted=int(np.argmax(r))
    fig, ax = plt.subplots(figsize=(13,4))
    ax.errorbar(z, r, yerr=[np.maximum(0,r-lo),np.maximum(0,hi-r)],
                fmt='o', capsize=3, markersize=5)
    ax.scatter([a],[r[a]],s=160, facecolors='none', edgecolors='green',
               linewidths=2.5, label='Actual selected')
    ax.scatter([predicted],[r[predicted]],marker='x',s=90,
               label='Predicted highest')
    ax.set_xticks(z); ax.set_xlabel('Candidate index'); ax.set_ylabel('Inferred reward')
    ax.set_title(f'Trace {TRACE_ID} / held-out step {d["step"]}: exact conditional reward intervals')
    ax.legend(); fig.tight_layout(); plt.show()

    ncols=5; nrows=int(np.ceil(len(z)/ncols))
    fig, axs=plt.subplots(nrows,ncols,figsize=(15,nrows*2.8),squeeze=False)
    for j,ax in enumerate(axs.flat):
        if j>=len(z): ax.axis('off');continue
        ax.imshow(d['patches'][j],cmap='gray')
        ax.set_title(f'{j}: {lo[j]:.2f} / {r[j]:.2f} / {hi[j]:.2f}'
                     + (' * chosen' if j==a else '') + (' + pred' if j==predicted else ''), fontsize=8)
        ax.set_xticks([]);ax.set_yticks([])
        if j==a:
            for spine in ax.spines.values():
                spine.set_visible(True);spine.set_edgecolor('green');spine.set_linewidth(3)
    fig.suptitle('Each tile: candidate | min / fitted / max conditional reward')
    fig.tight_layout(); plt.show()

# ----- Scan across the WHOLE AFM image using patches of the same shape -------
# Only locations where a full patch fits are evaluated; unmeasured centers
# remain NaN (not zero reward). No extrapolation at image borders.
ph,pw = np.asarray(D[0]['patches'][0]).squeeze().shape
H,W = full_image.shape
if H<ph or W<pw: raise ValueError('Original AFM image is smaller than candidate patch.')
half_top=ph//2; half_left=pw//2
ys=np.arange(0,H-ph+1,SPATIAL_STRIDE,dtype=int)
xs=np.arange(0,W-pw+1,SPATIAL_STRIDE,dtype=int)
if ys[-1] != H-ph: ys=np.r_[ys,H-ph]
if xs[-1] != W-pw: xs=np.r_[xs,W-pw]
centers=[]; spatial_features=[]
for y0 in ys:
    for x0 in xs:
        patch=full_image[y0:y0+ph,x0:x0+pw]
        spatial_features.append(features(patch))
        centers.append((x0+pw//2,y0+ph//2))
spatial_features=np.asarray(spatial_features)
spatial_Z=np.nan_to_num(scaler.transform(spatial_features))

# Out-of-distribution diagnostic: Euclidean distance to nearest training patch
# in standardized feature space (heuristic, not calibrated uncertainty).
train_Z=np.vstack([d['Z'] for d in D[:ntrain]])
training_nn=cdist(train_Z,train_Z)
np.fill_diagonal(training_nn,np.inf)
threshold=float(np.percentile(np.min(training_nn,axis=1),95)) if len(train_Z)>1 else np.inf
spatial_nn=np.min(cdist(spatial_Z,train_Z),axis=1)
out_of_distribution=spatial_nn>threshold

spatial_fit=spatial_Z@w
spatial_low=np.empty(len(spatial_Z)); spatial_high=np.empty(len(spatial_Z))
print(f'Computing exact LP bounds at {len(spatial_Z)} grid locations...')
for i,z in enumerate(spatial_Z):
    spatial_low[i],spatial_high[i]=reward_interval(z)
shape=(len(ys),len(xs))
R_fit=spatial_fit.reshape(shape)
R_low=spatial_low.reshape(shape)
R_high=spatial_high.reshape(shape)
R_width=(spatial_high-spatial_low).reshape(shape)
OOD=out_of_distribution.reshape(shape)
center_x=np.asarray(centers)[:,0].reshape(shape)
center_y=np.asarray(centers)[:,1].reshape(shape)

spatial_table=pd.DataFrame(dict(
    trace=TRACE_ID, x_center=center_x.ravel(),y_center=center_y.ravel(),
    reward_min=spatial_low,reward_fit=spatial_fit,reward_max=spatial_high,
    reward_interval_width=spatial_high-spatial_low,
    ood_nearest_neighbor_distance=spatial_nn,ood_flag=out_of_distribution))
spatial_table.to_csv(OUTDIR/'whole_image_reward_grid.csv', index=False)
np.savez_compressed(OUTDIR/'whole_image_reward_maps.npz',
    reward_fit=R_fit,reward_min=R_low,reward_max=R_high,
    reward_width=R_width,ood=OOD,x=center_x,y=center_y)

# Use scatter-grid plotted at actual patch centers to avoid misleading padding.
fig,axs=plt.subplots(2,3,figsize=(16,10),constrained_layout=True)
axs[0,0].imshow(full_image,cmap='gray',origin='upper')
axs[0,0].set_title('Original AFM image')
for ax, val, title in [
        (axs[0,1],spatial_fit,'Fitted reward'),
        (axs[0,2],spatial_low,'Feasible minimum reward'),
        (axs[1,0],spatial_high,'Feasible maximum reward'),
        (axs[1,1],spatial_high-spatial_low,'Feasible reward range'),
        (axs[1,2],spatial_nn,'Feature distance to training patches')]:
    im=ax.scatter(center_x.ravel(),center_y.ravel(),c=val,s=70,marker='s',cmap='viridis')
    ax.set_xlim(0,W); ax.set_ylim(H,0); ax.set_aspect('equal')
    ax.set_title(title)
    fig.colorbar(im,ax=ax,shrink=.75)
    if title=='Feature distance to training patches':
        ax.contour(center_x,center_y,OOD.astype(float),levels=[.5],colors='red',linewidths=1)

# `position` recorded by trace has not been verified to use (x,y) rather than
# (row,column), so positions are OFF by default unless convention is verified.
if PLOT_SELECTION_POSITIONS:
    print('Selected locations not overlaid: coordinate convention of `position` is unverified.')
plt.show()

print('\nOutput directory:', OUTDIR.resolve())
print('IMPORTANT: min/max are exact only within the USER-SPECIFIED constraint set.')
print('They are not unique reward bounds, statistical confidence bands, or posterior intervals.')
print('OOD flags are heuristic and reward intervals need not widen in unobserved regions.')


Improved visualization

In [ ]:
# ATHENA DAY 20 — run AFTER day20_single_trace_reward.py in same notebook kernel.
# One trace at a time: conditional feasible coefficient region, exact LP score
# bounds on recorded patches and a grid over the full AFM image.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog
from scipy.spatial.distance import cdist
from pathlib import Path

# ----- Controls -------------------------------------------------------------
SPATIAL_STRIDE = 4           # 4x denser in X and Y than previous stride=16
COEF_REL_RADIUS = 0.50       # per-coefficient feasible half-width, relative to fit
COEF_MIN_RADIUS = 0.25       # min half-width, in standardized descriptor units
PREF_SLACK_EXTRA = 0.10      # extra permissible pairwise reward violation
DISPLAY_HIDDEN = None       # None plots EVERY held-out decision; integer limits count
PLOT_SELECTION_POSITIONS = True  # requires coordinates to be image (x,y)
OUTDIR = Path(f'day20_trace_{TRACE_ID}_bounds')
OUTDIR.mkdir(exist_ok=True)

# The prior cell must have provided: D, ntrain, w, scaler, features, img, TRACE_ID.
needed = ['D','ntrain','w','scaler','features','img','TRACE_ID']
missing = [x for x in needed if x not in globals()]
if missing: raise RuntimeError('Run the previous single-trace reward cell first. Missing: '+str(missing))
full_image = np.asarray(img, dtype=float).squeeze()
if full_image.ndim != 2: raise ValueError('`img` must be a 2D AFM channel')
if not (0 < ntrain < len(D)): raise ValueError('Invalid train/test split')
if len(w) != D[0]['Z'].shape[1]: raise ValueError('Fitted weights do not match descriptors')
print(f'Trace {TRACE_ID}: {ntrain} training decisions; {len(D)-ntrain} hidden decisions')

# ----- Construct a bounded preference-consistent region ----------------------
# For each selected vs rejected comparison, require
#      (z_selected - z_rejected) . v >= -slack_ij.
# If the point estimate already violates a comparison, allow its observed
# violation plus PREF_SLACK_EXTRA. Thus fitted coefficients remain feasible.
# Box constraints around fitted weights resolve the otherwise unbounded scale.
A_rows, b_rows = [], []
for d in D[:ntrain]:
    Z = d['Z']; a = d['selected']
    diffs = Z[a][None,:] - np.delete(Z, a, axis=0)
    fitted_margins = diffs @ w
    allowable_slack = np.maximum(0.0, -fitted_margins) + PREF_SLACK_EXTRA
    A_rows.extend(-diffs)
    b_rows.extend(allowable_slack)
A_ub = np.asarray(A_rows, dtype=float)
b_ub = np.asarray(b_rows, dtype=float)
halfwidths = np.maximum(COEF_MIN_RADIUS, COEF_REL_RADIUS*np.abs(w))
coefficient_bounds = list(zip(w-halfwidths, w+halfwidths))
check = linprog(np.zeros(len(w)), A_ub=A_ub, b_ub=b_ub,
                bounds=coefficient_bounds, method='highs')
if not check.success: raise RuntimeError('Infeasible region; increase slack or coefficient radius.')
print('Feasible region constructed:', len(b_ub), 'training preference constraints')
print('WARNING: Bounds depend on COEF_REL_RADIUS, COEF_MIN_RADIUS, and PREF_SLACK_EXTRA.')

# The following LPs return exact min/max within this *defined* feasible set.
def reward_interval(z):
    z = np.asarray(z, dtype=float).ravel()
    low = linprog(z, A_ub=A_ub, b_ub=b_ub, bounds=coefficient_bounds, method='highs')
    high = linprog(-z, A_ub=A_ub, b_ub=b_ub, bounds=coefficient_bounds, method='highs')
    if not low.success or not high.success:
        raise RuntimeError(f'LP did not converge: {low.message}; {high.message}')
    return float(low.fun), float(-high.fun)

# ----- Evaluate all recorded candidate patches: train and held-out ----------
rows, metric_rows = [], []
for d_idx, d in enumerate(D):
    split = 'train' if d_idx < ntrain else 'test'
    Z = d['Z']; a = d['selected']; fit_rewards = Z @ w
    low = np.empty(len(Z)); high = np.empty(len(Z))
    for j, z in enumerate(Z): low[j], high[j] = reward_interval(z)
    d['bound_low'] = low; d['bound_high'] = high
    d['bound_fit'] = fit_rewards
    # No offset-invariant absolute reward scale is observable.
    # Shift intervals by their fitted candidate-set mean (same reference).
    ref = np.mean(fit_rewards)
    for j in range(len(Z)):
        rows.append(dict(trace=TRACE_ID, step=d['step'], split=split,
            candidate=j, selected=int(j==a), reward_fit=float(fit_rewards[j]),
            reward_min=low[j], reward_max=high[j],
            centered_fit=float(fit_rewards[j]-ref),
            centered_min=float(low[j]-ref), centered_max=float(high[j]-ref)))
    others = np.arange(len(Z)) != a
    # Robust preference means every feasible coefficient vector ranks actual
    # above rejected patch; sufficient condition: lower(actual)>upper(other).
    guaranteed_wins = int(np.sum(low[a] > high[others]))
    possible_wins = int(np.sum(high[a] >= low[others]))
    # These bounds are conservative because extrema can require different v.
    metric_rows.append(dict(step=d['step'], split=split,
        actual=a, predicted=int(np.argmax(fit_rewards)),
        selected_rank=1+int(np.sum(fit_rewards>fit_rewards[a])),
        guaranteed_wins_lower_bound=guaranteed_wins,
        possible_wins_upper_bound=possible_wins,
        number_of_rejected=int(others.sum()),
        chosen_reward_min=low[a], chosen_reward_fit=float(fit_rewards[a]),
        chosen_reward_max=high[a]))

patch_bounds = pd.DataFrame(rows)
bounds_validation = pd.DataFrame(metric_rows)
patch_bounds.to_csv(OUTDIR/'recorded_patch_reward_bounds.csv', index=False)
bounds_validation.to_csv(OUTDIR/'decision_bounds_validation.csv', index=False)
print('\nHidden decision summary:')
print(bounds_validation.query("split == 'test'").to_string(index=False))

# ----- Plot coefficient intervals (user-defined, not inferred intervals) -----
feature_labels = globals().get('FEATURE_NAMES', [f'f{i}' for i in range(len(w))])
fig, ax = plt.subplots(figsize=(9,6))
y = np.arange(len(w))
ax.errorbar(w, y, xerr=halfwidths, fmt='o', capsize=3)
ax.set_yticks(y, labels=feature_labels)
ax.axvline(0, ls='--', color='gray')
ax.invert_yaxis()
ax.set_xlabel('Reward coefficient and allowed box range')
ax.set_title(f'Trace {TRACE_ID}: imposed coefficient bounds (not confidence intervals)')
fig.tight_layout(); plt.show()

# ----- Reward intervals for every held-out decision -------------------------
# Red: chosen candidate. Blue: all rejected candidates.
# A vertical line indicates feasible min/max and a dot the fitted reward.
# A red horizontal line shows the fitted reward of the actual chosen patch.
def plot_decision_reward_intervals(d, trace_id=TRACE_ID, output_dir=OUTDIR,
                                   show_images=False, save=True):
    z = np.arange(len(d['Z']))
    a = int(d['selected'])
    fit = np.asarray(d['bound_fit'])
    lo = np.asarray(d['bound_low'])
    hi = np.asarray(d['bound_high'])
    pred = int(np.argmax(fit))
    fig, ax = plt.subplots(figsize=(max(11, len(z)*0.58), 5))
    for j in z:
        color = 'red' if j == a else 'royalblue'
        ax.vlines(j, lo[j], hi[j], colors=color, linewidth=2.2 if j==a else 1.35,
                  alpha=1 if j==a else .8)
        ax.hlines([lo[j],hi[j]],j-.14,j+.14,colors=color,linewidth=1.4)
        ax.scatter(j,fit[j],c=color,s=68 if j==a else 24,zorder=4)
    ax.axhline(fit[a],color='red',linestyle='--',alpha=.55,linewidth=1.1,
               label='Actual chosen: fitted reward')
    ax.scatter([],[],color='red',label=f'Chosen patch #{a}')
    ax.scatter([],[],color='royalblue',label='Rejected patches')
    if pred != a:
        ax.scatter([pred],[fit[pred]],marker='x',c='black',s=90,linewidth=2,
                   zorder=6,label=f'Model top patch #{pred}')
    ax.set_xticks(z)
    ax.set_xlabel('Candidate patch index')
    ax.set_ylabel('Inferred reward (conditional feasible min / fitted / max)')
    ax.set_title(f'Trace {trace_id} | step {d["step"]} | chosen={a}, model top={pred}')
    ax.grid(alpha=.17,axis='y')
    ax.legend(loc='best',fontsize=9)
    fig.tight_layout()
    if save:
        fig.savefig(output_dir/f'decision_{str(d["step"])}_reward_intervals.png',dpi=180)
    plt.show()
    if show_images:
        ncols=5; nrows=int(np.ceil(len(z)/ncols))
        fig, axs=plt.subplots(nrows,ncols,figsize=(15,nrows*2.8),squeeze=False)
        for j,axis in enumerate(axs.flat):
            if j>=len(z):axis.axis('off');continue
            axis.imshow(d['patches'][j],cmap='gray')
            axis.set_title(f'#{j}: [{lo[j]:.2f}, {hi[j]:.2f}]\nfit={fit[j]:.2f}'
                 +(' CHOSEN' if j==a else ''),fontsize=8,
                 color='red' if j==a else 'royalblue')
            axis.set_xticks([]);axis.set_yticks([])
            for spine in axis.spines.values():
                spine.set_visible(True)
                spine.set_edgecolor('red' if j==a else 'royalblue')
                spine.set_linewidth(2.5 if j==a else .8)
        fig.tight_layout()
        if save:fig.savefig(output_dir/f'decision_{str(d["step"])}_patches.png',dpi=150)
        plt.show()

hidden = D[ntrain:] if DISPLAY_HIDDEN is None else D[ntrain:ntrain+DISPLAY_HIDDEN]
for d in hidden:
    plot_decision_reward_intervals(d, show_images=False)

# Trajectory-level visualization: chosen patch's fitted reward and interval at
# each decision step; test steps are visually distinguished by a shaded band.
fig,ax=plt.subplots(figsize=(14,5))
steps=np.arange(len(D))
chosen_fit=np.array([d['bound_fit'][d['selected']] for d in D])
chosen_min=np.array([d['bound_low'][d['selected']] for d in D])
chosen_max=np.array([d['bound_high'][d['selected']] for d in D])
ax.fill_between(steps,chosen_min,chosen_max,color='red',alpha=.15,
                label='Chosen patch reward interval')
ax.plot(steps,chosen_fit,'-',color='red',linewidth=1.5,
        label='Chosen patch fitted reward')
ax.axvline(ntrain-.5,color='black',linestyle='--',linewidth=1,
           label='Start of held-out 20%')
ax.set_xlabel('Decision number within trace')
ax.set_ylabel('Inferred chosen-patch reward')
ax.set_title(f'Trace {TRACE_ID}: chosen patch reward and interval across all steps')
ax.legend();ax.grid(alpha=.2);fig.tight_layout()
fig.savefig(OUTDIR/'chosen_patch_reward_trajectory.png',dpi=180)
plt.show()

# ----- Scan across the WHOLE AFM image using patches of the same shape -------
# Only locations where a full patch fits are evaluated; unmeasured centers
# remain NaN (not zero reward). No extrapolation at image borders.
ph,pw = np.asarray(D[0]['patches'][0]).squeeze().shape
H,W = full_image.shape
if H<ph or W<pw: raise ValueError('Original AFM image is smaller than candidate patch.')
half_top=ph//2; half_left=pw//2
ys=np.arange(0,H-ph+1,SPATIAL_STRIDE,dtype=int)
xs=np.arange(0,W-pw+1,SPATIAL_STRIDE,dtype=int)
if ys[-1] != H-ph: ys=np.r_[ys,H-ph]
if xs[-1] != W-pw: xs=np.r_[xs,W-pw]
centers=[]; spatial_features=[]
for y0 in ys:
    for x0 in xs:
        patch=full_image[y0:y0+ph,x0:x0+pw]
        spatial_features.append(features(patch))
        centers.append((x0+pw//2,y0+ph//2))
spatial_features=np.asarray(spatial_features)
spatial_Z=np.nan_to_num(scaler.transform(spatial_features))

# Out-of-distribution diagnostic: Euclidean distance to nearest training patch
# in standardized feature space (heuristic, not calibrated uncertainty).
train_Z=np.vstack([d['Z'] for d in D[:ntrain]])
training_nn=cdist(train_Z,train_Z)
np.fill_diagonal(training_nn,np.inf)
threshold=float(np.percentile(np.min(training_nn,axis=1),95)) if len(train_Z)>1 else np.inf
spatial_nn=np.min(cdist(spatial_Z,train_Z),axis=1)
out_of_distribution=spatial_nn>threshold

spatial_fit=spatial_Z@w
spatial_low=np.empty(len(spatial_Z)); spatial_high=np.empty(len(spatial_Z))
print(f'Computing exact LP bounds at {len(spatial_Z)} grid locations...')
print('Note: stride=4 is ~16 times more points than stride=16; each point uses 2 linear programs.')
for i,z in enumerate(spatial_Z):
    spatial_low[i],spatial_high[i]=reward_interval(z)
shape=(len(ys),len(xs))
R_fit=spatial_fit.reshape(shape)
R_low=spatial_low.reshape(shape)
R_high=spatial_high.reshape(shape)
R_width=(spatial_high-spatial_low).reshape(shape)
OOD=out_of_distribution.reshape(shape)
center_x=np.asarray(centers)[:,0].reshape(shape)
center_y=np.asarray(centers)[:,1].reshape(shape)

spatial_table=pd.DataFrame(dict(
    trace=TRACE_ID, x_center=center_x.ravel(),y_center=center_y.ravel(),
    reward_min=spatial_low,reward_fit=spatial_fit,reward_max=spatial_high,
    reward_interval_width=spatial_high-spatial_low,
    ood_nearest_neighbor_distance=spatial_nn,ood_flag=out_of_distribution))
spatial_table.to_csv(OUTDIR/'whole_image_reward_grid.csv', index=False)
np.savez_compressed(OUTDIR/'whole_image_reward_maps.npz',
    reward_fit=R_fit,reward_min=R_low,reward_max=R_high,
    reward_width=R_width,ood=OOD,x=center_x,y=center_y)

# Use scatter-grid plotted at actual patch centers to avoid misleading padding.
fig,axs=plt.subplots(2,3,figsize=(16,10),constrained_layout=True)
axs[0,0].imshow(full_image,cmap='gray',origin='upper')
axs[0,0].set_title('Original AFM image')
for ax, val, title in [
        (axs[0,1],spatial_fit,'Fitted reward'),
        (axs[0,2],spatial_low,'Feasible minimum reward'),
        (axs[1,0],spatial_high,'Feasible maximum reward'),
        (axs[1,1],spatial_high-spatial_low,'Feasible reward range'),
        (axs[1,2],spatial_nn,'Feature distance to training patches')]:
    im=ax.scatter(center_x.ravel(),center_y.ravel(),c=val,s=70,marker='s',cmap='viridis')
    ax.set_xlim(0,W); ax.set_ylim(H,0); ax.set_aspect('equal')
    ax.set_title(title)
    fig.colorbar(im,ax=ax,shrink=.75)
    if title=='Feature distance to training patches':
        ax.contour(center_x,center_y,OOD.astype(float),levels=[.5],colors='red',linewidths=1)

# `position` recorded by trace has not been verified to use (x,y) rather than
# (row,column), so positions are OFF by default unless convention is verified.
if PLOT_SELECTION_POSITIONS:
    print('Selected locations not overlaid: coordinate convention of `position` is unverified.')
plt.show()

print('\nOutput directory:', OUTDIR.resolve())
print('IMPORTANT: min/max are exact only within the USER-SPECIFIED constraint set.')
print('They are not unique reward bounds, statistical confidence bands, or posterior intervals.')
print('OOD flags are heuristic and reward intervals need not widen in unobserved regions.')


In [ ]:

# ================================================================
# ATHENA DAY 20 — GLOBAL REWARD VISUALIZATION (CORRECTED)
#
# Paste after the single-trace reward-bounds calculation.
#
# FIXES:
# 1. No assumption that train_idx/test_idx match D positions
# 2. Recover split from recorded decision tables when possible
# 3. Never silently generate a different validation split
# 4. Mark random held-out decisions individually
#
# PLOTS:
# A. Blue violins: rejected candidate rewards
#    Red points/intervals: selected candidate
# B. Reward evolution vs decision step
# C. Reward advantage and ranking error vs step
# D. Concordance vs step
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pathlib import Path

# ================================================================
# 1. CONFIGURATION
# ================================================================

OUTPUT_DIR = Path("day20_global_reward_visualization")
OUTPUT_DIR.mkdir(exist_ok=True)

SHOW_INTERVALS = True
FIGURE_WIDTH = 19
FIGURE_DPI = 180

# ================================================================
# 2. VALIDATE EXISTING DATA
# ================================================================

if "D" not in globals() or len(D) == 0:
    raise ValueError(
        "Run the single-trace reward analysis first."
    )

N = len(D)

required_fields = [
    "step",
    "selected",
    "bound_fit",
    "bound_low",
    "bound_high"
]

for i, d in enumerate(D):
    missing = [k for k in required_fields if k not in d]
    if missing:
        raise ValueError(
            f"Decision {i} is missing: {missing}. "
            "Run the reward-bounds cell first."
        )

# ================================================================
# 3. RECOVER TRAIN/TEST ASSIGNMENTS SAFELY
# ================================================================

def normalize_split(value):
    if pd.isna(value):
        return None
    value = str(value).strip().lower()
    if value in ("train", "training"):
        return "train"
    if value in ("test", "heldout", "held-out", "validation"):
        return "test"
    return None


def recover_splits(D):
    n = len(D)

    # Priority 1: split attached directly to each decision
    if all("split" in d for d in D):
        values = [normalize_split(d["split"]) for d in D]
        if all(v is not None for v in values):
            print("Split source: D[*]['split']")
            return values

    # Priority 2: existing decision-level table, matched by step
    for table_name in [
        "choice_table",
        "decision_bounds_validation_df",
        "decision_df",
        "decision_table"
    ]:
        if table_name not in globals():
            continue

        table = globals()[table_name]

        if not isinstance(table, pd.DataFrame):
            continue

        if not {"step", "split"}.issubset(table.columns):
            continue

        subset = table[["step", "split"]].drop_duplicates()

        # Ignore mixed or contradictory assignments
        if subset["step"].duplicated().any():
            continue

        lookup = dict(zip(subset.step, subset.split))

        if not all(d["step"] in lookup for d in D):
            continue

        values = [
            normalize_split(lookup[d["step"]])
            for d in D
        ]

        if all(v is not None for v in values):
            print("Split source:", table_name)
            return values

    # Priority 3: existing indices, only if they are valid
    if "train_idx" in globals() and "test_idx" in globals():
        try:
            train_array = np.asarray(
                train_idx, dtype=int
            ).ravel()

            test_array = np.asarray(
                test_idx, dtype=int
            ).ravel()

            train_set = set(train_array.tolist())
            test_set = set(test_array.tolist())

            # Position-based interpretation
            if (
                train_set.isdisjoint(test_set)
                and train_set | test_set == set(range(n))
            ):
                print("Split source: positional train_idx/test_idx")
                return [
                    "test" if i in test_set else "train"
                    for i in range(n)
                ]

            # Original step-ID interpretation
            step_values = [d["step"] for d in D]

            if (
                len(set(step_values)) == n
                and train_set.isdisjoint(test_set)
                and train_set | test_set == set(step_values)
            ):
                print("Split source: step-ID train_idx/test_idx")
                return [
                    "test" if d["step"] in test_set else "train"
                    for d in D
                ]

        except (TypeError, ValueError):
            pass

    print(
        "WARNING: Original train/test assignment not recoverable.\n"
        "Plotting all decisions without held-out markers.\n"
        "No new split has been generated."
    )

    return ["unknown"] * n


split_labels = recover_splits(D)

test_positions = np.array([
    i for i, label in enumerate(split_labels)
    if label == "test"
], dtype=int)

train_positions = np.array([
    i for i, label in enumerate(split_labels)
    if label == "train"
], dtype=int)

print("\nDataset summary")
print("Total decisions:", N)
print("Training decisions:", len(train_positions))
print("Held-out decisions:", len(test_positions))

# ================================================================
# 4. BUILD PATCH AND DECISION TABLES
# ================================================================

patch_rows = []
decision_rows = []

for t, d in enumerate(D):

    fit = np.asarray(d["bound_fit"], dtype=float).ravel()
    low = np.asarray(d["bound_low"], dtype=float).ravel()
    high = np.asarray(d["bound_high"], dtype=float).ravel()

    chosen = int(d["selected"])
    K = len(fit)

    if not (
        K == len(low) == len(high)
        and 0 <= chosen < K
    ):
        raise ValueError(f"Invalid reward arrays at decision {t}")

    if K < 2:
        raise ValueError(
            f"Decision {t} has fewer than two candidates."
        )

    if not (
        np.all(np.isfinite(fit))
        and np.all(np.isfinite(low))
        and np.all(np.isfinite(high))
    ):
        raise ValueError(
            f"Non-finite rewards at decision {t}"
        )

    chosen_fit = fit[chosen]
    chosen_low = low[chosen]
    chosen_high = high[chosen]

    alternatives = np.delete(fit, chosen)

    rejected_mean = alternatives.mean()
    rejected_median = np.median(alternatives)
    rejected_max = alternatives.max()
    rejected_q75 = np.percentile(alternatives, 75)
    rejected_q95 = np.percentile(alternatives, 95)

    margin = chosen_fit - rejected_max

    concordance = (
        np.mean(chosen_fit > alternatives)
        + 0.5 * np.mean(chosen_fit == alternatives)
    )

    actual_rank = (
        1 + np.sum(fit > chosen_fit)
    )

    decision_rows.append({
        "position": t,
        "step": d["step"],
        "split": split_labels[t],
        "selected_candidate": chosen,
        "predicted_candidate": int(np.argmax(fit)),
        "correct": int(np.argmax(fit) == chosen),
        "chosen_reward": chosen_fit,
        "chosen_low": chosen_low,
        "chosen_high": chosen_high,
        "rejected_mean": rejected_mean,
        "rejected_median": rejected_median,
        "rejected_q75": rejected_q75,
        "rejected_q95": rejected_q95,
        "rejected_max": rejected_max,
        "chosen_minus_mean": chosen_fit - rejected_mean,
        "chosen_minus_best": margin,
        "ranking_violation": max(0, -margin),
        "concordance": concordance,
        "actual_rank": actual_rank
    })

    for j in range(K):
        patch_rows.append({
            "position": t,
            "step": d["step"],
            "split": split_labels[t],
            "candidate": j,
            "selected": int(j == chosen),
            "reward_fit": fit[j],
            "reward_min": low[j],
            "reward_max": high[j]
        })

patch_data = pd.DataFrame(patch_rows)
trace_reward_summary_df = pd.DataFrame(decision_rows)

x = np.arange(N)

# ================================================================
# 5. GLOBAL VIOLIN PLOT
# ================================================================

rejected_distributions = []

for t in range(N):
    values = patch_data.loc[
        (patch_data.position == t)
        & (patch_data.selected == 0),
        "reward_fit"
    ].to_numpy()

    rejected_distributions.append(values)

chosen_fit = trace_reward_summary_df.chosen_reward.to_numpy()
chosen_low = trace_reward_summary_df.chosen_low.to_numpy()
chosen_high = trace_reward_summary_df.chosen_high.to_numpy()

fig, ax = plt.subplots(
    figsize=(max(FIGURE_WIDTH, N * 0.18), 7)
)

# Violin plot
vp = ax.violinplot(
    rejected_distributions,
    positions=x,
    widths=0.82,
    showmeans=False,
    showmedians=False,
    showextrema=False
)

for body in vp["bodies"]:
    body.set_facecolor("royalblue")
    body.set_edgecolor("royalblue")
    body.set_alpha(0.25)

# Reward intervals
if SHOW_INTERVALS:
    ax.vlines(
        x,
        chosen_low,
        chosen_high,
        color="red",
        linewidth=1.4,
        alpha=0.85,
        zorder=4
    )

# Fitted chosen reward
ax.plot(
    x,
    chosen_fit,
    color="red",
    linewidth=1.0,
    alpha=0.75,
    zorder=4
)

ax.scatter(
    x,
    chosen_fit,
    color="red",
    s=16,
    zorder=5
)

# Mark held-out decisions
if len(test_positions):
    ax.scatter(
        test_positions,
        chosen_fit[test_positions],
        marker="s",
        s=48,
        facecolors="white",
        edgecolors="black",
        linewidths=1.2,
        zorder=6
    )

legend_handles = [
    Patch(
        facecolor="royalblue",
        alpha=0.25,
        label="Rejected candidates"
    ),
    Line2D(
        [0], [0],
        color="red",
        marker="o",
        label="Actual selected patch"
    ),
    Line2D(
        [0], [0],
        marker="s",
        linestyle="None",
        markerfacecolor="white",
        markeredgecolor="black",
        label="Held-out decision"
    )
]

ax.legend(handles=legend_handles, loc="best")

ax.set_title(
    "Selected Patch vs Counterfactual Reward Distributions",
    fontsize=15
)

ax.set_xlabel("Decision step")
ax.set_ylabel("Inferred reward")

tick_stride = max(1, N // 20)

ax.set_xticks(
    x[::tick_stride],
    [
        str(D[i]["step"])
        for i in x[::tick_stride]
    ]
)

ax.grid(axis="y", alpha=0.15)

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "global_reward_violin.png",
    dpi=FIGURE_DPI
)

plt.show()

# ================================================================
# 6. REWARD EVOLUTION AND RANKING ERROR
# ================================================================

fig, axs = plt.subplots(
    3, 1,
    figsize=(17, 11),
    sharex=True,
    constrained_layout=True
)

# Panel A: reward evolution
axs[0].plot(
    x,
    trace_reward_summary_df.chosen_reward,
    color="red",
    label="Actual selected patch"
)

axs[0].plot(
    x,
    trace_reward_summary_df.rejected_mean,
    color="royalblue",
    label="Mean rejected reward"
)

axs[0].plot(
    x,
    trace_reward_summary_df.rejected_max,
    color="navy",
    linestyle=":",
    label="Best rejected reward"
)

axs[0].set_title("Reward Evolution Across Decisions")
axs[0].set_ylabel("Inferred reward")
axs[0].legend()

# Panel B: reward advantage
axs[1].plot(
    x,
    trace_reward_summary_df.chosen_minus_mean,
    color="purple",
    label="Selected minus mean rejected"
)

axs[1].plot(
    x,
    trace_reward_summary_df.chosen_minus_best,
    color="firebrick",
    label="Selected minus best rejected"
)

axs[1].axhline(
    0,
    color="gray",
    linestyle="--",
    linewidth=1
)

axs[1].set_title("Reward Advantage")
axs[1].set_ylabel("Reward difference")
axs[1].legend()

# Panel C: pairwise concordance
axs[2].plot(
    x,
    trace_reward_summary_df.concordance,
    color="teal",
    label="Pairwise concordance"
)

axs[2].axhline(
    0.5,
    color="gray",
    linestyle="--",
    linewidth=1,
    label="Random-ranking reference"
)

axs[2].set_title("Ranking Consistency")
axs[2].set_ylabel("Concordance")
axs[2].set_ylim(-0.05, 1.05)
axs[2].set_xlabel("Decision step")
axs[2].legend()

# Mark randomly held-out decisions
for ax, column in zip(
    axs,
    [
        "chosen_reward",
        "chosen_minus_best",
        "concordance"
    ]
):
    if len(test_positions):
        ax.scatter(
            test_positions,
            trace_reward_summary_df.iloc[
                test_positions
            ][column],
            marker="s",
            s=40,
            facecolors="white",
            edgecolors="black",
            zorder=6
        )

    ax.grid(alpha=0.15)

axs[2].set_xticks(
    x[::tick_stride],
    [
        str(D[i]["step"])
        for i in x[::tick_stride]
    ]
)

fig.savefig(
    OUTPUT_DIR / "reward_evolution.png",
    dpi=FIGURE_DPI
)

plt.show()

# ================================================================
# 7. ADDITIONAL ERROR-VS-STEP PLOT
# ================================================================

fig, axs = plt.subplots(
    2, 1,
    figsize=(16, 8),
    sharex=True,
    constrained_layout=True
)

axs[0].plot(
    x,
    trace_reward_summary_df.ranking_violation,
    color="firebrick"
)

axs[0].set_title(
    "Ranking Violation vs Decision Step"
)

axs[0].set_ylabel(
    "max(0, best rejected − selected)"
)

axs[1].plot(
    x,
    trace_reward_summary_df.actual_rank,
    color="royalblue"
)

axs[1].axhline(
    1,
    color="gray",
    linestyle="--"
)

axs[1].set_ylabel(
    "Rank of actual selection"
)

axs[1].set_xlabel("Decision step")

for ax, col in zip(
    axs,
    ["ranking_violation", "actual_rank"]
):
    if len(test_positions):
        ax.scatter(
            test_positions,
            trace_reward_summary_df.iloc[
                test_positions
            ][col],
            marker="s",
            s=40,
            facecolors="white",
            edgecolors="black",
            zorder=5
        )

    ax.grid(alpha=0.15)

fig.savefig(
    OUTPUT_DIR / "ranking_error_vs_step.png",
    dpi=FIGURE_DPI
)

plt.show()

# ================================================================
# 8. SUMMARY STATISTICS
# ================================================================

print("\n==============================")
print("REWARD MODEL VALIDATION")
print("==============================")

metrics = [
    "correct",
    "actual_rank",
    "concordance",
    "chosen_minus_mean",
    "chosen_minus_best",
    "ranking_violation"
]

summary_table = (
    trace_reward_summary_df
    .groupby("split")[metrics]
    .mean()
    .round(4)
)

print(summary_table.to_string())

# ================================================================
# 9. EXPORT RESULTS
# ================================================================

patch_data.to_csv(
    OUTPUT_DIR / "all_patch_rewards.csv",
    index=False
)

trace_reward_summary_df.to_csv(
    OUTPUT_DIR / "decision_reward_summary.csv",
    index=False
)

summary_table.to_csv(
    OUTPUT_DIR / "train_test_metrics.csv"
)

print("\nFiles saved to:")
print(OUTPUT_DIR.resolve())



# Iterative Reconstruction of Latent Reward Functions

## 1. Scientific Objective

The objective is to reconstruct the decision-making logic of a human or AI agent from a recorded sequence of microscopy image selections.

At each decision step, the agent is presented with multiple candidate image patches and selects one. We know which patch was selected, but the numerical rewards underlying the choice are not observed.

Previously, we reconstructed a linear reward function directly from the choices. Here, we introduce a more flexible approach in which **the latent rewards of individual patches and the systematic reward function are estimated iteratively**.

The approach is inspired by the Expectation-Maximization (EM) algorithm used in latent-variable models, including Gaussian Mixture Models (GMMs).

The central scientific questions are:

1. Can we reconstruct continuous latent rewards for the selected and rejected image patches?
2. Does iteratively updating these rewards improve the inferred linear decision model?
3. Does the updated reward model predict decisions excluded from training more accurately?
4. How does iteration change the spatial reward landscape across the original microscopy image?

The implementation analyzes **one decision trace at a time**, because different traces may correspond to different decision-making rules.

---

## 2. Input Data and Independent Training/Test Split

The code operates directly on the `traces` dictionary loaded by the original ATHENA Day 20 notebook.

One trace is selected through

`TRACE_ID = 6`

Each trace contains a sequence of decisions indexed by $t$.

At decision step $t$, the candidate images are

$$
\mathcal C_t =
\{I_{t,1},I_{t,2},\ldots,I_{t,K_t}\},
$$

where $K_t$ is the number of candidates.

The observed selection is represented by $a_t$.

The code extracts all candidate patches and their descriptors from the selected trace.

It then randomly partitions **complete decisions**, rather than individual image patches, into:

- 80% training decisions.
- 20% held-out decisions.

A fixed random seed ensures reproducibility.

All patches belonging to one decision remain together in the same partition.

The hidden selections are not used to estimate the reward-model coefficients.

They are used only after training to evaluate how well the inferred reward reproduces the observed choices.

This is a random decision-level validation split. It tests generalization within the recorded trajectory, but does not necessarily establish transferability to an entirely different experiment or an independent decision-maker.

---

## 3. Image Descriptor Representation

Each candidate patch is converted into a vector of 16 numerical descriptors:

$$
\mathbf f(I)=
[f_1(I),f_2(I),\ldots,f_{16}(I)]^T.
$$

The descriptors include:

| Category | Descriptors |
|---|---|
| Intensity | Mean, standard deviation, 5th and 95th percentiles, contrast |
| Distribution shape | Skewness and kurtosis |
| Gradients | Mean gradient, gradient standard deviation, gradient energy |
| Spatial structure | Absolute Laplacian, edge anisotropy |
| Texture | Histogram entropy, high-frequency Fourier power |
| Correlations | Horizontal and vertical nearest-neighbor correlations |

The descriptors are standardized using statistics computed from the training patches only:

$$
z_k(I)=
\frac{f_k(I)-\mu_k}{\sigma_k}.
$$

The resulting vector is

$$
\mathbf z(I)=
[z_1(I),\ldots,z_{16}(I)]^T.
$$

The same training-derived normalization is applied to validation patches and to spatial patches extracted from the original microscopy image.

This ensures that the direct and iterative models operate in the same descriptor space.

---

## 4. Baseline: Direct Linear Preference Model

The first model assumes that the reward is a deterministic linear function of the descriptors:

$$
\boxed{
R(I)=\mathbf w^T\mathbf z(I).
}
$$

The coefficients $\mathbf w$ are learned directly from the observed choices.

For decision $t$, the reward of candidate $i$ is

$$
R_{t,i}=\mathbf w^T\mathbf z_{t,i}.
$$

We assume that selection probabilities follow a softmax distribution:

$$
\boxed{
P(a_t=i\mid\mathbf w)=
\frac{\exp(R_{t,i}/\tau)}
{\sum_{j=1}^{K_t}\exp(R_{t,j}/\tau)}.
}
$$

Here, $\tau$ is the choice temperature, initially fixed to 1.

The model is trained by minimizing the regularized negative log-likelihood:

$$
\boxed{
J_{\mathrm{direct}}(\mathbf w)=
\frac{1}{T_{\mathrm{train}}}
\sum_{t\in\mathrm{train}}
\left[
\log\sum_j
\exp\left(
\frac{\mathbf w^T\mathbf z_{t,j}}{\tau}
\right)
-
\frac{\mathbf w^T\mathbf z_{t,a_t}}{\tau}
\right]
+
\frac{\lambda}{2}\|\mathbf w\|^2.
}
$$

The regularization parameter is initially

$$
\lambda=0.10.
$$

Optimization is performed using L-BFGS-B.

The resulting coefficient vector is denoted

$$
\mathbf w_{\mathrm{direct}}.
$$

This model provides the baseline against which the iterative approach is compared.

---

## 5. Latent Reward Model

The direct model assumes that image descriptors completely determine the reward.

We now relax this assumption.

Instead of treating the reward as exactly equal to the linear prediction, we introduce a latent reward variable:

$$
\boxed{
r_{t,i}=
\mathbf w^T\mathbf z_{t,i}+
\epsilon_{t,i}.
}
$$

We assume that the residual reward contributions follow a Gaussian distribution:

$$
\epsilon_{t,i}\sim
\mathcal N(0,\sigma_r^2).
$$

Equivalently,

$$
\boxed{
r_{t,i}\mid\mathbf w,\mathbf z_{t,i}
\sim
\mathcal N(
\mathbf w^T\mathbf z_{t,i},
\sigma_r^2
).
}
$$

The two components have distinct meanings.

**Systematic reward:** The transferable linear function of measurable image descriptors.

**Latent reward:** The decision-specific utility that may contain additional contributions not represented by the descriptors.

The probability of selecting candidate $i$ depends on the latent reward vector:

$$
\boxed{
P(a_t=i\mid\mathbf r_t)=
\frac{\exp(r_{t,i}/\tau)}
{\sum_j\exp(r_{t,j}/\tau)}.
}
$$

The initial implementation uses

$$
\sigma_r=0.70,
\qquad
\tau=1.
$$

These are modeling parameters, not quantities measured from the trace.

The latent reward variance controls how far individual patch rewards may deviate from the systematic linear model.

---

## 6. Iterative Reconstruction Algorithm

The iterative algorithm alternates between two operations:

1. Estimate latent patch rewards using the observed selections and the current linear model.
2. Update the linear coefficients using the reconstructed latent rewards.

The algorithm begins with the direct linear preference coefficients:

$$
\mathbf w^{(0)}=
\mathbf w_{\mathrm{direct}}.
$$

### 6.1. E-like step: Reconstruct latent patch rewards

At iteration $k$, the current model predicts the systematic rewards

$$
\mathbf m_t^{(k)}=
\mathbf Z_t\mathbf w^{(k)},
$$

where $\mathbf Z_t$ contains the standardized descriptors of all candidates at decision step $t$.

The posterior distribution of the latent reward vector is proportional to

$$
p(\mathbf r_t\mid a_t,\mathbf w^{(k)})
\propto
P(a_t\mid\mathbf r_t)
p(\mathbf r_t\mid\mathbf w^{(k)}).
$$

The implementation calculates the **posterior mode** by minimizing

$$
\boxed{
\mathcal L_t(\mathbf r_t)=
\frac{
\|\mathbf r_t-\mathbf Z_t\mathbf w^{(k)}\|^2
}{
2\sigma_r^2
}
+
\log\sum_j\exp(r_{t,j}/\tau)
-
\frac{r_{t,a_t}}{\tau}.
}
$$

The first term keeps the reconstructed latent rewards close to the systematic linear predictions.

The remaining terms incorporate the actual observed selection.

The resulting estimate is

$$
\boxed{
\hat{\mathbf r}_t^{(k)}
=
\arg\min_{\mathbf r_t}
\mathcal L_t(\mathbf r_t).
}
$$

The optimization is performed independently for every training decision.

Consequently, each decision produces a reconstructed latent reward for every candidate image.

The selected patch generally receives an upward adjustment relative to the other candidates, while rejected patches may receive downward adjustments.

These adjustments are determined by the probabilistic model rather than by assigning arbitrary rewards of 1 or 0.

### 6.2. M-like step: Update the systematic reward model

After reconstructing the latent rewards for all training decisions, the code updates the linear model.

Let $N$ be the total number of training candidate patches.

The new coefficients minimize

$$
\boxed{
J_{\mathrm{update}}(\mathbf w)=
\frac{1}{2N\sigma_r^2}
\sum_{t,i}
\left(
\hat r_{t,i}^{(k)}
-
\mathbf w^T\mathbf z_{t,i}
\right)^2
+
\frac{\lambda}{2}\|\mathbf w\|^2.
}
$$

This is a regularized linear least-squares problem.

The implementation solves it using matrix operations rather than a general-purpose nonlinear optimizer.

The updated coefficients are

$$
\mathbf w^{(k+1)}.
$$

These coefficients define a new systematic reward function:

$$
R^{(k+1)}(I)=
\mathbf w^{(k+1)T}\mathbf z(I).
$$

The algorithm then returns to the latent reward estimation step.

---

## 7. Iteration and Convergence

The E-like and M-like steps repeat until the coefficients stabilize or the maximum number of iterations is reached.

The current settings are

`MAX_ITER = 35`

and

`TOL = 1e-5`.

The relative coefficient change is

$$
\boxed{
\Delta_w^{(k)}=
\frac{
\|\mathbf w^{(k+1)}-\mathbf w^{(k)}\|_2
}{
1+\|\mathbf w^{(k)}\|_2
}.
}
$$

The algorithm terminates when

$$
\Delta_w^{(k)}<10^{-5}
$$

or after 35 updates.

The code records coefficient changes and the root-mean-square difference between reconstructed latent rewards and the linear predictions.

The resulting convergence plot shows whether the alternating updates reach a stable solution.

**Important distinction:** The current implementation uses posterior modes, not posterior expectations. It is therefore an approximate alternating latent-variable algorithm, sometimes described as hard-EM-like, rather than exact EM.

Convergence of the coefficients does not establish that the recovered reward function is uniquely identifiable or that the marginal likelihood of the observed decisions improves monotonically.

---

## 8. Held-Out Validation

After training, the direct and iterative models are evaluated using exactly the same randomly held-out 20% of decisions.

For the iterative model, validation uses only the transferable systematic reward:

$$
\boxed{
R_{\mathrm{iterative}}(I)=
\mathbf w_{\mathrm{iterative}}^T
\mathbf z(I).
}
$$

The latent rewards of hidden decisions are **not** reconstructed using their observed selections.

Doing so would leak the validation labels into the predictions.

For each held-out decision, the models calculate rewards for every candidate patch and predict the candidate with the highest systematic reward:

$$
\hat a_t=\arg\max_i R_{t,i}.
$$

The two models are compared using:

**Top-1 selection accuracy**

$$
\mathrm{Accuracy}=
\frac{1}{T_{\mathrm{test}}}
\sum_t\mathbb 1(\hat a_t=a_t).
$$

**Rank of the selected patch**

$$
\mathrm{Rank}_t=
1+\sum_j
\mathbb 1(R_{t,j}>R_{t,a_t}).
$$

**Pairwise concordance**

$$
C_t=
\frac{1}{K_t-1}
\sum_{j\ne a_t}
\left[
\mathbb 1(R_{t,a_t}>R_{t,j})
+
\frac12\mathbb 1(R_{t,a_t}=R_{t,j})
\right].
$$

**Reward advantage**

$$
\boxed{
\Delta R_t=
R_{t,a_t}-
\max_{j\ne a_t}R_{t,j}.
}
$$

**Ranking violation**

$$
E_t=\max(0,-\Delta R_t).
$$

**Negative log-likelihood**

$$
\mathrm{NLL}=
-\frac{1}{T_{\mathrm{test}}}
\sum_t
\log P(a_t\mid\mathbf w).
$$

Top-1 accuracy, selected-patch rank, pairwise concordance, and negative log-likelihood provide complementary measures of behavioral reconstruction.

The numerical reward advantage is model-scale-dependent and should be interpreted with care when comparing different models.

---

## 9. Visualization of Reward Distributions Across the Trace

The code generates a global visualization containing all decisions in chronological order.

The horizontal axis represents the original decision sequence.

The vertical axis represents inferred systematic reward.

For each step:

- Blue violin: Distribution of rewards assigned to rejected candidate patches.
- Red point: Reward assigned to the actual selected patch.
- Red line: Connects the fitted rewards of selected patches across successive decisions.
- Black outlined square: Indicates a randomly held-out decision.

Two panels are generated:

1. Direct linear reward model.
2. Iteratively reconstructed linear reward model.

Both use the same decisions and descriptor representation.

A successful reward model should generally assign the selected patch a reward near or above the upper end of the rejected-candidate distribution.

The most important evidence of improvement is the behavior of the **held-out decisions**, because the training selections directly influence the estimated model.

Unlike the earlier constrained reward-bound plots, these iterative-model violin plots show point estimates of systematic reward rather than reward confidence or feasibility intervals.

---

## 10. Evolution of Reward Agreement Along the Trace

The second visualization examines how inferred rewards and ranking performance change with decision step.

The plotted quantities include:

$$
R_{\mathrm{chosen}}(t),
$$

$$
\overline R_{\mathrm{rejected}}(t),
$$

$$
\Delta R_t=
R_{\mathrm{chosen}}(t)
-
R_{\mathrm{best\ rejected}}(t),
$$

and

$$
C_t=\text{pairwise concordance}.
$$

The direct and iterative models are shown together.

This permits examination of several questions:

- Does the inferred reward consistently favor the actual selections?
- Are there portions of the trajectory where the model becomes less accurate?
- Does the iterative method improve reward ranking throughout the experiment?
- Are failures concentrated in particular decision steps?

The held-out decisions are marked by faint vertical indicators.

Since the test decisions are randomly distributed, there is no single train/test boundary.

These plots describe reward agreement as a function of decision step. They do not represent a model being retrained sequentially at every step.

---

## 11. Visualization of Latent Reward Evolution During Iteration

The third analysis examines how the latent reward estimates of individual candidate patches evolve across successive algorithm iterations.

Several training decisions are selected for visualization.

For each decision, we plot

$$
\hat r_{t,i}^{(k)}
$$

against iteration number $k$.

The actual selected patch is displayed in red.

All rejected patches are displayed in blue.

These plots show how the latent rewards respond to successive updates of the systematic model.

In particular, they allow us to determine whether the selected patch becomes increasingly separated from competing candidates and whether the reconstructed rewards stabilize.

However, increasing separation within training decisions is not sufficient evidence that the recovered reward function is transferable.

That must be established through the held-out validation.

---

## 12. Spatial Reconstruction of the Reward Function

After estimating the direct and iterative models, we apply both reward functions to a dense collection of patches extracted from the full AFM image.

Let $I_{x,y}$ be the image patch associated with spatial position $(x,y)$.

The direct reward field is

$$
R_{\mathrm{direct}}(x,y)=
\mathbf w_{\mathrm{direct}}^T
\mathbf z(I_{x,y}).
$$

The iterative reward field is

$$
R_{\mathrm{iterative}}(x,y)=
\mathbf w_{\mathrm{iterative}}^T
\mathbf z(I_{x,y}).
$$

The difference is

$$
\boxed{
\Delta R(x,y)=
R_{\mathrm{iterative}}(x,y)
-
R_{\mathrm{direct}}(x,y).
}
$$

The code extracts patches on a regular spatial grid with

`MAP_STRIDE = 4`.

This corresponds to four times denser sampling along each image direction compared with a stride of 16 pixels, or approximately 16 times as many patch locations.

Three maps are generated:

1. Direct linear reward map.
2. Iterative linear reward map.
3. Difference between iterative and direct rewards.

These maps show whether the iterative reconstruction changes which spatial regions are assigned high scientific preference.

The maps are evaluated from the learned systematic functions, not by reconstructing latent rewards using unobserved choices at new spatial positions.

Therefore, the spatial reward maps represent transferable model predictions.

They do not include posterior uncertainty or the constrained reward intervals from the previous analysis.

---

## 13. Output Files

The code stores its results in a directory named according to the selected trace:

`day20_selfcontained_trace_<TRACE_ID>`

The exported files include:

| File | Content |
|---|---|
| `decisions.csv` | Decision-level predictions and validation metrics for both models |
| `patch_rewards.csv` | Inferred systematic rewards for every recorded candidate |
| `iterations.csv` | Convergence diagnostics for the alternating algorithm |
| `coefficients.csv` | Direct and iterative linear reward coefficients |
| `spatial_rewards.csv` | Spatial reward values for both models, when the full AFM image is available |

These files allow direct comparison of the models and further analysis of individual decisions.

---

## 14. Scientific Interpretation and Limitations

The iterative approach introduces a latent-reward layer between image descriptors and observed selections.

This provides a mechanism for reconstructing decision-specific utilities while maintaining a transferable linear model.

However, several limitations remain.

### 14.1. Rewards are not directly observed

The numerical rewards are inferred from choices.

Selection data constrain relative preferences, but do not uniquely identify absolute rewards.

The scale depends on the choice temperature, latent variance, and regularization.

### 14.2. The iterative model introduces additional assumptions

The model assumes Gaussian residual variation around a shared linear reward function.

These assumptions permit latent-reward reconstruction, but they are not established by the original selection data.

### 14.3. Alternation does not create additional information

The algorithm repeatedly uses the same training decisions.

The benefit, if any, arises from the assumed latent-variable structure rather than new observational evidence.

### 14.4. Latent reward fit and systematic reward generalization are different

The reconstructed latent rewards can explain training decisions more accurately than the systematic linear function.

However, these decision-specific rewards are not directly transferable to unseen image patches.

For autonomous optimization, the systematic reward function must predict new choices successfully.

### 14.5. The present model is static

Reward is assumed to depend on image descriptors, not explicitly on experimental history.

If the agent values novelty, uncertainty reduction, or exploration, then the reward may depend on the previous sequence of observations:

$$
R_t(I)=R(I,\mathcal H_t).
$$

Such history-dependent effects are not represented in the current linear descriptor model.

### 14.6. Randomly held-out decisions may be correlated

Even though the test selections are excluded from fitting, neighboring decisions may contain similar patches and experimental contexts.

The validation therefore measures predictive performance within the recorded trajectory, rather than demonstrating generalization to unrelated experiments.

---

## 15. Conclusions

The present implementation compares two approaches to inverse reward reconstruction:

**Direct linear preference learning**, which learns the reward coefficients directly from selection likelihoods.

**Iterative latent reward reconstruction**, which alternates between estimating patch-level latent utilities and updating the linear reward model.

Both methods use identical image descriptors, the same selected trace, and the same randomly held-out validation decisions.

This controlled comparison allows us to determine whether explicit latent-reward reconstruction improves the representation of the decision-making logic.

The principal outputs are:

- Direct and iterative reward coefficients.
- Latent reward trajectories during fitting.
- Held-out selection and ranking performance.
- Chosen-versus-counterfactual reward distributions across the entire trace.
- Reward advantage and concordance as functions of decision step.
- Dense spatial reward maps across the original AFM image.

The central criterion for success is **improved prediction of held-out decisions by the transferable systematic reward model**, rather than improved fit to the training decisions alone.

A subsequent extension can compare this posterior-mode alternating method with exact or approximate Bayesian inference, nonlinear reward functions, and history-dependent selection models.


In [ ]:
# ATHENA Day 20 — standalone single-trace inverse reward learning
# Run after the ORIGINAL notebook (requires only `traces`; `img` optional).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import sobel, laplace
from scipy.stats import skew, kurtosis
from scipy.special import softmax, logsumexp
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler
from pathlib import Path

TRACE_ID = 6               # choose ONE trace (0–10 in original notebook)
TEST_FRACTION = 0.20       # randomly held-out complete decisions
SPLIT_SEED = 42
RIDGE = 0.10
LATENT_SD = 0.70          # fixed latent utility residual standard deviation
TAU = 1.0                 # softmax choice temperature
MAX_ITER = 35
TOL = 1e-5
MAP_STRIDE = 4            # 4 px rather than previous 16 px
SHOW_LATENT_STEPS = 3
MAKE_SPATIAL_MAP = True
SAVE_RESULTS = True

assert 'traces' in globals(), 'Run the original notebook data-loading cells first.'
trace = traces[TRACE_ID]
steps = sorted(trace.keys()) if isinstance(trace, dict) else list(range(len(trace)))

FEATURES = ['mean','std','p05','p95','contrast','skew','kurtosis',
            'gradient_mean','gradient_std','gradient_energy','laplacian',
            'entropy','highfreq','anisotropy','corr_x','corr_y']

def describe(patch):
    a = np.asarray(patch, dtype=float).squeeze()
    if a.ndim != 2: raise ValueError(f'Expected a 2D patch, got {a.shape}')
    finite = a[np.isfinite(a)]
    if finite.size == 0: raise ValueError('Empty/invalid patch')
    a = np.where(np.isfinite(a), a, np.median(finite))
    sd = float(a.std())
    z = (a-a.mean())/(sd+1e-9)
    gx,gy = sobel(z,axis=1)/8, sobel(z,axis=0)/8
    g = np.hypot(gx,gy)
    pwr = abs(np.fft.fftshift(np.fft.fft2(z)))**2
    fx = np.fft.fftshift(np.fft.fftfreq(a.shape[1]))[None,:]
    fy = np.fft.fftshift(np.fft.fftfreq(a.shape[0]))[:,None]
    hist = np.histogram(z,bins=32,range=(-4,4))[0].astype(float)
    probs = hist[hist>0]/max(hist.sum(),1)
    out = [a.mean(),sd,np.percentile(a,5),np.percentile(a,95),
           np.percentile(a,95)-np.percentile(a,5),
           skew(a.ravel()) if sd>1e-9 else 0.,
           kurtosis(a.ravel()) if sd>1e-9 else 0.,
           g.mean(),g.std(),(g*g).mean(),abs(laplace(z)).mean(),
           -np.sum(probs*np.log(probs))/np.log(32),
           pwr[np.hypot(fx,fy)>.25].sum()/(pwr.sum()+1e-9),
           abs(gx).mean()-abs(gy).mean(),
           (z[:,:-1]*z[:,1:]).mean() if z.shape[1]>1 else 0.,
           (z[:-1,:]*z[1:,:]).mean() if z.shape[0]>1 else 0.]
    return np.nan_to_num(out,nan=0,posinf=0,neginf=0)

D=[]
for step in steps:
    entry=trace[step]
    patches=[np.asarray(p).squeeze() for p in entry['patches']]
    a=int(entry['selected_candidate'])
    if not 0<=a<len(patches): raise ValueError(f'Bad selected index at step {step}')
    D.append(dict(step=step,patches=patches,selected=a,
                  X=np.vstack([describe(p) for p in patches])))

rng=np.random.default_rng(SPLIT_SEED)
n=len(D)
if n<5: raise ValueError('At least five decisions are required for 80/20 analysis')
perm=rng.permutation(n)
n_test=max(1,int(round(n*TEST_FRACTION)))
test_idx=np.sort(perm[:n_test]); train_idx=np.sort(perm[n_test:])
train_set=set(train_idx.tolist()); test_set=set(test_idx.tolist())
scaler=StandardScaler().fit(np.vstack([D[i]['X'] for i in train_idx]))
for d in D: d['Z']=np.nan_to_num(scaler.transform(d['X']))
p=len(FEATURES)
print(f'Trace {TRACE_ID}: {n} decisions, {len(train_idx)} train, {len(test_idx)} randomly held out')

# Direct conditional-logit model (one coefficient vector for this trace).
def direct_objective(w):
    loss=0.;grad=np.zeros_like(w)
    for i in train_idx:
        d=D[i]; Z=d['Z']; a=d['selected']; r=Z@w/TAU; q=softmax(r)
        loss += logsumexp(r)-r[a]
        grad += Z.T@(q-np.eye(len(q))[a])/TAU
    return loss/len(train_idx)+RIDGE/2*(w@w),grad/len(train_idx)+RIDGE*w
fit=minimize(direct_objective,np.zeros(p),jac=True,method='L-BFGS-B')
if not fit.success: print('Direct optimizer warning:',fit.message)
w_direct=fit.x.copy()

# E-like step: posterior MODE of latent utility per training decision.
# r ~ N(Z@w, LATENT_SD^2 I); P(choice=a|r)=softmax(r/TAU)[a].
def latent_mode(d,w):
    m=d['Z']@w; a=d['selected']; invvar=1/(LATENT_SD**2)
    def obj(r):
        q=softmax(r/TAU)
        val=.5*invvar*np.sum((r-m)**2)+logsumexp(r/TAU)-r[a]/TAU
        grad=invvar*(r-m)+(q-np.eye(len(q))[a])/TAU
        return val,grad
    res=minimize(obj,m,jac=True,method='L-BFGS-B',options={'maxiter':150,'ftol':1e-11})
    if not res.success and np.linalg.norm(res.jac)>1e-4:
        raise RuntimeError(f'Latent optimization failed at step {d["step"]}: {res.message}')
    return res.x

Xtrain=np.vstack([D[i]['Z'] for i in train_idx]); N=len(Xtrain)
A=Xtrain.T@Xtrain/N+RIDGE*LATENT_SD**2*np.eye(p)
w_iter=w_direct.copy(); trajectory=[w_iter.copy()]; history=[]
tracked=train_idx[:min(SHOW_LATENT_STEPS,len(train_idx))]
latent_tracks={int(i):[] for i in tracked}
for iteration in range(MAX_ITER):
    modes=[latent_mode(D[i],w_iter) for i in train_idx]
    for idx,m in zip(train_idx,modes):
        if int(idx) in latent_tracks: latent_tracks[int(idx)].append(m.copy())
    target=np.concatenate(modes)
    w_next=np.linalg.solve(A,Xtrain.T@target/N)
    change=np.linalg.norm(w_next-w_iter)/(1+np.linalg.norm(w_iter))
    history.append(dict(iteration=iteration+1,coefficient_change=change,
                        latent_residual=np.sqrt(np.mean((target-Xtrain@w_iter)**2))))
    trajectory.append(w_next.copy());w_iter=w_next
    if change<TOL: break
for i in tracked: latent_tracks[int(i)].append(latent_mode(D[int(i)],w_iter))
print(f'Iterative model: {len(history)} updates; final relative coefficient change {history[-1]["coefficient_change"]:.3g}')

# Evaluate transferable systematic rewards; held-out labels do not enter fitting.
def evaluate(w,name):
    dr=[]; pr=[]
    for t,d in enumerate(D):
        r=d['Z']@w; a=d['selected']; others=np.delete(r,a);q=softmax(r/TAU)
        dr.append(dict(model=name,t=t,step=d['step'],split='test' if t in test_set else 'train',
                       actual=a,predicted=int(np.argmax(r)),top1=int(a==np.argmax(r)),
                       rank=int(1+np.sum(r>r[a])),concordance=float(np.mean(r[a]>others)+.5*np.mean(r[a]==others)),
                       chosen=r[a],rejected_mean=others.mean(),rejected_best=others.max(),
                       margin=r[a]-others.max(),violation=max(0.,others.max()-r[a]),
                       nll=-np.log(max(q[a],1e-15))))
        for j,value in enumerate(r):
            pr.append(dict(model=name,t=t,step=d['step'],split='test' if t in test_set else 'train',
                           candidate=j,selected=int(j==a),reward=value))
    return pd.DataFrame(dr),pd.DataFrame(pr)
base_dec,base_patch=evaluate(w_direct,'Direct linear')
iter_dec,iter_patch=evaluate(w_iter,'Iterative latent')
print('\nHeld-out comparisons (test only):')
print(pd.concat([base_dec,iter_dec]).query('split == "test"').groupby('model')
      [['top1','rank','concordance','margin','violation','nll']].mean().round(4))

# Figure 1: per-step rejected violins and chosen red rewards. Random test steps outlined.
def violin_panel(ax,decision,patch,label):
    values=[patch.loc[(patch.t==i)&(patch.selected==0),'reward'].to_numpy() for i in range(n)]
    vp=ax.violinplot(values,positions=np.arange(n),widths=.82,
                     showmeans=False,showmedians=False,showextrema=False)
    for body in vp['bodies']:
        body.set_facecolor('royalblue');body.set_edgecolor('royalblue');body.set_alpha(.22)
    ax.plot(decision.t,decision.chosen,color='red',lw=1,alpha=.8)
    ax.scatter(decision.t,decision.chosen,c='red',s=17,zorder=4)
    held=decision[decision.split=='test']
    ax.scatter(held.t,held.chosen,marker='s',facecolors='none',edgecolors='black',s=47,zorder=5)
    ax.set_ylabel('Reward');ax.set_title(label);ax.grid(axis='y',alpha=.15)
fig,axs=plt.subplots(2,1,figsize=(max(16,n*.18),10),sharex=True)
violin_panel(axs[0],base_dec,base_patch,'Direct linear: red chosen / blue counterfactual / outlined held-out')
violin_panel(axs[1],iter_dec,iter_patch,'Iterative linear model: same random held-out decisions')
ticks=np.arange(0,n,max(1,n//20));axs[1].set_xticks(ticks,[str(D[t]['step']) for t in ticks]);axs[1].set_xlabel('Decision step')
plt.tight_layout();plt.show()

# Figure 2: reward and ranking error over the complete trace.
fig,axs=plt.subplots(3,1,figsize=(15,10),sharex=True)
for df,name,color in [(base_dec,'Direct','gray'),(iter_dec,'Iterative','firebrick')]:
    axs[0].plot(df.t,df.chosen,color=color,label=f'{name}: chosen',alpha=.85)
    axs[0].plot(df.t,df.rejected_mean,color=color,linestyle=':',label=f'{name}: mean rejected',alpha=.65)
    axs[1].plot(df.t,df.margin,color=color,label=name)
    axs[2].plot(df.t,df.concordance,color=color,label=name)
for ax in axs:
    for t in test_idx: ax.axvline(t,color='black',alpha=.045,lw=.9)
    ax.grid(alpha=.15);ax.legend(loc='best')
axs[0].set_ylabel('Inferred reward');axs[1].set_ylabel('Chosen − best rejected')
axs[1].axhline(0,color='black',lw=.8,ls='--');axs[2].set_ylabel('Pairwise concordance');axs[2].set_ylim(-.04,1.04)
axs[2].set_xticks(ticks,[str(D[t]['step']) for t in ticks]);axs[2].set_xlabel('Decision step (faint lines = random test steps)')
plt.tight_layout();plt.show()

# Figure 3: algorithm convergence + descriptor coefficient comparison.
fig,axs=plt.subplots(1,2,figsize=(13,4.6))
h=pd.DataFrame(history)
axs[0].semilogy(h.iteration,h.coefficient_change,'o-');axs[0].set(xlabel='Iteration',ylabel='Relative coefficient change',title='Alternating model convergence')
y=np.arange(p);axs[1].barh(y-.2,w_direct,height=.4,label='Direct');axs[1].barh(y+.2,w_iter,height=.4,label='Iterative')
axs[1].set_yticks(y,FEATURES);axs[1].legend();axs[1].set_title('Linear reward coefficients')
for ax in axs:ax.grid(alpha=.15)
plt.tight_layout();plt.show()

# Figure 4: actual selected latent utility in red, rejected in blue, vs iteration.
for t in tracked:
    arr=np.stack(latent_tracks[int(t)])
    fig,ax=plt.subplots(figsize=(9,4.2));a=D[int(t)]['selected']
    for j in range(arr.shape[1]):
        ax.plot(np.arange(arr.shape[0]),arr[:,j],color='red' if j==a else 'royalblue',
                lw=2.3 if j==a else .9,alpha=1 if j==a else .25)
    ax.set(xlabel='Alternating iteration',ylabel='Reconstructed latent reward',
           title=f'Training step {D[int(t)]["step"]}: chosen red, rejected blue')
    ax.grid(alpha=.15);plt.tight_layout();plt.show()

# Optional map on original AFM image. No hidden choices needed; descriptor scaler fitted on training only.
spatial_df=None
if MAKE_SPATIAL_MAP:
    if 'img' not in globals():
        print('Spatial map skipped: run original notebook image cell (img) first.')
    else:
        image=np.asarray(img,dtype=float).squeeze()
        ph,pw=D[0]['patches'][0].shape
        if image.ndim!=2 or ph>image.shape[0] or pw>image.shape[1]:
            print('Spatial map skipped: invalid image / patch dimensions')
        else:
            ys=np.unique(np.r_[np.arange(0,image.shape[0]-ph+1,MAP_STRIDE),image.shape[0]-ph])
            xs=np.unique(np.r_[np.arange(0,image.shape[1]-pw+1,MAP_STRIDE),image.shape[1]-pw])
            feature_rows=[]
            for y0 in ys:
                for x0 in xs: feature_rows.append(describe(image[y0:y0+ph,x0:x0+pw]))
            Zmap=np.nan_to_num(scaler.transform(np.vstack(feature_rows)))
            r0=(Zmap@w_direct).reshape(len(ys),len(xs))
            r1=(Zmap@w_iter).reshape(len(ys),len(xs))
            delta=r1-r0
            fig,axs=plt.subplots(1,3,figsize=(17,5))
            for ax,field,title in zip(axs,[r0,r1,delta],['Direct reward','Iterative reward','Iterative − direct']):
                im=ax.imshow(field,origin='upper',extent=[xs[0]+pw/2,xs[-1]+pw/2,ys[-1]+ph/2,ys[0]+ph/2],aspect='equal')
                ax.set_title(title);ax.set_xlabel('Image x (px)');ax.set_ylabel('Image y (px)');fig.colorbar(im,ax=ax,shrink=.8)
            plt.tight_layout();plt.show()
            spatial_df=pd.DataFrame(dict(x=np.tile(xs+pw/2,len(ys)),y=np.repeat(ys+ph/2,len(xs)),
                                         reward_direct=r0.ravel(),reward_iterative=r1.ravel(),difference=delta.ravel()))
            print('Mapped',len(spatial_df),'full-image patch locations at stride',MAP_STRIDE)

if SAVE_RESULTS:
    out=Path(f'day20_selfcontained_trace_{TRACE_ID}');out.mkdir(exist_ok=True)
    pd.concat([base_dec,iter_dec]).to_csv(out/'decisions.csv',index=False)
    pd.concat([base_patch,iter_patch]).to_csv(out/'patch_rewards.csv',index=False)
    h.to_csv(out/'iterations.csv',index=False)
    pd.DataFrame({'descriptor':FEATURES,'direct':w_direct,'iterative':w_iter}).to_csv(out/'coefficients.csv',index=False)
    if spatial_df is not None:spatial_df.to_csv(out/'spatial_rewards.csv',index=False)
    print('Saved outputs to',out.resolve())

print('Done. Latent rewards were reconstructed only for training decisions.')
print('Test rankings use transferable Z @ w predictions; no hidden-label leakage.')
print('This is a posterior-mode alternating approximation, not exact EM.')


In [ ]:

# ================================================================
# ATHENA DAY 20 — FIT MISSING ITERATIVE REWARD MODEL
#
# Uses existing:
#   D
#   w_direct / direct_w / w
#   train_idx
#
# Outputs:
#   w_em
#   initial_w
#   em_history_df
#
# Then rerun:
#   1. DataFrame reconstruction cell
#   2. B/C/D visualization cell
# ================================================================

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import softmax, logsumexp

# Configuration
EM_SIGMA = 0.7
EM_TAU = 1.0
EM_L2 = 0.10
EM_MAX_ITER = 35
EM_TOL = 1e-5

assert "D" in globals(), "Decision data D is missing."
assert "train_idx" in globals(), "train_idx is missing."

# Recover initial direct coefficients
if "w_direct" in globals():
    initial_w = np.asarray(w_direct, float).copy()
elif "direct_w" in globals():
    initial_w = np.asarray(direct_w, float).copy()
elif "w" in globals():
    initial_w = np.asarray(w, float).copy()
else:
    raise ValueError("No fitted direct reward coefficients found.")

initial_w = initial_w.ravel()

train_ids = np.asarray(train_idx, dtype=int).ravel()

if (
    np.any(train_ids < 0)
    or np.any(train_ids >= len(D))
):
    raise ValueError(
        "train_idx contains indices outside D. "
        "Recover the original split before fitting."
    )

if len(np.unique(train_ids)) != len(train_ids):
    raise ValueError("Duplicate training indices.")

# Collect training descriptors
X = np.vstack([
    np.asarray(D[i]["Z"], float)
    for i in train_ids
])

assert X.shape[1] == len(initial_w)

# ---------------------------------------------------------------
# E-like step: infer latent rewards
# ---------------------------------------------------------------

def estimate_latent_rewards_em(decision, coefficients):

    Z = np.asarray(decision["Z"], float)
    selected = int(decision["selected"])

    prior_mean = Z @ coefficients
    sigma2 = EM_SIGMA**2

    target = np.zeros(len(prior_mean))
    target[selected] = 1.0

    def objective(r):

        probs = softmax(r / EM_TAU)

        loss = (
            0.5 * np.sum((r - prior_mean)**2) / sigma2
            + logsumexp(r / EM_TAU)
            - r[selected] / EM_TAU
        )

        grad = (
            (r - prior_mean) / sigma2
            + (probs - target) / EM_TAU
        )

        return loss, grad

    result = minimize(
        objective,
        prior_mean.copy(),
        method="L-BFGS-B",
        jac=True
    )

    if not result.success and np.linalg.norm(result.jac) > 1e-4:
        raise RuntimeError(result.message)

    return result.x

# ---------------------------------------------------------------
# Alternating E/M updates
# ---------------------------------------------------------------

w_em = initial_w.copy()

N = len(X)

A = (
    X.T @ X / N
    + EM_L2 * EM_SIGMA**2 * np.eye(X.shape[1])
)

em_history = []

for iteration in range(EM_MAX_ITER):

    # Estimate latent rewards for training decisions
    latent_rewards = np.concatenate([
        estimate_latent_rewards_em(D[i], w_em)
        for i in train_ids
    ])

    # Fit systematic linear reward model
    b = X.T @ latent_rewards / N

    updated_w = np.linalg.solve(A, b)

    change = (
        np.linalg.norm(updated_w - w_em)
        /
        (1 + np.linalg.norm(w_em))
    )

    em_history.append({
        "iteration": iteration + 1,
        "coefficient_change": change,
        "coefficient_norm": np.linalg.norm(updated_w)
    })

    w_em = updated_w

    print(
        f"Iteration {iteration + 1:2d}: "
        f"relative change = {change:.6e}"
    )

    if change < EM_TOL:
        break

em_history_df = pd.DataFrame(em_history)

# Aliases used by visualization
w_iterative = w_em.copy()
em_w = w_em.copy()

print("\nIterative reward model fitted successfully.")
print("Direct coefficients:", initial_w.shape)
print("Iterative coefficients:", w_em.shape)
print("Iterations:", len(em_history_df))
print(
    "Coefficient difference:",
    np.linalg.norm(w_em - initial_w)
)


In [ ]:

# ================================================================
# ATHENA DAY 20 — RECONSTRUCT B/C/D VISUALIZATION INPUTS
#
# Run immediately BEFORE the B/C/D visualization cell.
# Does not refit either model or change train/test assignments.
# ================================================================

import numpy as np
import pandas as pd
from scipy.special import softmax

# ---------------------------------------------------------------
# 1. Locate the fitted models
# ---------------------------------------------------------------

def find_existing(names):
    for name in names:
        if name in globals():
            return globals()[name], name
    return None, None

direct_w, direct_name = find_existing([
    "initial_w",
    "w_direct",
    "direct_w",
    "w"
])

iterative_w, iterative_name = find_existing([
    "w_em",
    "w_iterative",
    "em_w"
])

if direct_w is None or iterative_w is None:
    raise ValueError(
        "Could not find both fitted coefficient vectors. "
        "Run the iterative reward-learning model first. "
        "Available coefficient variables: " +
        str([
            k for k in globals()
            if k in [
                "initial_w", "w_direct", "direct_w",
                "w", "w_em", "w_iterative", "em_w"
            ]
        ])
    )

print("Direct model:", direct_name)
print("Iterative model:", iterative_name)

direct_w = np.asarray(direct_w, dtype=float).ravel()
iterative_w = np.asarray(iterative_w, dtype=float).ravel()

if "D" not in globals():
    raise ValueError("Decision data D not found.")

# ---------------------------------------------------------------
# 2. Recover the original validation split
# ---------------------------------------------------------------

if "test_idx" not in globals():
    raise ValueError(
        "test_idx not found. Do not generate a new split: "
        "the original held-out assignments are needed."
    )

test_set = set(np.asarray(test_idx, dtype=int).tolist())

# ---------------------------------------------------------------
# 3. Construct reward and validation DataFrames
# ---------------------------------------------------------------

def reconstruct_results(coefficients, model_name):

    decision_rows = []
    patch_rows = []

    for t, d in enumerate(D):

        Z = np.asarray(d["Z"], dtype=float)

        selected = int(d["selected"])

        if Z.shape[1] != len(coefficients):
            raise ValueError(
                f"Feature dimension mismatch at step {t}: "
                f"{Z.shape[1]} vs {len(coefficients)}"
            )

        rewards = Z @ coefficients
        probabilities = softmax(rewards)

        alternatives = np.delete(rewards, selected)

        chosen_reward = rewards[selected]

        predicted = int(np.argmax(rewards))

        split = "test" if t in test_set else "train"

        concordance = (
            np.mean(chosen_reward > alternatives)
            + 0.5 * np.mean(chosen_reward == alternatives)
        )

        decision_rows.append({
            "model": model_name,
            "position": t,
            "step": d["step"],
            "split": split,
            "actual": selected,
            "predicted": predicted,
            "top1": int(predicted == selected),
            "chosen_reward": chosen_reward,
            "rejected_mean": alternatives.mean(),
            "rejected_max": alternatives.max(),
            "margin": chosen_reward - alternatives.max(),
            "concordance": concordance,
            "rank": 1 + np.sum(rewards > chosen_reward),
            "neg_log_probability": -np.log(
                max(probabilities[selected], 1e-15)
            )
        })

        for j, reward in enumerate(rewards):

            patch_rows.append({
                "model": model_name,
                "position": t,
                "step": d["step"],
                "split": split,
                "candidate": j,
                "selected": int(j == selected),
                "reward": reward
            })

    return (
        pd.DataFrame(decision_rows),
        pd.DataFrame(patch_rows)
    )


direct_decisions, direct_patches = reconstruct_results(
    direct_w, "Direct"
)

em_decisions, em_patches = reconstruct_results(
    iterative_w, "Iterative"
)

print("\nSuccessfully reconstructed:")
print("direct_decisions:", direct_decisions.shape)
print("direct_patches:", direct_patches.shape)
print("em_decisions:", em_decisions.shape)
print("em_patches:", em_patches.shape)

print("\nHeld-out validation:")
print(
    pd.concat(
        [direct_decisions, em_decisions]
    )
    .query("split == 'test'")
    .groupby("model")[
        ["top1", "rank", "concordance", "margin"]
    ]
    .mean()
    .round(4)
)


In [ ]:
# ================================================================
# ATHENA DAY 20 — VISUALIZATIONS B, C, D
#
# Run AFTER the self-contained iterative reward-learning cell.
#
# B: Selected vs rejected reward distributions across decisions
# C: Ranked selected rewards with monotonicity diagnostics
# D: Spatial reward field over the AFM image
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pathlib import Path

# ---------------------------------------------------------------
# 1. RECOVER RESULTS FROM PREVIOUS CELL
# ---------------------------------------------------------------

OUTDIR = Path("day20_BCD_visualizations")
OUTDIR.mkdir(exist_ok=True)

# Required model outputs
required_globals = [
    "direct_decisions",
    "direct_patches",
    "em_decisions",
    "em_patches",
]

for name in required_globals:
    if name not in globals():
        raise ValueError(
            f"Missing variable '{name}'. "
            "Run the self-contained iterative reward-learning cell first."
        )

direct_decisions_df = direct_decisions.copy()
direct_patches_df = direct_patches.copy()
em_decisions_df = em_decisions.copy()
em_patches_df = em_patches.copy()

# Basic trace information
if "D" not in globals():
    raise ValueError("Missing D. Run the self-contained iterative reward-learning cell first.")

n_decisions = len(D)

# Make sure decision position exists
for df_name, df in [
    ("direct_decisions", direct_decisions_df),
    ("em_decisions", em_decisions_df),
]:
    if "position" not in df.columns:
        if "step" in df.columns:
            # fallback: sort by step
            df = df.sort_values("step").reset_index(drop=True)
            df["position"] = np.arange(len(df))
            if df_name == "direct_decisions":
                direct_decisions_df = df
            else:
                em_decisions_df = df
        else:
            raise ValueError(f"{df_name} needs either 'position' or 'step'.")

for df_name, df in [
    ("direct_patches", direct_patches_df),
    ("em_patches", em_patches_df),
]:
    if "position" not in df.columns:
        if "step" in df.columns:
            # map from decision data
            ref = direct_decisions_df[["step", "position"]].drop_duplicates()
            df = df.merge(ref, on="step", how="left")
            if df["position"].isna().any():
                raise ValueError(f"Could not recover positions for {df_name}.")
            if df_name == "direct_patches":
                direct_patches_df = df
            else:
                em_patches_df = df
        else:
            raise ValueError(f"{df_name} needs either 'position' or 'step'.")

# Sort consistently
direct_decisions_df = direct_decisions_df.sort_values("position").reset_index(drop=True)
em_decisions_df = em_decisions_df.sort_values("position").reset_index(drop=True)
direct_patches_df = direct_patches_df.sort_values(["position", "candidate"]).reset_index(drop=True)
em_patches_df = em_patches_df.sort_values(["position", "candidate"]).reset_index(drop=True)

# Recover held-out positions if available
if "split" in direct_decisions_df.columns:
    heldout_positions = direct_decisions_df.loc[
        direct_decisions_df["split"].astype(str).str.lower() == "test",
        "position"
    ].to_numpy(dtype=int)
else:
    heldout_positions = np.array([], dtype=int)

# Step labels
if "step" in direct_decisions_df.columns:
    step_labels = direct_decisions_df["step"].to_numpy()
else:
    step_labels = np.arange(n_decisions)

# ---------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ---------------------------------------------------------------

def plot_reward_distributions(decision_df, patch_df, title, ax):
    """
    B: Blue violins = rejected candidates
       Red points/line = selected candidate
       Black square = held-out decision
    """
    positions = np.arange(len(decision_df))

    violins = []
    for pos in positions:
        vals = patch_df.loc[
            (patch_df["position"] == pos) & (patch_df["selected"] == 0),
            "reward"
        ].to_numpy()
        violins.append(vals)

    vp = ax.violinplot(
        violins,
        positions=positions,
        widths=0.84,
        showmeans=False,
        showmedians=False,
        showextrema=False
    )

    for body in vp["bodies"]:
        body.set_facecolor("royalblue")
        body.set_edgecolor("royalblue")
        body.set_alpha(0.25)

    y = decision_df["chosen_reward"].to_numpy()

    ax.plot(
        positions, y,
        color="red",
        linewidth=1.0,
        alpha=0.8,
        zorder=4
    )
    ax.scatter(
        positions, y,
        color="red",
        s=16,
        zorder=5
    )

    if len(heldout_positions):
        mask = np.isin(positions, heldout_positions)
        ax.scatter(
            positions[mask], y[mask],
            marker="s",
            facecolors="white",
            edgecolors="black",
            s=45,
            linewidths=1.2,
            zorder=6
        )

    ax.set_title(title)
    ax.set_ylabel("Reward")
    ax.grid(axis="y", alpha=0.15)

def compute_monotonicity_metrics(decision_df):
    """
    For selected-patch rewards along the trace:
      local_violation[t] = max(0, R_t - R_{t-1}) for t>=1
      cumulative_violation[t] = sum of local violations up to t
      running_max_increase = same cumulative measure
    If the trace is 'best remaining patch first', selected rewards
    should be non-increasing.
    """
    rewards = decision_df["chosen_reward"].to_numpy()
    n = len(rewards)

    local_violation = np.zeros(n)
    delta = np.zeros(n)

    for t in range(1, n):
        delta[t] = rewards[t] - rewards[t - 1]
        local_violation[t] = max(0.0, delta[t])

    cumulative_violation = np.cumsum(local_violation)

    # Monotone non-increasing envelope (running minimum)
    monotone_envelope = np.minimum.accumulate(rewards)

    return {
        "rewards": rewards,
        "delta": delta,
        "local_violation": local_violation,
        "cumulative_violation": cumulative_violation,
        "monotone_envelope": monotone_envelope,
    }

def try_get_selected_positions():
    """
    Attempt to recover physical selected coordinates for overlay on panel D.
    This is optional.
    """
    candidate_keys = [
        "selected_position",
        "selected_pos",
        "selected_location",
        "position_xy",
        "xy",
    ]

    coords = []
    for d in D:
        found = None
        for key in candidate_keys:
            if key in d:
                value = d[key]
                arr = np.asarray(value).ravel()
                if len(arr) >= 2:
                    found = arr[:2].astype(float)
                    break
        coords.append(found)

    if all(c is not None for c in coords):
        return np.vstack(coords)

    return None

# ---------------------------------------------------------------
# 3. PANEL B — SELECTED VS REJECTED REWARD DISTRIBUTIONS
# ---------------------------------------------------------------

fig, axs = plt.subplots(
    2, 1,
    figsize=(max(16, 0.18 * n_decisions), 9),
    sharex=True
)

plot_reward_distributions(
    direct_decisions_df,
    direct_patches_df,
    "B. Direct Linear Model: Selected vs Rejected Reward Distributions",
    axs[0]
)

plot_reward_distributions(
    em_decisions_df,
    em_patches_df,
    "B. Iterative Latent-Reward Model: Selected vs Rejected Reward Distributions",
    axs[1]
)

tick_stride = max(1, n_decisions // 20)

axs[1].set_xticks(
    np.arange(0, n_decisions, tick_stride),
    [str(step_labels[i]) for i in range(0, n_decisions, tick_stride)]
)
axs[1].set_xlabel("Decision step (black squares = held-out decisions)")

legend_handles = [
    Patch(facecolor="royalblue", edgecolor="royalblue", alpha=0.25, label="Rejected candidates"),
    Line2D([0], [0], color="red", marker="o", label="Actual selected patch"),
    Line2D([0], [0], marker="s", linestyle="None",
           markerfacecolor="white", markeredgecolor="black",
           label="Held-out decision"),
]

axs[0].legend(handles=legend_handles, loc="best")

plt.tight_layout()
plt.savefig(OUTDIR / "panel_B_reward_distributions.png", dpi=180)
plt.show()

# ---------------------------------------------------------------
# 4. PANEL C — RANKED SELECTED REWARDS + MONOTONICITY DIAGNOSTICS
# ---------------------------------------------------------------

direct_mono = compute_monotonicity_metrics(direct_decisions_df)
em_mono = compute_monotonicity_metrics(em_decisions_df)

fig, axs = plt.subplots(
    3, 1,
    figsize=(max(16, 0.18 * n_decisions), 11),
    sharex=True,
    constrained_layout=True
)

x = np.arange(n_decisions)

# C1: Selected rewards along trace, with monotone envelope
axs[0].plot(
    x, direct_mono["rewards"],
    color="gray", linewidth=1.2, label="Direct model"
)
axs[0].plot(
    x, em_mono["rewards"],
    color="firebrick", linewidth=1.2, label="Iterative model"
)

axs[0].plot(
    x, direct_mono["monotone_envelope"],
    color="gray", linestyle="--", alpha=0.8, label="Direct non-increasing envelope"
)
axs[0].plot(
    x, em_mono["monotone_envelope"],
    color="firebrick", linestyle="--", alpha=0.8, label="Iterative non-increasing envelope"
)

# Highlight local monotonicity violations
direct_violation_idx = np.where(direct_mono["local_violation"] > 0)[0]
em_violation_idx = np.where(em_mono["local_violation"] > 0)[0]

axs[0].scatter(
    direct_violation_idx,
    direct_mono["rewards"][direct_violation_idx],
    color="gray", marker="^", s=36, zorder=5
)
axs[0].scatter(
    em_violation_idx,
    em_mono["rewards"][em_violation_idx],
    color="firebrick", marker="^", s=36, zorder=5
)

if len(heldout_positions):
    axs[0].scatter(
        heldout_positions,
        em_mono["rewards"][heldout_positions],
        facecolors="white",
        edgecolors="black",
        marker="s",
        s=40,
        zorder=6
    )

axs[0].set_title("C. Selected Rewards Along the Trace")
axs[0].set_ylabel("Selected-patch reward")
axs[0].legend(loc="best")
axs[0].grid(alpha=0.15)

# C2: local monotonicity violations
axs[1].bar(
    x - 0.18,
    direct_mono["local_violation"],
    width=0.36,
    color="gray",
    alpha=0.7,
    label="Direct"
)
axs[1].bar(
    x + 0.18,
    em_mono["local_violation"],
    width=0.36,
    color="firebrick",
    alpha=0.7,
    label="Iterative"
)

if len(heldout_positions):
    ymax = max(
        np.max(direct_mono["local_violation"]) if len(direct_mono["local_violation"]) else 0,
        np.max(em_mono["local_violation"]) if len(em_mono["local_violation"]) else 0,
        1e-9
    )
    axs[1].scatter(
        heldout_positions,
        np.full_like(heldout_positions, ymax * 0.95, dtype=float),
        facecolors="white",
        edgecolors="black",
        marker="s",
        s=30,
        zorder=6
    )

axs[1].set_title("Local Monotonicity Violation")
axs[1].set_ylabel("max(0, R_t − R_{t−1})")
axs[1].legend(loc="best")
axs[1].grid(alpha=0.15)

# C3: cumulative monotonicity violation
axs[2].plot(
    x, direct_mono["cumulative_violation"],
    color="gray", linewidth=1.5, label="Direct"
)
axs[2].plot(
    x, em_mono["cumulative_violation"],
    color="firebrick", linewidth=1.5, label="Iterative"
)

if len(heldout_positions):
    axs[2].scatter(
        heldout_positions,
        em_mono["cumulative_violation"][heldout_positions],
        facecolors="white",
        edgecolors="black",
        marker="s",
        s=40,
        zorder=6
    )

axs[2].set_title("Cumulative Monotonicity Violation")
axs[2].set_ylabel("Cumulative violation")
axs[2].set_xlabel("Decision step")
axs[2].legend(loc="best")
axs[2].grid(alpha=0.15)

axs[2].set_xticks(
    np.arange(0, n_decisions, tick_stride),
    [str(step_labels[i]) for i in range(0, n_decisions, tick_stride)]
)

plt.savefig(OUTDIR / "panel_C_monotonicity.png", dpi=180)
plt.show()

# Monotonicity summary
mono_summary = pd.DataFrame({
    "model": ["Direct", "Iterative"],
    "n_local_violations": [
        int(np.sum(direct_mono["local_violation"] > 0)),
        int(np.sum(em_mono["local_violation"] > 0)),
    ],
    "mean_local_violation": [
        float(np.mean(direct_mono["local_violation"])),
        float(np.mean(em_mono["local_violation"])),
    ],
    "total_cumulative_violation": [
        float(direct_mono["cumulative_violation"][-1]),
        float(em_mono["cumulative_violation"][-1]),
    ],
})
print("\nMonotonicity summary:")
print(mono_summary.to_string(index=False))

# ---------------------------------------------------------------
# 5. PANEL D — SPATIAL REWARD FIELD OVER THE AFM IMAGE
# ---------------------------------------------------------------

# Try to recover spatial results from previous cell
has_spatial = all(name in globals() for name in [
    "spatial_centers",
    "reward_initial",
    "reward_iterative",
    "reward_difference",
])

if not has_spatial:
    print(
        "\nPanel D not generated: spatial reward arrays not found.\n"
        "Run the self-contained iterative cell with MAKE_SPATIAL_MAP = True."
    )
else:
    centers = np.asarray(spatial_centers)
    reward_direct = np.asarray(reward_initial)
    reward_iter = np.asarray(reward_iterative)
    reward_diff = np.asarray(reward_difference)

    unique_x = np.unique(centers[:, 0])
    unique_y = np.unique(centers[:, 1])

    grid_shape = (len(unique_y), len(unique_x))

    if np.prod(grid_shape) != len(reward_direct):
        raise ValueError("Spatial reward arrays do not match the spatial grid.")

    direct_map = reward_direct.reshape(grid_shape)
    iterative_map = reward_iter.reshape(grid_shape)
    difference_map = reward_diff.reshape(grid_shape)

    if "img" not in globals():
        raise ValueError("Original AFM image 'img' not found.")
    afm_image = np.asarray(img).squeeze()

    extent = (
        unique_x.min(),
        unique_x.max(),
        unique_y.max(),
        unique_y.min()
    )

    selected_xy = try_get_selected_positions()

    fig, axs = plt.subplots(
        1, 4,
        figsize=(20, 5),
        constrained_layout=True
    )

    # Original image
    im0 = axs[0].imshow(afm_image, cmap="gray", aspect="auto")
    axs[0].set_title("D. Original AFM image")
    axs[0].set_xlabel("x (pixels)")
    axs[0].set_ylabel("y (pixels)")

    if selected_xy is not None:
        axs[0].scatter(
            selected_xy[:, 0],
            selected_xy[:, 1],
            s=18,
            facecolors="none",
            edgecolors="red",
            linewidths=0.8
        )

    plt.colorbar(im0, ax=axs[0], shrink=0.8)

    # Direct reward map
    im1 = axs[1].imshow(
        direct_map,
        extent=extent,
        origin="upper",
        aspect="auto",
        cmap="viridis"
    )
    axs[1].set_title("Direct reward field")
    axs[1].set_xlabel("x (pixels)")
    axs[1].set_ylabel("y (pixels)")
    if selected_xy is not None:
        axs[1].scatter(
            selected_xy[:, 0],
            selected_xy[:, 1],
            s=18,
            facecolors="none",
            edgecolors="red",
            linewidths=0.8
        )
    plt.colorbar(im1, ax=axs[1], shrink=0.8)

    # Iterative reward map
    im2 = axs[2].imshow(
        iterative_map,
        extent=extent,
        origin="upper",
        aspect="auto",
        cmap="viridis"
    )
    axs[2].set_title("Iterative reward field")
    axs[2].set_xlabel("x (pixels)")
    axs[2].set_ylabel("y (pixels)")
    if selected_xy is not None:
        axs[2].scatter(
            selected_xy[:, 0],
            selected_xy[:, 1],
            s=18,
            facecolors="none",
            edgecolors="red",
            linewidths=0.8
        )
    plt.colorbar(im2, ax=axs[2], shrink=0.8)

    # Difference map
    vmax = np.max(np.abs(difference_map))
    im3 = axs[3].imshow(
        difference_map,
        extent=extent,
        origin="upper",
        aspect="auto",
        cmap="coolwarm",
        vmin=-vmax,
        vmax=vmax
    )
    axs[3].set_title("Iterative − Direct")
    axs[3].set_xlabel("x (pixels)")
    axs[3].set_ylabel("y (pixels)")
    if selected_xy is not None:
        axs[3].scatter(
            selected_xy[:, 0],
            selected_xy[:, 1],
            s=18,
            facecolors="none",
            edgecolors="black",
            linewidths=0.8
        )
    plt.colorbar(im3, ax=axs[3], shrink=0.8)

    plt.savefig(OUTDIR / "panel_D_spatial_reward_field.png", dpi=180)
    plt.show()

# ---------------------------------------------------------------
# 6. EXPORT SUMMARY TABLES
# ---------------------------------------------------------------

panelB_summary = pd.concat([
    direct_decisions_df.assign(model="Direct"),
    em_decisions_df.assign(model="Iterative")
], ignore_index=True)

panelB_summary.to_csv(OUTDIR / "decision_level_summary_BCD.csv", index=False)
mono_summary.to_csv(OUTDIR / "monotonicity_summary.csv", index=False)

print("\nSaved outputs to:")
print(OUTDIR.resolve())

In [ ]:
# ATHENA Day 20: All-trace inverse reward analysis
# Paste as ONE new cell after original notebook data-loading cells (traces and img).
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.ndimage import sobel, laplace
from scipy.stats import skew, kurtosis
from scipy.special import logsumexp, softmax
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler
from pathlib import Path

TEST_FRACTION = 0.20
SEED = 42
RIDGE = 0.10
LATENT_SD = 0.70
TAU = 1.0
EM_ITERATIONS = 18
EM_TOL = 1e-5
GRID_STRIDE = 16  # sparse spatial reward grid
EXCLUSION_RADIUS_PIXELS = None  # default half patch width; avoid same region in proposed path
DISPLAY_TRACE_DETAILS = True
OUT = Path('day20_all_trace_results'); OUT.mkdir(exist_ok=True)

assert 'traces' in globals() and len(traces), 'Run original notebook trace-loading cell first.'
FEATURE_NAMES=['mean','std','p05','p95','contrast','skew','kurtosis','gradient_mean',
               'gradient_std','gradient_energy','laplacian','entropy','highfreq',
               'anisotropy','corr_x','corr_y']

def descriptor(patch):
    a=np.asarray(patch,dtype=float).squeeze()
    if a.ndim!=2: raise ValueError(f'Patch must be 2D, got {a.shape}')
    valid=a[np.isfinite(a)]
    if not valid.size: raise ValueError('Patch is entirely non-finite')
    a=np.where(np.isfinite(a),a,np.median(valid))
    s=float(a.std()); z=(a-a.mean())/(s+1e-9)
    gx=sobel(z,axis=1)/8.; gy=sobel(z,axis=0)/8.; grad=np.hypot(gx,gy)
    power=np.abs(np.fft.fftshift(np.fft.fft2(z)))**2
    fx=np.fft.fftshift(np.fft.fftfreq(a.shape[1]))[None,:]
    fy=np.fft.fftshift(np.fft.fftfreq(a.shape[0]))[:,None]
    hist=np.histogram(z,bins=32,range=(-4,4))[0]
    prob=hist[hist>0]/max(1,hist.sum())
    values=[a.mean(),s,np.percentile(a,5),np.percentile(a,95),
            np.percentile(a,95)-np.percentile(a,5),
            skew(a.ravel()) if s>1e-9 else 0,
            kurtosis(a.ravel()) if s>1e-9 else 0,
            grad.mean(),grad.std(),(grad**2).mean(),np.abs(laplace(z)).mean(),
            -np.sum(prob*np.log(prob))/np.log(32),
            power[np.hypot(fx,fy)>.25].sum()/(power.sum()+1e-9),
            np.abs(gx).mean()-np.abs(gy).mean(),
            (z[:,:-1]*z[:,1:]).mean() if z.shape[1]>1 else 0,
            (z[:-1,:]*z[1:,:]).mean() if z.shape[0]>1 else 0]
    return np.nan_to_num(np.array(values,dtype=float),nan=0,posinf=0,neginf=0)

def take_position(value):
    if value is None: return (np.nan,np.nan)
    try:
        if isinstance(value,dict):
            if 'x' in value and 'y' in value: return (float(value['x']),float(value['y']))
            return (np.nan,np.nan)
        arr=np.asarray(value,dtype=float).ravel()
        return (float(arr[0]),float(arr[1])) if arr.size>=2 else (np.nan,np.nan)
    except (TypeError,ValueError): return (np.nan,np.nan)

# Cache patch descriptors by bytes and shape across trajectories to reduce repeated work.
feature_cache={}
def cached_descriptor(p):
    a=np.asarray(p,dtype=np.float32).squeeze()
    key=(a.shape,a.tobytes())
    if key not in feature_cache: feature_cache[key]=descriptor(a)
    return feature_cache[key]

image=np.asarray(img,dtype=float).squeeze() if 'img' in globals() else None
if image is not None and image.ndim!=2: image=None
trace_ids=list(traces.keys())
prepared={}
for trace_id in trace_ids:
    tr=traces[trace_id]
    step_ids=sorted(tr.keys()) if isinstance(tr,dict) else range(len(tr))
    steps=[]
    for step_id in step_ids:
        record=tr[step_id]
        patches=[np.asarray(p).squeeze() for p in record['patches']]
        a=int(record['selected_candidate'])
        if len(patches)<2 or a<0 or a>=len(patches): raise ValueError(f'Trace {trace_id}, step {step_id}: invalid choice')
        cx,cy=take_position(record.get('position'))
        steps.append(dict(step=step_id,selected=a,X=np.vstack([cached_descriptor(p) for p in patches]),
                          patches=patches,actual_x=cx,actual_y=cy))
    if len(steps)<5: print(f'Trace {trace_id}: skipped, fewer than 5 decisions');continue
    prepared[trace_id]=steps
print('Prepared',len(prepared),'traces; unique patch descriptors:',len(feature_cache))

# Spatial descriptors are image-dependent, not trace-dependent; standardize separately per trace.
grid=None
if image is not None and prepared:
    first=next(iter(prepared.values()))[0]['patches'][0]
    ph,pw=first.shape
    if ph<=image.shape[0] and pw<=image.shape[1]:
        ys=np.unique(np.r_[np.arange(0,image.shape[0]-ph+1,GRID_STRIDE),image.shape[0]-ph])
        xs=np.unique(np.r_[np.arange(0,image.shape[1]-pw+1,GRID_STRIDE),image.shape[1]-pw])
        grid_desc=np.vstack([cached_descriptor(image[y:y+ph,x:x+pw]) for y in ys for x in xs])
        grid_xy=np.array([(x+pw/2,y+ph/2) for y in ys for x in xs],float)
        grid=dict(shape=(len(ys),len(xs)),features=grid_desc,xy=grid_xy,patch_shape=(ph,pw))
        print('Sparse reward grid:',grid['shape'],'=',len(grid_xy),'locations')
    else: print('Full image smaller than patch; spatial maps skipped.')
else: print('No full img array; spatial maps skipped.')

def fit_one(trace_id,decisions):
    n=len(decisions)
    rng=np.random.default_rng(SEED+int(trace_id))
    perm=rng.permutation(n);nt=max(1,int(round(n*TEST_FRACTION)))
    test=set(map(int,perm[:nt])); train=[i for i in range(n) if i not in test]
    scaler=StandardScaler().fit(np.vstack([decisions[i]['X'] for i in train]))
    for d in decisions:d['Z']=np.nan_to_num(scaler.transform(d['X']),nan=0,posinf=0,neginf=0)
    p=len(FEATURE_NAMES)
    def direct_obj(w):
        loss=0.;grad=np.zeros(p)
        for idx in train:
            d=decisions[idx]; Z=d['Z'];a=d['selected'];s=Z@w/TAU;q=softmax(s)
            loss+=logsumexp(s)-s[a]
            grad+=Z.T@q/TAU-Z[a]/TAU
        return loss/len(train)+.5*RIDGE*(w@w),grad/len(train)+RIDGE*w
    sol=minimize(direct_obj,np.zeros(p),jac=True,method='L-BFGS-B',options={'maxiter':500})
    if not sol.success: print(f'Trace {trace_id}: direct fit warning:',sol.message)
    w_direct=sol.x.copy();w_em=w_direct.copy()
    X=np.vstack([decisions[i]['Z'] for i in train]);N=X.shape[0]
    A=X.T@X/N+RIDGE*LATENT_SD**2*np.eye(p)
    history=[]
    for iteration in range(EM_ITERATIONS):
        latents=[]
        for idx in train:
            d=decisions[idx];Z=d['Z'];a=d['selected'];m=Z@w_em
            def latent_obj(r):
                q=softmax(r/TAU)
                f=.5*np.sum((r-m)**2)/LATENT_SD**2+logsumexp(r/TAU)-r[a]/TAU
                g=(r-m)/LATENT_SD**2+(q-np.eye(len(q))[a])/TAU
                return f,g
            res=minimize(latent_obj,m,jac=True,method='L-BFGS-B',options={'maxiter':120})
            latents.append(res.x)
        new_w=np.linalg.solve(A,(X.T@np.concatenate(latents))/N)
        change=np.linalg.norm(new_w-w_em)/(1+np.linalg.norm(w_em))
        history.append(dict(trace=trace_id,iteration=iteration+1,change=change))
        w_em=new_w
        if change<EM_TOL:break
    rows=[];patch_rows=[]
    for model,w in [('Direct',w_direct),('Iterative',w_em)]:
        for t,d in enumerate(decisions):
            r=d['Z']@w;a=d['selected'];other=np.delete(r,a);q=softmax(r/TAU)
            rows.append(dict(trace=trace_id,model=model,t=t,step=d['step'],split='test' if t in test else 'train',
                            selected=a,predicted=int(np.argmax(r)),accuracy=int(a==np.argmax(r)),
                            rank=1+int(np.sum(r>r[a])),concordance=float(np.mean(r[a]>other)+.5*np.mean(r[a]==other)),
                            nll=float(-np.log(max(q[a],1e-15))),chosen_reward=float(r[a]),
                            rejected_mean=float(other.mean()),rejected_max=float(other.max()),
                            margin=float(r[a]-other.max()),actual_x=d['actual_x'],actual_y=d['actual_y']))
            for j,v in enumerate(r):patch_rows.append(dict(trace=trace_id,model=model,t=t,step=d['step'],
                split='test' if t in test else 'train',candidate=j,selected=int(j==a),reward=float(v)))
    return dict(decision=pd.DataFrame(rows),patch=pd.DataFrame(patch_rows),history=pd.DataFrame(history),
                w_direct=w_direct,w_em=w_em,scaler=scaler,test=test)

results={}
for trace_id,decisions in prepared.items():
    print('Fitting trace',trace_id,'...',flush=True)
    results[trace_id]=fit_one(trace_id,decisions)

all_dec=pd.concat([r['decision'] for r in results.values()],ignore_index=True)
all_patch=pd.concat([r['patch'] for r in results.values()],ignore_index=True)
all_hist=pd.concat([r['history'] for r in results.values()],ignore_index=True)
metrics=['accuracy','rank','concordance','nll','margin']
summary=(all_dec[all_dec.split=='test'].groupby(['trace','model'])[metrics].mean().reset_index())
print('\nHeld-out performance by trace:');print(summary.round(3).to_string(index=False))
summary.to_csv(OUT/'heldout_summary.csv',index=False);all_dec.to_csv(OUT/'decisions.csv',index=False)
all_patch.to_csv(OUT/'patch_rewards.csv',index=False);all_hist.to_csv(OUT/'iterations.csv',index=False)

# B: per-trace statistics, with honest held-out comparison.
fig,axs=plt.subplots(2,2,figsize=(15,9),constrained_layout=True)
for ax,metric,title in zip(axs.ravel(),['accuracy','concordance','rank','nll'],
                           ['Held-out top-1 accuracy ↑','Held-out pairwise concordance ↑','Held-out chosen rank ↓','Held-out negative log-likelihood ↓']):
    pt=summary.pivot(index='trace',columns='model',values=metric).reindex(results.keys())
    pos=np.arange(len(pt));width=.37
    for offset,name in [(-width/2,'Direct'),(width/2,'Iterative')]:
        if name in pt:ax.bar(pos+offset,pt[name],width,label=name)
    ax.set_xticks(pos,[str(t) for t in pt.index]);ax.set_xlabel('Trace ID');ax.set_title(title);ax.grid(axis='y',alpha=.18)
    if metric=='accuracy':ax.axhline(.05,ls='--',color='gray',lw=1,label='Random (20 candidates)')
axs[0,0].legend();fig.savefig(OUT/'all_trace_statistics.png',dpi=170);plt.show()

# Show per-trace decisions: selected red against rejected blue, and actual/predicted candidate sequence.
if DISPLAY_TRACE_DETAILS:
    for trace_id,r in results.items():
        de=r['decision'];pa=r['patch'];n=len(prepared[trace_id]);xx=np.arange(n)
        fig,axs=plt.subplots(2,2,figsize=(16,7.5),constrained_layout=True)
        for col,model in enumerate(['Direct','Iterative']):
            ds=de[de.model==model].sort_values('t')
            ps=pa[pa.model==model]
            rejected=[ps[(ps.t==t)&(ps.selected==0)].reward.to_numpy() for t in xx]
            vp=axs[0,col].violinplot(rejected,positions=xx,widths=.8,showmeans=False,showmedians=False,showextrema=False)
            for body in vp['bodies']:body.set_facecolor('royalblue');body.set_alpha(.24)
            axs[0,col].plot(xx,ds.chosen_reward,color='red',lw=1)
            held=ds[ds.split=='test'];axs[0,col].scatter(held.t,held.chosen_reward,s=22,facecolors='white',edgecolors='black',marker='s',zorder=5)
            axs[0,col].set_title(f'Trace {trace_id} — {model} | red chosen / blue rejected');axs[0,col].set_ylabel('Reward')
            axs[1,col].plot(xx,ds.selected,color='firebrick',lw=1,label='Actual candidate index')
            axs[1,col].plot(xx,ds.predicted,color='royalblue',lw=.8,alpha=.85,label='Model-proposed index')
            axs[1,col].scatter(held.t,held.selected,s=20,facecolors='white',edgecolors='black',marker='s',zorder=5)
            axs[1,col].set_xlabel('Decision step');axs[1,col].set_ylabel('Candidate index');axs[1,col].legend(fontsize=8)
            for row in (0,1):axs[row,col].grid(alpha=.15)
        fig.savefig(OUT/f'trace_{trace_id}_choices.png',dpi=160);plt.show()

# C/D: sparse spatial maps, and recorded vs proposed spatial paths.
# Proposed spatial path is a NEW GREEDY policy across a fixed sparse grid,
# not the original agent's candidate set or an exact counterfactual replay.
def greedy_path(rewards,xy,count,radius):
    available=np.ones(len(rewards),bool);chosen=[]
    for _ in range(min(count,len(rewards))):
        if not available.any():break
        rank=np.where(available,rewards,-np.inf);j=int(np.argmax(rank))
        chosen.append(j)
        sep=np.sum((xy-xy[j])**2,axis=1)**.5
        available &= sep>=radius
    return np.array(chosen,dtype=int)

spatial_rows=[]
if grid is not None:
    # Pixel grid is common, but patch dimensions must match each trace.
    for trace_id,r in results.items():
        decisions=prepared[trace_id]
        if decisions[0]['patches'][0].shape!=grid['patch_shape']:
            print('Trace',trace_id,'patch shape differs; spatial grid skipped');continue
        Z=np.nan_to_num(r['scaler'].transform(grid['features']),nan=0,posinf=0,neginf=0)
        actual=np.asarray([[d['actual_x'],d['actual_y']] for d in decisions])
        valid=np.isfinite(actual).all(axis=1)
        radius=(grid['patch_shape'][1]/2) if EXCLUSION_RADIUS_PIXELS is None else EXCLUSION_RADIUS_PIXELS
        for model,w in [('Direct',r['w_direct']),('Iterative',r['w_em'])]:
            values=Z@w; field=values.reshape(grid['shape'])
            proposed_idx=greedy_path(values,grid['xy'],len(decisions),radius)
            predicted_xy=grid['xy'][proposed_idx]
            for k,j in enumerate(proposed_idx):
                spatial_rows.append(dict(trace=trace_id,model=model,order=k,proposed_x=grid['xy'][j,0],
                                         proposed_y=grid['xy'][j,1],proposed_reward=values[j]))
            fig,axs=plt.subplots(1,2,figsize=(11,5),constrained_layout=True)
            axs[0].imshow(image,cmap='gray',origin='upper')
            if valid.any():
                av=actual[valid];axs[0].plot(av[:,0],av[:,1],'-o',color='crimson',ms=2,lw=.8,label='Actual path')
                axs[0].scatter(av[0,0],av[0,1],s=45,c='yellow',edgecolors='black',label='Actual start')
            axs[0].set_title(f'Trace {trace_id}: recorded path');axs[0].legend(fontsize=8,loc='best') if valid.any() else None
            ax=axs[1]; im=ax.imshow(field,extent=(grid['xy'][:,0].min(),grid['xy'][:,0].max(),
                      grid['xy'][:,1].max(),grid['xy'][:,1].min()),origin='upper',cmap='viridis',aspect='equal')
            if len(predicted_xy):
                ax.plot(predicted_xy[:,0],predicted_xy[:,1],'-o',color='white',ms=3,lw=.9,label='Proposed greedy path')
                ax.scatter(predicted_xy[0,0],predicted_xy[0,1],s=50,color='yellow',edgecolors='black')
            if valid.any(): ax.scatter(actual[valid,0],actual[valid,1],s=11,facecolors='none',edgecolors='red',label='Actual chosen locations')
            ax.set_title(f'{model}: sparse reward + proposed path');ax.legend(fontsize=8,loc='best')
            fig.colorbar(im,ax=ax,shrink=.75,label='Predicted reward')
            for a in axs:a.set_xlabel('x (pixel)');a.set_ylabel('y (pixel)')
            fig.savefig(OUT/f'trace_{trace_id}_{model.lower()}_spatial_path.png',dpi=160);plt.show()
else: print('Spatial images and paths not created; full image img is required.')
if spatial_rows:pd.DataFrame(spatial_rows).to_csv(OUT/'proposed_spatial_paths.csv',index=False)
print('\nCompleted. Outputs:',OUT.resolve())
print('NOTE: Proposed spatial greedy paths are illustrative new policies, not exact replays of recorded candidate choices.')


Now let's try the latent nonlinear models


# ATHENA Hackathon Day 20: Nonlinear Reward Function Learning from Decision Traces

## 1. Scientific Objective

The objective of this experiment is to reconstruct the decision-making logic of human or AI agents from their recorded selections of microscopy image patches.

At each decision step, an agent is presented with a set of candidate image patches and selects one. The selection reveals information about the agent's underlying preferences, even though the numerical reward assigned to each patch is unknown.

In previous analyses, we approximated the reward using a linear combination of image descriptors:

$$
R(I)=\mathbf w^T\mathbf z(I).
$$

We now investigate whether **nonlinear reward functions** can better reproduce the observed decisions.

Three models are compared:

1. Linear preference learning.
2. Generalized additive models (GAMs) using spline transformations.
3. Gaussian-process-inspired preference learning using random Fourier features (GP-RFF).

The central scientific question is:

**Can nonlinear mappings of interpretable microscopy descriptors recover decision-making criteria more accurately than a linear reward function?**

The analysis also examines whether the inferred reward functions produce physically interpretable spatial preference maps and selection trajectories.

Each decision trace is analyzed independently because the traces may represent different decision-making objectives.

---

## 2. Experimental Data

The analysis operates on the `traces` dictionary and the original microscopy image `img` loaded by the ATHENA Day 20 notebook.

A trace consists of a sequence of decisions:

$$
\mathcal D_q=\{D_{q,1},D_{q,2},\ldots,D_{q,T_q}\},
$$

where $q$ identifies the trace.

At decision step $t$, the agent is presented with candidate patches

$$
\mathcal C_{q,t}=
\{I_{q,t,1},I_{q,t,2},\ldots,I_{q,t,K_t}\}
$$

and selects candidate $a_{q,t}$.

The recorded data provide the candidate images and the selected candidate index.

For every selected patch, the observation implies a relative preference over rejected alternatives.

Under deterministic reward maximization:

$$
R(I_{q,t,a_{q,t}})
\geq R(I_{q,t,j}),
\qquad j\neq a_{q,t}.
$$

However, the implemented models use a probabilistic choice formulation, allowing for stochastic or inconsistent decisions.

### Independent analysis of traces

The code processes every available trace separately.

For each trace, it constructs a new descriptor dataset, training/test split, preprocessing transformation, and fitted reward model.

No reward coefficients are shared across traces.

This makes it possible to investigate how different decision-making strategies correspond to different inferred reward landscapes.

---

## 3. Image Descriptor Representation

Every candidate patch is represented by a vector of 16 numerical descriptors:

$$
\mathbf f(I)=
[f_1(I),f_2(I),\ldots,f_{16}(I)]^T.
$$

The descriptors include:

| Category | Descriptors |
|---|---|
| Intensity | Mean, standard deviation, 5th percentile, 95th percentile, robust contrast |
| Intensity distribution | Skewness, kurtosis |
| Spatial gradients | Mean gradient, gradient standard deviation, gradient energy |
| Spatial curvature | Mean absolute Laplacian |
| Texture | Intensity histogram entropy, high-frequency Fourier power |
| Directionality | Edge anisotropy |
| Spatial correlation | Horizontal and vertical nearest-neighbor correlations |

The descriptors are designed to characterize image appearance using interpretable quantities.

They do not explicitly identify individual domains, defects, particles, or physical phases.

### Standardization

For each trace, descriptor $k$ is standardized according to

$$
z_k(I)=
\frac{f_k(I)-\mu_k}{\sigma_k}.
$$

The means and standard deviations are calculated using only the training candidate patches.

The resulting standardized vector is

$$
\mathbf z(I)=
[z_1(I),\ldots,z_{16}(I)]^T.
$$

The same scaler is applied to held-out patches and spatial patches extracted from the complete microscopy image.

This ensures that evaluation and spatial mapping use the representation learned during training.

---

## 4. Preference-Based Reward Learning

The underlying rewards are not numerically observed.

Instead, we know which candidate was selected from each candidate set.

For a model parameterized by $\theta$, let its reward prediction be

$$
R_\theta(\mathbf z).
$$

We assume that the probability of selecting candidate $i$ is

$$
\boxed{
P(a_t=i\mid\mathcal C_t,\theta)=
\frac{\exp[R_\theta(\mathbf z_{t,i})]}
{\sum_{j=1}^{K_t}\exp[R_\theta(\mathbf z_{t,j})]}.
}
$$

This is a listwise softmax choice model.

It preserves the fact that a decision involved one selected candidate among an entire set of alternatives.

The regularized training objective is

$$
\boxed{
\mathcal L(\theta)=
\frac{1}{T_{\mathrm{train}}}
\sum_{t\in\mathcal D_{\mathrm{train}}}
\left[
\log\sum_{j=1}^{K_t}
\exp(R_\theta(\mathbf z_{t,j}))
-
R_\theta(\mathbf z_{t,a_t})
\right]
+
\frac{\lambda}{2}\|\theta\|_2^2.
}
$$

The model is trained by minimizing this objective using L-BFGS-B.

Importantly, all three models use the same choice likelihood.

The comparison therefore isolates the effect of changing the representation of the reward function.

The implemented nonlinear comparison does not use the iterative latent-reward EM procedure developed previously.

---

## 5. Model 1: Linear Preference Learning

The baseline reward is

$$
\boxed{
R_{\mathrm{linear}}(\mathbf z)=
\mathbf w^T\mathbf z.
}
$$

The 16 coefficients determine how each image descriptor contributes to the inferred reward.

Positive coefficients correspond to descriptors associated with higher preference, conditional on the other descriptors.

Negative coefficients correspond to descriptors associated with lower preference.

The model is simple, computationally efficient, and interpretable.

However, it cannot represent nonlinear preferences or interactions between descriptors.

For example, the decision-maker may favor intermediate contrast rather than maximum contrast.

Such a preference cannot generally be represented by a purely linear function.

The implementation uses L2 regularization with

$$
\lambda_{\mathrm{linear}}=0.10.
$$

This model serves as the reference for evaluating nonlinear reward learning.

---

## 6. Model 2: Generalized Additive Model

The generalized additive model allows each descriptor to contribute through a nonlinear function.

The reward is

$$
\boxed{
R_{\mathrm{GAM}}(\mathbf z)=
\sum_{k=1}^{16}g_k(z_k).
}
$$

Each $g_k$ is represented using a cubic spline basis.

Let

$$
\mathbf B_k(z_k)=
[B_{k,1}(z_k),\ldots,B_{k,M_k}(z_k)].
$$

Then

$$
g_k(z_k)=
\sum_{m=1}^{M_k}\beta_{k,m}B_{k,m}(z_k).
$$

Combining all spline basis functions gives

$$
\boldsymbol\phi_{\mathrm{GAM}}(\mathbf z)
=
[
\mathbf B_1(z_1),\ldots,\mathbf B_{16}(z_{16})
].
$$

The reward becomes linear in the expanded spline representation:

$$
\boxed{
R_{\mathrm{GAM}}(\mathbf z)=
\boldsymbol\beta^T
\boldsymbol\phi_{\mathrm{GAM}}(\mathbf z).
}
$$

Although the model is nonlinear in the original descriptors, it remains linear in the fitted spline coefficients.

This allows the same softmax preference-learning objective to be used.

### Implementation

The spline basis is constructed using `SplineTransformer` with:

- Four knots per descriptor.
- Cubic splines.
- Quantile-based knot placement.
- Constant extrapolation outside the spline domain.
- L2 coefficient regularization.

The default regularization is

$$
\lambda_{\mathrm{GAM}}=0.25.
$$

### Interpretation

The GAM can represent reward contributions that increase, decrease, saturate, or possess intermediate optima.

For example, an agent might favor images with moderate gradient energy while rejecting both nearly uniform images and excessively noisy images.

A major advantage of the GAM is that the learned contribution $g_k(z_k)$ can be plotted directly.

### Limitation

The current GAM is additive.

It does not explicitly include interactions such as

$$
g_{kl}(z_k,z_l).
$$

Therefore, dependencies involving combinations of descriptors may remain difficult to represent.

---

## 7. Model 3: Gaussian-Process-Inspired Preference Learning

The third model introduces a flexible nonlinear representation inspired by Gaussian-process kernels.

A Gaussian process defines a distribution over functions:

$$
f(\mathbf z)\sim
\mathcal{GP}(0,k(\mathbf z,\mathbf z')).
$$

For a radial-basis-function kernel,

$$
\boxed{
k(\mathbf z,\mathbf z')=
\sigma_f^2
\exp\left[
-\frac{
\|\mathbf z-\mathbf z'\|^2
}{
2\ell^2
}
\right].
}
$$

Here, $\ell$ controls the characteristic length scale of variations in descriptor space.

Rather than implementing an exact Gaussian process, the code approximates this kernel using random Fourier features.

### Random Fourier feature representation

We generate random feature vectors

$$
\boldsymbol\omega_m\sim
\mathcal N(0,\ell^{-2}\mathbf I)
$$

and random phases

$$
b_m\sim\mathcal U(0,2\pi).
$$

The feature mapping is

$$
\boxed{
\phi_m(\mathbf z)=
\sigma_f
\sqrt{\frac{2}{M}}
\cos(
\boldsymbol\omega_m^T\mathbf z+b_m
).
}
$$

The full feature vector is

$$
\boldsymbol\phi_{\mathrm{RFF}}(\mathbf z)
=
[\phi_1(\mathbf z),\ldots,\phi_M(\mathbf z)]^T.
$$

The reward function becomes

$$
\boxed{
R_{\mathrm{GP-RFF}}(\mathbf z)=
\boldsymbol\beta^T
\boldsymbol\phi_{\mathrm{RFF}}(\mathbf z).
}
$$

This provides a nonlinear approximation to the underlying kernel-based reward function.

### Implementation parameters

The code uses:

$$
M=96
$$

random Fourier features,

$$
\ell=3.0
$$

in standardized 16-dimensional descriptor space, and

$$
\lambda_{\mathrm{GP-RFF}}=0.15.
$$

The random features are generated separately for each trace using a reproducible random seed.

### Approximate uncertainty

For the GP-RFF model, the code calculates a local Gaussian approximation to the distribution of the coefficient vector.

This is obtained from the curvature of the regularized choice objective near its optimum.

Denote the approximate coefficient covariance by

$$
\boldsymbol\Sigma_\beta.
$$

The corresponding reward variance is approximated by

$$
\boxed{
\sigma_R^2(\mathbf z)=
\boldsymbol\phi_{\mathrm{RFF}}(\mathbf z)^T
\boldsymbol\Sigma_\beta
\boldsymbol\phi_{\mathrm{RFF}}(\mathbf z).
}
$$

This provides a model-dependent estimate of reward uncertainty.

The current B/C/D visualizations primarily use the fitted reward values, not these uncertainty estimates.

### Important distinction

GP-RFF is **not an exact Gaussian-process preference model**.

It is a finite-dimensional random-feature approximation to a stationary-kernel Gaussian-process prior, fitted using a preference likelihood.

Its results depend on the selected length scale, number of random features, regularization, and random feature realization.

---

## 8. Random Training/Test Split

For each trace, the code randomly divides complete decisions into:

$$
80\%\ \text{training}
$$

and

$$
20\%\ \text{testing}.
$$

All candidate patches belonging to one decision remain together.

The three models use the same split within a given trace.

A fixed seed makes the partition reproducible.

The split is constructed independently for each trace, using its trace identifier to modify the seed.

No information from held-out selections is used during descriptor standardization or model fitting.

### Limitation of the random split

Randomly withholding decisions measures interpolation performance within the recorded trajectory.

It does not necessarily establish transfer to a new experiment because candidate images and decisions from nearby steps may be correlated.

A future analysis should compare random validation with temporal and spatially separated validation.

---

## 9. Held-Out Evaluation Metrics

For every held-out decision, each model assigns a reward to every candidate patch.

The predicted selection is

$$
\hat a_t=
\arg\max_i R_\theta(\mathbf z_{t,i}).
$$

We calculate several complementary metrics.

### Top-1 accuracy

$$
\boxed{
\mathrm{Accuracy}=
\frac{1}{T_{\mathrm{test}}}
\sum_t
\mathbb 1(\hat a_t=a_t).
}
$$

This measures the frequency with which the highest predicted reward corresponds to the actual selected patch.

For 20 candidates, uniform random selection has an expected accuracy of 5%.

### Selected-patch rank

$$
\boxed{
r_t=
1+
\sum_j
\mathbb 1(
R_{t,j}>R_{t,a_t}
).
}
$$

Lower rank indicates better agreement.

### Pairwise concordance

$$
\boxed{
C_t=
\frac{1}{K_t-1}
\sum_{j\ne a_t}
\left[
\mathbb 1(R_{t,a_t}>R_{t,j})
+
\frac12
\mathbb 1(R_{t,a_t}=R_{t,j})
\right].
}
$$

This measures the fraction of rejected candidates receiving lower inferred rewards than the actual selection, with half credit for ties.

### Negative log-likelihood

$$
\boxed{
\mathrm{NLL}=
-\frac{1}{T_{\mathrm{test}}}
\sum_t
\log P(a_t\mid\mathcal C_t,\theta).
}
$$

Lower negative log-likelihood indicates that the model assigns greater probability to the actual choices.

### Reward margin

$$
\boxed{
\Delta R_t=
R_{t,a_t}-
\max_{j\ne a_t}R_{t,j}.
}
$$

A positive margin indicates that the actual selection receives the highest inferred reward.

Because numerical reward scales depend on model regularization and representation, margins should be interpreted cautiously when comparing different models.

---

## 10. Visualization B: Selected Versus Rejected Rewards

The first visualization shows the reconstructed reward distribution for every decision in each trace.

The horizontal axis represents decision step.

The vertical axis represents fitted reward.

For each decision:

- A blue violin represents the reward distribution of rejected candidates.
- A red point represents the reward assigned to the actual selected patch.
- A red line connects the selected-patch rewards across the trace.
- A black-outlined square identifies a randomly held-out decision.

Three vertically aligned panels are generated:

1. Linear preference model.
2. GAM preference model.
3. GP-RFF preference model.

### Interpretation

A well-performing model should generally assign the actual selected patch a reward near or above the upper end of the rejected-patch distribution.

The most important evaluation concerns the randomly held-out decisions.

A model may obtain excellent agreement on training decisions while performing poorly on held-out selections.

The violin visualization makes this distinction visible throughout the trajectory.

### Limitations

The violins represent the empirical distribution of fitted rewards over rejected candidate patches.

They do not represent posterior uncertainty over the reward function.

Similarly, the red point represents one fitted reward estimate, rather than an independently measured reward.

---

## 11. Visualization C: Sequential Reward and Monotonicity

The second visualization examines the reward assigned to selected patches along the recorded decision sequence.

For a given model, define

$$
R_t^*=
R_\theta(I_{t,a_t}).
$$

The first panel plots $R_t^*$ as a function of decision step for all three models.

The second panel measures upward reward jumps:

$$
\boxed{
V_t=
\max(0,R_t^*-R_{t-1}^*).
}
$$

The third panel calculates cumulative upward variation:

$$
\boxed{
V_{\mathrm{cum}}(t)=
\sum_{s=2}^{t}V_s.
}
$$

### Why monotonicity is interesting

If an agent selects the highest-reward remaining patch from a fixed pool, using a reward function that does not change over time, the selected rewards should satisfy

$$
R_1^*\geq R_2^*\geq\cdots\geq R_T^*.
$$

In this situation, upward jumps indicate disagreement between the inferred reward function and the expected sequential ordering.

However, if the agent receives a new set of candidate patches at every step, monotonic decrease is not generally required.

Similarly, changing experimental objectives or history-dependent rewards can produce nonmonotonic reward sequences.

Thus, monotonicity is a meaningful diagnostic only after the selection mechanism used to construct the trace has been established.

The current code measures monotonicity but does not impose it during model fitting.

---

## 12. Visualization D: Spatial Reward Landscapes and Decision Paths

The third visualization evaluates the learned reward functions across the complete AFM image.

Let $I_{x,y}$ denote an image patch centered at position $(x,y)$.

For each model, we calculate

$$
\boxed{
R_\theta(x,y)=
R_\theta(\mathbf z[I_{x,y}]).
}
$$

This generates a two-dimensional reward field over the original microscopy image.

### Sparse-grid evaluation

The full image is sampled using patches with the same dimensions as the recorded candidate patches.

The default spatial stride is

`GRID_STRIDE = 16`.

For each grid location, the code:

1. Extracts the image patch.
2. Calculates the 16 descriptors.
3. Standardizes the descriptors using the selected trace's training scaler.
4. Evaluates the fitted reward model.
5. Stores the resulting spatial reward.

The descriptors for the sparse spatial grid are calculated once and reused across traces.

The final reward mapping remains trace-specific because each trace has its own scaler and model.

### Actual decision path

The recorded selected positions are connected in chronological order.

This visualization represents the spatial trajectory of the original agent or human selections, where valid coordinates are available in the trace.

### Proposed decision path

The code also constructs a new path using the reconstructed reward field.

At each proposed step, the location with the highest remaining predicted reward is selected:

$$
\boxed{
(x_t^*,y_t^*)=
\arg\max_{(x,y)\in\mathcal A_t}
R_\theta(x,y).
}
$$

After selecting a location, nearby grid positions are excluded from subsequent consideration.

The exclusion radius is controlled by

`PATH_EXCLUSION_PIXELS = 12`.

The procedure continues until the requested number of spatial selections is reached or no admissible locations remain.

### Interpretation

The actual and proposed paths answer different questions.

The actual path shows what the original decision-maker selected.

The proposed path shows what a new greedy agent would select using the reconstructed reward function and a spatial exclusion rule.

The two paths are not directly equivalent, because the reconstructed path can select locations outside the original candidate sets.

Differences between paths therefore do not necessarily indicate errors in the learned reward model.

A stricter comparison would require the spatial coordinates of every original candidate patch and would replay model decisions using exactly the candidate sets available to the original agent.

---

## 13. Comparison Across All Traces

After fitting the models separately, the code combines the held-out statistics into a consolidated comparison.

Four summary panels are generated:

1. Top-1 accuracy.
2. Selected-patch rank.
3. Pairwise concordance.
4. Negative log-likelihood.

Each trace appears on the horizontal axis.

The three reward models are compared within each trace.

This makes it possible to identify whether nonlinear reward modeling helps certain decision strategies more than others.

### Questions addressed

- Are some traces explainable using simple linear preferences?
- Do other traces require nonlinear transformations?
- Does the GAM improve prediction through nonlinear contributions of individual descriptors?
- Does the GP-RFF model capture relationships missed by the additive model?
- Are nonlinear improvements consistent across different traces?
- Do improved prediction metrics correspond to meaningful changes in the spatial reward landscape?

The primary evidence for improved reward reconstruction is better held-out preference prediction.

Greater apparent complexity of the reward maps is not, by itself, evidence of a better model.

---

## 14. Computational Considerations

The linear model has relatively few parameters and is computationally inexpensive.

The GAM expands each descriptor into spline basis functions, increasing the number of fitted coefficients.

The GP-RFF model uses a fixed nonlinear feature representation with 96 random Fourier features.

All three models retain a linear final mapping in their respective feature spaces.

This allows them to use the same regularized softmax optimization routine.

The spatial reward descriptors are calculated once for the original image and reused across models and traces.

A smaller spatial stride increases the number of evaluated image patches and therefore the computational cost.

The GP-RFF approximation avoids the computational expense of fitting an exact Gaussian-process preference model to every candidate patch.

However, its accuracy depends on the quality of the random-feature approximation.

---

## 15. Output Files

The analysis saves results in

`day20_nonlinear_all_traces/`

### Numerical outputs

**`heldout_model_comparison.csv`**

Contains held-out model performance for every trace.

**`decision_predictions.csv`**

Contains the recorded and predicted selections, ranks, reward margins, probabilities, and validation metrics for each decision.

**`all_candidate_rewards.csv`**

Contains the inferred reward assigned by each model to every candidate patch.

**`proposed_spatial_paths.csv`**

Contains the spatial coordinates and predicted rewards of locations selected by the greedy reconstructed path.

### Figures

For each trace, the code produces:

- `trace_<ID>_B_violins.png`
- `trace_<ID>_C_monotonicity.png`
- `trace_<ID>_D_spatial_paths.png`

A consolidated figure,

`all_trace_model_statistics.png`,

compares performance across the available traces.

---

## 16. Scientific Limitations

### Preference observations do not uniquely identify numerical rewards

Different reward functions can reproduce the same candidate rankings.

The learned functions are therefore preference models, not direct measurements of absolute scientific utility.

### Reward scales differ between models

The linear, GAM, and GP-RFF models use different feature representations and regularization parameters.

Their numerical reward magnitudes should not automatically be compared as if they share a common physical scale.

### The GAM does not model arbitrary feature interactions

The additive structure allows nonlinear effects of individual descriptors but does not represent every possible relationship among descriptors.

### GP-RFF is an approximate GP model

Finite random features approximate an RBF kernel.

The method does not provide the full inference and uncertainty guarantees of an exact Gaussian-process preference model.

### Hyperparameters are fixed

The spline complexity, GP length scale, random-feature dimension, and regularization strengths are initially specified rather than optimized by nested cross-validation.

Therefore, observed differences among models can reflect both model structure and hyperparameter choices.

### The analysis is not history-aware

The reward functions depend on the current patch descriptors.

They do not explicitly include previous measurements, exploration history, novelty relative to earlier selections, or model uncertainty at the time of the original decision.

### Spatial mapping extrapolates beyond observed candidates

Some patches in the full-image grid may occupy descriptor regions poorly represented by the training data.

Their predicted rewards may therefore be unreliable.

A subsequent implementation should include out-of-distribution diagnostics or calibrated uncertainty maps.

---

## 17. Conclusions and Next Steps

This experiment extends inverse reward learning from a single linear model to a comparison of three preference-based reward representations.

The linear model provides an interpretable baseline.

The GAM introduces nonlinear contributions of individual image descriptors while retaining considerable interpretability.

The GP-RFF model introduces a flexible nonlinear feature representation approximating a Gaussian-process reward prior.

All models are trained independently for each decision trace using observed candidate selections and evaluated using the same randomly held-out 20% of decisions.

The analysis produces three complementary representations of decision-making:

**B — Reward distributions:** How the inferred model ranks selected patches relative to rejected alternatives.

**C — Sequential reward evolution:** Whether the model reproduces the temporal ordering of decisions and, where appropriate, a monotonically decreasing sequence of selected rewards.

**D — Spatial reward and paths:** Where the inferred decision function assigns high reward over the microscopy image, and how a greedy policy based on that function would navigate the sample.

The most important next step is to establish which nonlinear reward representation improves held-out selection prediction while preserving scientifically interpretable spatial behavior.

If nonlinear modeling provides a substantial improvement, the framework can be extended toward preference-based Deep Kernel Learning, in which a learned latent representation and probabilistic surrogate jointly predict the utility of future microscopy measurements.

This would connect inverse reconstruction of human or agent preferences with autonomous experimental optimization.


In [ ]:
# ================================================================
# ATHENA DAY 20 — ALL-TRACE NONLINEAR PREFERENCE MODELS
# Run directly after ORIGINAL notebook loads `traces` and `img`.
# Independent models per trace: linear, spline GAM, RFF-GP.
# Random held-out 20% of complete decisions per trace.
# B: selected vs rejected reward violins
# C: selected rewards and monotonicity diagnostics
# D: spatial reward maps and actual vs proposed paths
# ================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import sobel, laplace
from scipy.special import logsumexp, softmax
from scipy.optimize import minimize
from scipy.linalg import cho_factor, cho_solve
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.metrics import pairwise_distances
from pathlib import Path

# ------------- Configuration -------------
TEST_FRACTION = 0.20
SPLIT_SEED = 42
MODEL_L2 = {'Linear': 0.10, 'GAM': 0.25, 'GP-RFF': 0.15}
GAM_KNOTS = 4
GAM_DEGREE = 3
GP_RFF_DIM = 96               # approximate stationary-kernel GP; increase to 192 for accuracy
GP_LENGTH_SCALE = 3.0        # in standardized 16D descriptor space
GP_PRIOR_SCALE = 1.0
GRID_STRIDE = 16              # 4 for dense maps; 16 for sparse
PATH_EXCLUSION_PIXELS = 12   # greedy proposed spatial path spacing
SHOW_EACH_TRACE = True
SAVE_DPI = 130
OUT = Path('day20_nonlinear_all_traces')
OUT.mkdir(exist_ok=True)

assert 'traces' in globals(), "Run the original notebook data-loading cell to create `traces`."
assert 'img' in globals(), "Run the original notebook image-loading cell to create `img`."
full_image = np.asarray(img, dtype=float).squeeze()
assert full_image.ndim == 2

FEATURE_NAMES = ['mean', 'std', 'p05', 'p95', 'contrast', 'skew', 'kurtosis',
                 'gradient_mean', 'gradient_std', 'gradient_energy', 'laplacian_abs',
                 'entropy', 'high_frequency', 'anisotropy', 'corr_x', 'corr_y']

def image_features(patch):
    a = np.asarray(patch, dtype=np.float64).squeeze()
    if a.ndim != 2: raise ValueError(f'Expected 2D patch, got {a.shape}')
    good = np.isfinite(a)
    if not good.any(): return np.zeros(16)
    valid = a[good]
    a = np.where(good, a, np.median(valid))
    mean, std = a.mean(), a.std()
    z = (a-mean)/(std+1e-9)
    gx, gy = sobel(z,axis=1)/8, sobel(z,axis=0)/8
    g = np.hypot(gx,gy)
    F = np.abs(np.fft.fftshift(np.fft.fft2(z)))**2
    fx = np.fft.fftshift(np.fft.fftfreq(a.shape[1]))[None,:]
    fy = np.fft.fftshift(np.fft.fftfreq(a.shape[0]))[:,None]
    hf = F[np.sqrt(fx**2+fy**2)>0.25].sum()/(F.sum()+1e-9)
    hist = np.histogram(z,bins=32,range=(-4,4))[0]
    prob = hist[hist>0]/max(1,hist.sum())
    entropy = -np.sum(prob*np.log(prob))/np.log(32)
    v = np.array([mean,std,np.percentile(a,5),np.percentile(a,95),
                  np.percentile(a,95)-np.percentile(a,5),
                  skew(valid) if std>1e-9 else 0,
                  kurtosis(valid) if std>1e-9 else 0,
                  g.mean(),g.std(),np.mean(g*g),np.mean(np.abs(laplace(z))),
                  entropy,hf,np.abs(gx).mean()-np.abs(gy).mean(),
                  np.mean(z[:,:-1]*z[:,1:]) if a.shape[1]>1 else 0,
                  np.mean(z[:-1,:]*z[1:,:]) if a.shape[0]>1 else 0])
    return np.nan_to_num(v, nan=0, posinf=0, neginf=0)

# Bounded pre-computation of descriptors for the entire input image, reused in all traces.
first_entry = None
for tr in traces.values():
    if len(tr): first_entry = next(iter(tr.values())) if isinstance(tr,dict) else tr[0]; break
assert first_entry is not None
ph,pw = np.asarray(first_entry['patches'][0]).squeeze().shape
h,w = full_image.shape
if ph>h or pw>w: raise ValueError('Candidate patch larger than full image')
ys = np.unique(np.r_[np.arange(0,h-ph+1,GRID_STRIDE),h-ph]).astype(int)
xs = np.unique(np.r_[np.arange(0,w-pw+1,GRID_STRIDE),w-pw]).astype(int)
centers = np.array([(x+pw/2,y+ph/2) for y in ys for x in xs],float)
print('Extracting sparse full-image descriptors:',len(centers),'patches')
spatial_raw = np.vstack([image_features(full_image[y:y+ph,x:x+pw]) for y in ys for x in xs])
grid_shape = (len(ys),len(xs))

# Conditional listwise softmax objective, preserving candidate-set structure.
def fit_choice(phi_list, chosen, lam, init=None):
    p = phi_list[0].shape[1]
    init = np.zeros(p) if init is None else init
    def obj(beta):
        loss = 0.; grad = np.zeros_like(beta)
        for X,y in zip(phi_list,chosen):
            r=X@beta; probs=softmax(r)
            loss += logsumexp(r)-r[y]
            grad += X.T@probs-X[y]
        nt=len(phi_list)
        return loss/nt + 0.5*lam*np.dot(beta,beta), grad/nt + lam*beta
    result=minimize(obj,init,jac=True,method='L-BFGS-B',options={'maxiter':250,'ftol':1e-9})
    if not result.success: print('  optimization notice:',result.message)
    return result.x

class RewardModel:
    def __init__(self, kind, seed):
        self.kind=kind; self.seed=seed
    def fit(self,Ztrain,groups,chosen):
        self.scaler = None
        if self.kind=='Linear':
            self.featurize=lambda Z: Z
        elif self.kind=='GAM':
            self.spline=SplineTransformer(n_knots=GAM_KNOTS,degree=GAM_DEGREE,
                                          include_bias=False,knots='quantile',extrapolation='constant')
            self.spline.fit(Ztrain)
            self.featurize=lambda Z: self.spline.transform(Z)
        else:
            # Gaussian-process RBF kernel represented by random Fourier features.
            rng=np.random.default_rng(self.seed)
            self.omega=rng.normal(0,1/GP_LENGTH_SCALE,(Ztrain.shape[1],GP_RFF_DIM))
            self.phase=rng.uniform(0,2*np.pi,GP_RFF_DIM)
            self.featurize=lambda Z: GP_PRIOR_SCALE*np.sqrt(2/GP_RFF_DIM)*np.cos(Z@self.omega+self.phase)
        phi=[self.featurize(x) for x in groups]
        self.beta=fit_choice(phi,chosen,MODEL_L2[self.kind])
        # Approximate Gaussian posterior covariance (Laplace at MAP) for GP-RFF.
        if self.kind=='GP-RFF':
            p=len(self.beta)
            precision=MODEL_L2[self.kind]*np.eye(p)
            for X in phi:
                pr=softmax(X@self.beta)
                precision += X.T@(pr[:,None]*X - pr[:,None]*(pr@X)[None,:])/len(phi)
            self.covariance=np.linalg.inv(precision+1e-8*np.eye(p))
        return self
    def predict(self,Z): return self.featurize(Z)@self.beta
    def std(self,Z):
        if self.kind!='GP-RFF': return None
        P=self.featurize(Z)
        return np.sqrt(np.maximum(np.einsum('ij,jk,ik->i',P,self.covariance,P),0))

# Load each trace in chronological order, independently.
summary_rows=[]; decision_rows=[]; patch_rows=[]; path_rows=[]
models_by_trace={}; splits_by_trace={}; trace_steps={}

def best_remaining_path(reward, xy, length, radius):
    remaining=np.ones(len(reward),dtype=bool)
    indices=[]
    for k in range(min(length,len(reward))):
        scores=np.where(remaining,reward,-np.inf)
        j=int(np.argmax(scores))
        if not np.isfinite(scores[j]): break
        indices.append(j)
        remaining &= np.sum((xy-xy[j])**2,axis=1)>radius**2
    return np.asarray(indices,dtype=int)

def plot_B(trace_id,entries,models,scaler,test_idx):
    n=len(entries); pos=np.arange(n)
    fig,axes=plt.subplots(len(models),1,figsize=(max(15,n*.17),3.9*len(models)),sharex=True,squeeze=False)
    for ax,(kind,model) in zip(axes.flat,models.items()):
        violins=[]; chosen=[]
        for e in entries:
            r=model.predict(scaler.transform(e['F']))
            selected=e['selected']; chosen.append(r[selected]); violins.append(np.delete(r,selected))
        vp=ax.violinplot(violins,positions=pos,widths=.8,showextrema=False)
        for body in vp['bodies']:
            body.set_facecolor('royalblue');body.set_alpha(.23);body.set_edgecolor('royalblue')
        ax.plot(pos,chosen,color='red',lw=.85)
        ax.scatter(pos,chosen,c='red',s=12,zorder=4)
        ax.scatter(test_idx,np.asarray(chosen)[test_idx],marker='s',s=33,
                   facecolors='white',edgecolors='black',zorder=5)
        ax.set_ylabel('Reward');ax.set_title(f'Trace {trace_id} | {kind}: red=chosen, blue=rejected')
        ax.grid(axis='y',alpha=.15)
    stride=max(1,n//20);axes[-1,0].set_xticks(pos[::stride],[str(entries[i]['step']) for i in pos[::stride]])
    axes[-1,0].set_xlabel('Decision step; white squares = randomly held out')
    fig.tight_layout();fig.savefig(OUT/f'trace_{trace_id}_B_violins.png',dpi=SAVE_DPI)
    if SHOW_EACH_TRACE:plt.show()
    else:plt.close(fig)

def plot_C(trace_id,entries,models,scaler,test_idx):
    n=len(entries); x=np.arange(n)
    fig,ax=plt.subplots(3,1,figsize=(14,10),sharex=True)
    color={'Linear':'gray','GAM':'seagreen','GP-RFF':'darkorange'}
    for kind,m in models.items():
        y=np.array([m.predict(scaler.transform(e['F']))[e['selected']] for e in entries])
        change=np.r_[0,np.diff(y)];positive=np.maximum(0,change)
        ax[0].plot(x,y,lw=1.2,label=kind,color=color[kind]);
        ax[1].plot(x,positive,lw=1.1,label=kind,color=color[kind]);
        ax[2].plot(x,np.cumsum(positive),lw=1.2,label=kind,color=color[kind]);
    ax[0].scatter(test_idx,y[test_idx],facecolors='none',edgecolors='black',s=28,label='Held out')
    ax[0].set_ylabel('Chosen reward');ax[1].set_ylabel('Positive reward jump');ax[2].set_ylabel('Cumulative upward jumps')
    ax[2].set_xlabel('Decision step')
    for a in ax:a.legend();a.grid(alpha=.18)
    stride=max(1,n//15);ax[-1].set_xticks(x[::stride],[str(entries[i]['step']) for i in x[::stride]])
    fig.suptitle(f'Trace {trace_id}: monotonicity is expected ONLY for a fixed-pool best-remaining policy')
    fig.tight_layout();fig.savefig(OUT/f'trace_{trace_id}_C_monotonicity.png',dpi=SAVE_DPI)
    if SHOW_EACH_TRACE:plt.show()
    else:plt.close(fig)

def plot_D(trace_id,entries,models,scaler,positions):
    zz=scaler.transform(spatial_raw)
    actual=np.array([e['position'] if e['position'] is not None else (np.nan,np.nan) for e in entries],float)
    fig,axes=plt.subplots(len(models),3,figsize=(12,4*len(models)),squeeze=False)
    for row,(kind,m) in enumerate(models.items()):
        rew=m.predict(zz)
        proposed=best_remaining_path(rew,centers,len(entries),PATH_EXCLUSION_PIXELS)
        for k,j in enumerate(proposed):
            path_rows.append({'trace':trace_id,'model':kind,'rank':k+1,
                              'x':centers[j,0],'y':centers[j,1],'reward':rew[j]})
        good=np.isfinite(actual).all(axis=1)
        axes[row,0].imshow(full_image,cmap='gray',origin='upper')
        if good.any():
            axes[row,0].plot(actual[good,0],actual[good,1],'-o',ms=2,lw=.8,c='red')
            axes[row,0].scatter(actual[good,0][0],actual[good,1][0],c='yellow',s=45,edgecolors='black')
        axes[row,0].set_title(f'{kind} | actual chosen path')
        map_rewards=rew.reshape(grid_shape)
        heat=axes[row,1].imshow(map_rewards,extent=(centers[:,0].min(),centers[:,0].max(),
                   centers[:,1].max(),centers[:,1].min()),origin='upper',cmap='viridis',aspect='auto')
        if good.any():axes[row,1].scatter(actual[good,0],actual[good,1],c='red',s=5)
        axes[row,1].set_title(f'{kind} | inferred reward field')
        fig.colorbar(heat,ax=axes[row,1],shrink=.8)
        axes[row,2].imshow(full_image,cmap='gray',origin='upper')
        axes[row,2].plot(centers[proposed,0],centers[proposed,1],'-o',ms=2,lw=.8,c='cyan')
        axes[row,2].scatter(centers[proposed[0],0],centers[proposed[0],1],c='yellow',s=45,edgecolors='black')
        axes[row,2].set_title(f'{kind} | proposed greedy spatial path')
        for a in axes[row]:a.set_xlim(0,w);a.set_ylim(h,0);a.set_xlabel('x (pixels)');a.set_ylabel('y (pixels)')
    fig.tight_layout();fig.savefig(OUT/f'trace_{trace_id}_D_spatial_paths.png',dpi=SAVE_DPI)
    if SHOW_EACH_TRACE:plt.show()
    else:plt.close(fig)

for q,trace in traces.items():
    steps=sorted(trace.keys()) if isinstance(trace,dict) else list(range(len(trace)))
    entries=[]
    for step in steps:
        record=trace[step]
        if 'patches' not in record or 'selected_candidate' not in record:continue
        patches=record['patches'];selected=int(record['selected_candidate'])
        if len(patches)<2 or not(0<=selected<len(patches)):continue
        F=np.vstack([image_features(patch) for patch in patches])
        position=record.get('position',None)
        if position is not None:
            try: position=tuple(np.asarray(position,dtype=float).ravel()[:2])
            except Exception: position=None
            if position is not None and len(position)!=2:position=None
        entries.append(dict(step=step,F=F,selected=selected,position=position))
    if len(entries)<5:
        print('Skipping trace',q,'(insufficient decisions)');continue
    n=len(entries);rng=np.random.default_rng(SPLIT_SEED+int(q))
    perm=rng.permutation(n);nt=max(1,round(n*TEST_FRACTION))
    test_idx=np.sort(perm[:nt]);train_idx=np.sort(perm[nt:])
    scaler=StandardScaler().fit(np.vstack([entries[i]['F'] for i in train_idx]))
    groups=[scaler.transform(entries[i]['F']) for i in train_idx]
    chosen=[entries[i]['selected'] for i in train_idx]
    Ztrain=np.vstack(groups)
    models={}
    for kind in ['Linear','GAM','GP-RFF']:
        model=RewardModel(kind,seed=1234+int(q)).fit(Ztrain,groups,chosen)
        models[kind]=model
        for ix,e in enumerate(entries):
            reward=model.predict(scaler.transform(e['F']))
            y=e['selected'];other=np.delete(reward,y);pr=softmax(reward)
            rec={'trace':q,'model':kind,'step':e['step'],'position':ix,
                 'split':'test' if ix in test_idx else 'train',
                 'selected':y,'predicted':int(np.argmax(reward)),
                 'top1':int(np.argmax(reward)==y),
                 'rank':int(1+(reward>reward[y]).sum()),
                 'concordance':float((reward[y]>other).mean()+.5*(reward[y]==other).mean()),
                 'nll':float(-np.log(max(pr[y],1e-15))),
                 'margin':float(reward[y]-other.max()),
                 'chosen_reward':float(reward[y])}
            decision_rows.append(rec)
            for j,r in enumerate(reward):
                patch_rows.append({'trace':q,'model':kind,'position':ix,'step':e['step'],
                                   'split':rec['split'],'candidate':j,'selected':int(j==y),
                                   'reward':float(r)})
        testrows=[r for r in decision_rows if r['trace']==q and r['model']==kind and r['split']=='test']
        summary_rows.append({'trace':q,'model':kind,'test_n':len(testrows),
                             **{m:float(np.mean([r[m] for r in testrows])) for m in ['top1','rank','concordance','nll','margin']}})
    models_by_trace[q]=(models,scaler)
    splits_by_trace[q]=(train_idx,test_idx)
    trace_steps[q]=entries
    print(f'Trace {q}: {n} steps, {len(train_idx)} train, {len(test_idx)} test')
    plot_B(q,entries,models,scaler,test_idx)
    plot_C(q,entries,models,scaler,test_idx)
    plot_D(q,entries,models,scaler,None)

summary=pd.DataFrame(summary_rows)
summary.to_csv(OUT/'heldout_model_comparison.csv',index=False)
pd.DataFrame(decision_rows).to_csv(OUT/'decision_predictions.csv',index=False)
pd.DataFrame(patch_rows).to_csv(OUT/'all_candidate_rewards.csv',index=False)
pd.DataFrame(path_rows).to_csv(OUT/'proposed_spatial_paths.csv',index=False)
print('\nHELD-OUT COMPARISON (each trace fitted independently):')
print(summary.round(3).to_string(index=False))

# Aggregate statistics across traces (each trace counts equally).
fig,ax=plt.subplots(2,2,figsize=(14,8),constrained_layout=True)
for a,(field,title) in zip(ax.flat,[('top1','Top-1 accuracy ↑'),('rank','Selected rank ↓'),
                                    ('concordance','Pairwise concordance ↑'),('nll','Choice NLL ↓')]):
    for model in ['Linear','GAM','GP-RFF']:
        s=summary[summary.model==model]
        a.plot(s.trace.astype(str),s[field],'-o',label=model)
    a.set_title(title);a.set_xlabel('Trace ID');a.grid(alpha=.15);a.legend()
fig.savefig(OUT/'all_trace_model_statistics.png',dpi=SAVE_DPI)
plt.show()
print('Saved all data and PNG figures to:',OUT.resolve())
print('GP-RFF = random-Fourier-feature approximation to a GP preference prior (not exact GP).')
print('Proposed paths use greedy spatial reward with exclusion, NOT the original candidate sets.')


DKL version


# ATHENA Hackathon Day 20: Two-Dimensional Deep Kernel Learning from Microscopy Image Patches

## 1. Scientific Objective

The objective of this experiment is to reconstruct the reward function governing human or AI-agent decisions directly from microscopy image patches, without manually specifying image descriptors.

In previous analyses, reward functions were learned from 16 predefined image descriptors, including intensity, contrast, gradients, entropy, and spatial correlations.

These approaches assume that the relevant image characteristics can be adequately represented by the chosen descriptors.

Here, we investigate a more flexible hypothesis:

**Can a neural network discover a low-dimensional representation of microscopy images that captures the information governing the observed decisions?**

We use a Deep Kernel Learning (DKL)-inspired architecture with a two-dimensional learned latent space and a nonlinear Gaussian-process-inspired reward model.

The analysis has four principal objectives:

1. Learn a two-dimensional representation of image patches associated with the observed decisions.
2. Reconstruct a nonlinear reward function over the learned latent space.
3. Evaluate whether the reward function reproduces decisions excluded from training.
4. Reconstruct the spatial reward landscape across the original microscopy image and compare it with the observed selection density.

Each decision trace is analyzed independently because different traces may represent different decision-making strategies.

---

## 2. Experimental Data

The code operates directly on the `traces` dictionary and full microscopy image `img` loaded by the original ATHENA Day 20 notebook.

For decision step $t$, the candidate image set is

$$
\mathcal C_t=
\{I_{t,1},I_{t,2},\ldots,I_{t,K_t}\}.
$$

The recorded selection is

$$
a_t\in\{1,\ldots,K_t\}.
$$

The selected image is

$$
I_t^*=I_{t,a_t}.
$$

The complete decision trace is

$$
\mathcal D=
\{(\mathcal C_t,a_t)\}_{t=1}^{T}.
$$

The reward assigned to each candidate is not directly observed.

Instead, each decision provides relative preference information: the selected candidate was favored over the alternatives available at that step.

The model learns from these choices using a probabilistic selection likelihood.

### Independent training and validation

For each trace, complete decisions are randomly divided into:

- 80% training decisions.
- 20% held-out test decisions.

An additional validation subset is taken from the training portion for early stopping.

All patches belonging to one decision remain together.

The test selections are not used during model training or early stopping.

A reproducible random seed is used for each trace.

---

## 3. Direct Image-Patch Representation

Unlike the previous linear and nonlinear descriptor-based models, this implementation operates directly on the image pixels.

Each candidate patch is converted into a numerical image tensor.

The patches are resized to

$$
32\times32
$$

pixels using bilinear interpolation.

The resized images are then passed to a convolutional neural network.

The basic representation pipeline is

$$
\boxed{
I
\longrightarrow
\text{CNN}
\longrightarrow
(z_1,z_2).
}
$$

The convolutional encoder learns features from the image data rather than relying on predefined quantities such as contrast or gradient energy.

This creates the possibility of discovering visual patterns associated with preferences that are not adequately captured by conventional global image statistics.

However, resizing may remove information about fine spatial structures. The chosen input resolution therefore represents a tradeoff between computational efficiency and preservation of microscopic detail.

---

## 4. Two-Dimensional Latent Representation

Let the convolutional encoder be

$$
f_\phi(I),
$$

where $\phi$ represents the neural-network parameters.

The encoder maps every image patch to a two-dimensional latent vector:

$$
\boxed{
\mathbf z(I)=
f_\phi(I)=
\begin{bmatrix}
z_1(I)\\
z_2(I)
\end{bmatrix}.
}
$$

The two latent coordinates are learned through preference-based training.

They are not assigned predefined physical meanings.

For example, the network might learn to organize images according to morphology, boundaries, contrast, or other structural characteristics that influence the recorded decisions.

The use of only two latent dimensions serves two purposes.

First, it limits the complexity of the learned representation.

Second, it allows the full latent space to be visualized directly.

### Latent-space identifiability

The latent coordinates are not uniquely determined.

Equivalent reward functions may be represented by rotated, reflected, or otherwise transformed latent spaces when the downstream model permits such transformations.

Therefore, the numerical values of $z_1$ and $z_2$ should not automatically be interpreted as physical quantities.

A subsequent interpretation step could correlate the learned coordinates with conventional image descriptors or independently measured physical characteristics.

---

## 5. Nonlinear Reward Function in Latent Space

The learned two-dimensional representation is used as the input to a nonlinear reward model.

The reward is

$$
\boxed{
R(I)=g_\theta[f_\phi(I)].
}
$$

Here:

- $f_\phi$ is the convolutional image encoder.
- $g_\theta$ is the nonlinear reward model.
- $R(I)$ is the inferred systematic preference score.

The architecture is inspired by Deep Kernel Learning, in which a neural network learns a representation that is subsequently modeled using a Gaussian-process kernel.

### Gaussian-process kernel

A conventional Gaussian-process model assumes

$$
g(\mathbf z)\sim
\mathcal{GP}(0,k(\mathbf z,\mathbf z')).
$$

For an RBF kernel,

$$
\boxed{
k(\mathbf z,\mathbf z')=
\sigma_f^2
\exp\left[
-\frac{
\|\mathbf z-\mathbf z'\|^2
}{
2\ell^2
}
\right].
}
$$

The length scale $\ell$ controls how rapidly reward varies in the two-dimensional latent space.

Images with similar latent coordinates tend to receive correlated reward predictions.

---

## 6. Random Fourier Feature Approximation

The implementation uses random Fourier features (RFF) to approximate the RBF kernel.

Instead of fitting an exact Gaussian-process preference model, the code constructs a finite-dimensional nonlinear feature mapping.

Random frequencies are sampled as

$$
\boldsymbol\omega_m\sim
\mathcal N(0,\ell^{-2}\mathbf I),
$$

and random phases as

$$
b_m\sim\mathcal U(0,2\pi).
$$

The random Fourier feature mapping is

$$
\boxed{
\psi_m(\mathbf z)=
\sqrt{\frac{2}{M}}
\cos(
\boldsymbol\omega_m^T\mathbf z+b_m
).
}
$$

The complete nonlinear feature representation is

$$
\boldsymbol\psi(\mathbf z)=
[\psi_1(\mathbf z),\ldots,\psi_M(\mathbf z)]^T.
$$

The reward becomes

$$
\boxed{
R(I)=
\boldsymbol\beta^T
\boldsymbol\psi[f_\phi(I)].
}
$$

The implementation uses

$$
M=96
$$

random Fourier features and an initial kernel length scale of

$$
\ell=1.2.
$$

The random frequencies and phases are fixed once initialized.

The CNN parameters and reward coefficients $\boldsymbol\beta$ are learned jointly.

### Important distinction

This implementation is a **finite-feature DKL approximation**, not exact Gaussian-process inference.

The random Fourier features approximate a stationary RBF kernel in the learned two-dimensional latent space.

A Gaussian prior on the output coefficients provides a GP-inspired probabilistic structure, but the model does not construct an exact GP posterior.

---

## 7. Preference-Based Training

We do not observe numerical reward targets for the candidate images.

Therefore, ordinary regression against measured reward values is not possible.

Instead, the model is trained using the recorded choices.

For decision step $t$, the network predicts a reward for every candidate:

$$
R_{t,i}=
\boldsymbol\beta^T
\boldsymbol\psi[f_\phi(I_{t,i})].
$$

The selection probability is modeled as

$$
\boxed{
P(a_t=i\mid\mathcal C_t)=
\frac{
\exp(R_{t,i}/\tau)
}{
\sum_{j=1}^{K_t}
\exp(R_{t,j}/\tau)
}.
}
$$

Here, $\tau$ is the reward-selection temperature.

The implementation initially uses

$$
\tau=1.
$$

Smaller temperatures produce more deterministic preference probabilities.

Larger temperatures distribute probability more broadly among the candidates.

### Training objective

The principal loss is the negative log-likelihood of the observed selections:

$$
\boxed{
\mathcal L_{\mathrm{choice}}=
-\frac{1}{T_{\mathrm{train}}}
\sum_t
\log P(a_t\mid\mathcal C_t).
}
$$

The implementation adds a Gaussian-prior penalty on the random-feature reward coefficients:

$$
\mathcal L_{\mathrm{prior}}=
\frac{1}{2\sigma_\beta^2}
\|\boldsymbol\beta\|^2.
$$

The code also applies weight decay during neural-network optimization.

The network is optimized using AdamW, gradient clipping, and early stopping based on a validation subset drawn from the training decisions.

The default maximum training duration is

`DKL_EPOCHS = 150`

with learning rate

`DKL_LR = 0.001`.

The patience for early stopping is

`DKL_PATIENCE = 25`.

### Why this works without numerical rewards

Every selected image provides information about its reward relative to the rejected candidates.

During training, the reward model is updated to increase the probability of the selected candidate.

Simultaneously, the CNN changes the latent representation to make the observed preferences easier to explain.

Thus, the image representation and reward function are learned together.

---

## 8. Approximate Reward Uncertainty

After training, the model constructs a local Gaussian approximation for the random-feature reward coefficients.

Let

$$
\boldsymbol\beta^*
$$

denote the fitted coefficients.

The negative log-posterior curvature is approximated through a Hessian matrix

$$
\mathbf H.
$$

The coefficient covariance is approximated by

$$
\boxed{
\boldsymbol\Sigma_\beta
\approx
\mathbf H^{-1}.
}
$$

For an image with random Fourier features $\boldsymbol\psi(I)$, the conditional reward variance is

$$
\boxed{
\sigma_R^2(I)=
\boldsymbol\psi(I)^T
\boldsymbol\Sigma_\beta
\boldsymbol\psi(I).
}
$$

This provides an approximate measure of uncertainty associated with the reward coefficients.

### Limitation of the uncertainty estimate

The calculation conditions on the trained CNN representation.

It does not fully propagate uncertainty in the neural-network weights, the random Fourier frequencies, or the assumed kernel hyperparameters.

Consequently, the resulting uncertainty map should be interpreted as an approximate, model-conditional reward uncertainty rather than a fully calibrated posterior uncertainty.

---

## 9. Held-Out Evaluation

The model is evaluated on the randomly withheld 20% of decisions.

For each held-out decision, all candidate images are processed through the CNN and nonlinear reward model.

The predicted selection is

$$
\boxed{
\hat a_t=
\arg\max_i R(I_{t,i}).
}
$$

The hidden selection index is used only for evaluation.

### Top-1 accuracy

$$
\mathrm{Accuracy}=
\frac{1}{T_{\mathrm{test}}}
\sum_t
\mathbb 1(\hat a_t=a_t).
$$

### Selected-patch rank

$$
\mathrm{Rank}_t=
1+
\sum_j
\mathbb 1(R_{t,j}>R_{t,a_t}).
$$

### Pairwise concordance

$$
\boxed{
C_t=
\frac{1}{K_t-1}
\sum_{j\neq a_t}
\left[
\mathbb 1(R_{t,a_t}>R_{t,j})
+
\frac12\mathbb 1(R_{t,a_t}=R_{t,j})
\right].
}
$$

### Negative log-likelihood

$$
\boxed{
\mathrm{NLL}=
-\frac{1}{T_{\mathrm{test}}}
\sum_t
\log P(a_t\mid\mathcal C_t).
}
$$

These metrics determine whether the model captures decision-making behavior beyond the training decisions.

Because numerical ground-truth rewards are not available, the validation measures choice prediction and reward ranking rather than direct numerical reward reconstruction error.

---

## 10. Visualization B: Selected Versus Rejected Rewards

The first visualization plots reward distributions across the complete decision trace.

The horizontal axis represents decision step.

The vertical axis represents the inferred reward.

For each decision:

- Blue violin: Distribution of rewards assigned to rejected candidate patches.
- Red point: Reward of the actual selected patch.
- Red line: Connects selected-patch reward estimates across decision steps.
- Black-outlined square: Identifies a held-out decision.

A successful model should generally place the selected-patch reward near or above the upper end of the rejected-candidate distribution.

Held-out decisions provide the most meaningful assessment because they were not used in training.

The blue violin represents the variation of fitted rewards across rejected candidates, not posterior uncertainty.

---

## 11. Visualization C: Reward Evolution and Monotonicity

The second visualization examines the inferred reward of selected patches across decision steps.

For each step,

$$
R_t^*=R(I_{t,a_t}).
$$

The code plots $R_t^*$ against decision step.

A running non-increasing envelope is also displayed:

$$
\boxed{
M_t=
\min_{s\leq t}R_s^*.
}
$$

The local monotonicity violation is

$$
\boxed{
V_t=
\max(0,R_t^*-R_{t-1}^*).
}
$$

The cumulative violation is

$$
V_{\mathrm{cum}}(t)=
\sum_{s=2}^{t}V_s.
$$

### Interpretation

If a trace was generated by repeatedly choosing the highest-reward remaining patch from a fixed candidate pool, the selected rewards should decrease monotonically.

However, monotonicity is not expected when new candidate sets are generated at each step or when the reward depends on previous observations.

The current model does not impose monotonicity during training.

This visualization therefore acts as a diagnostic rather than a constraint.

---

## 12. Visualization D: Spatial Reward Landscape

After training, the CNN and reward model are applied to image patches extracted across the original AFM image.

Let $I_{x,y}$ be a patch centered at location $(x,y)$.

The spatial reward field is

$$
\boxed{
R(x,y)=
\boldsymbol\beta^T
\boldsymbol\psi[f_\phi(I_{x,y})].
}
$$

The image is evaluated on a regular spatial grid.

The default grid spacing is

`DKL_GRID_STRIDE = 16`.

A smaller stride produces a denser map.

For each spatial location, the code calculates:

1. Two-dimensional latent coordinates.
2. Predicted reward.
3. Approximate reward standard deviation.

### Spatial plots

The implementation displays:

- Original AFM image with recorded selected locations.
- Reconstructed spatial reward field.
- Approximate reward uncertainty field.
- Proposed greedy spatial path on the reward landscape.

The greedy path is generated by repeatedly selecting the highest-reward grid location while excluding nearby previously selected locations.

This is a new policy derived from the inferred reward function.

It should not be interpreted as an exact replay of the original agent's decisions.

---

## 13. Visualization E: Learned Two-Dimensional Latent Space

This is a new visualization enabled by the two-dimensional image encoder.

Every candidate image is represented by

$$
\mathbf z(I)=(z_1,z_2).
$$

The code projects candidate images directly into this learned space.

### Panel E1: Selected versus rejected patches

The latent coordinates of selected patches are displayed in red.

Rejected patches are displayed in blue.

This reveals whether the learned representation organizes preferred and rejected images into distinguishable regions.

However, complete separation is not required because the same image characteristics may receive different relative rankings depending on the competing candidates.

### Panel E2: Reward in latent space

The same latent coordinates are colored according to the inferred reward.

This visualization reveals the nonlinear preference landscape learned by the random-feature reward model.

One important question is whether selected patches occupy one compact high-reward region or multiple separated high-reward regions.

The latter could indicate that the decision-maker values several distinct classes of visual structures.

### Scientific interpretation

Because the latent coordinates are learned from preferences rather than physical labels, their meanings are initially unknown.

A subsequent analysis could evaluate correlations between $z_1$, $z_2$, and the conventional 16 image descriptors.

We could also inspect representative microscopy patches from different regions of latent space.

This would help establish whether the learned coordinates correspond to interpretable physical characteristics.

---

## 14. Visualization F: Actual Selection KDE Versus Reward-Derived Spatial Probability

The final visualization compares the spatial distribution of actual selections with the distribution implied by the inferred reward function.

This addresses an important limitation of greedy-path comparisons.

A reconstructed reward function may correctly identify the spatial regions preferred by an agent while producing a different exact trajectory under greedy selection.

Instead of comparing only paths, we compare spatial preference distributions.

### 14.1. KDE of actual selections

Let the actual selected positions be

$$
\mathcal P=
\{(x_1,y_1),\ldots,(x_T,y_T)\}.
$$

We estimate their spatial density using a Gaussian kernel:

$$
\boxed{
p_{\mathrm{actual}}(x,y)
\propto
\sum_{t=1}^{T}
\exp\left[
-\frac{
(x-x_t)^2+(y-y_t)^2
}{
2h^2
}
\right].
}
$$

The kernel bandwidth is initially

`DKL_KDE_SIGMA = 12`

pixels.

The density is normalized on the spatial grid.

### 14.2. Reward-derived spatial probability

The reconstructed reward map is converted to a spatial probability distribution using a softmax transformation:

$$
\boxed{
p_R(x,y)=
\frac{
\exp[R(x,y)/\tau_{\mathrm{policy}}]
}{
\sum_{x',y'}
\exp[R(x',y')/\tau_{\mathrm{policy}}]
}.
}
$$

The initial policy temperature is

`DKL_POLICY_TEMP = 1.0`.

A smaller policy temperature concentrates probability near the highest-reward spatial locations.

A larger temperature produces a broader distribution.

### 14.3. Quantitative comparison

The code compares the actual-selection KDE and reward-derived probability distribution using spatial correlation and Jensen–Shannon divergence.

Spatial correlation is

$$
\rho=
\operatorname{corr}
\left(
p_{\mathrm{actual}},
p_R
\right).
$$

Jensen–Shannon divergence is

$$
\boxed{
D_{\mathrm{JS}}(P,Q)=
\frac12D_{\mathrm{KL}}(P\|M)
+
\frac12D_{\mathrm{KL}}(Q\|M),
}
$$

where

$$
M=\frac12(P+Q).
$$

Higher spatial correlation and lower Jensen–Shannon divergence indicate better agreement between the observed selection density and reward-derived probability.

### Limitation

The global reward-derived probability does not explicitly account for which candidate patches were available at each decision step.

The original candidate-generation mechanism may have influenced the spatial distribution of selections.

Therefore, KDE agreement is an exploratory spatial diagnostic rather than a complete reconstruction of the original decision policy.

The current KDE comparison also uses the available actual selection locations from the trace, rather than restricting the KDE exclusively to held-out decisions.

A stricter spatial validation should compare reward-derived probabilities against held-out selection locations while accounting for candidate availability.

---

## 15. Implementation and Configuration

The principal parameters are:

| Parameter | Default | Purpose |
|---|---:|---|
| `DKL_TRACE_IDS` | `[6]` | Traces to analyze |
| `DKL_TEST_FRAC` | 0.20 | Fraction of held-out decisions |
| `DKL_IMAGE_SIZE` | 32 | Input patch resolution |
| `DKL_EPOCHS` | 150 | Maximum training epochs |
| `DKL_LR` | 0.001 | Learning rate |
| `DKL_WEIGHT_DECAY` | 0.001 | Neural-network regularization |
| `DKL_PATIENCE` | 25 | Early stopping patience |
| `DKL_RFF` | 96 | Number of random Fourier features |
| `DKL_LENGTHSCALE` | 1.2 | RBF kernel length scale |
| `DKL_GRID_STRIDE` | 16 | Spatial mapping resolution |
| `DKL_KDE_SIGMA` | 12 | KDE bandwidth in pixels |
| `DKL_POLICY_TEMP` | 1.0 | Spatial probability temperature |

The code can use a GPU when PyTorch detects CUDA support.

Otherwise, it executes on the CPU.

To process all available traces independently, change:

`DKL_TRACE_IDS = sorted(traces.keys())`

For a spatial grid four times denser in each direction, use:

`DKL_GRID_STRIDE = 4`

---

## 16. Scientific Limitations

### Limited number of independent training decisions

Although each decision may contain 20 image patches, the trace contains only one observed selection per decision.

Therefore, 80 training decisions do not provide the same information as 1,600 independently measured numerical rewards.

A CNN may overfit this amount of supervision.

### Two-dimensional latent space may be insufficient

Some decision criteria may require more than two independent latent variables.

Restricting the representation to two dimensions improves visualization but can reduce the model's ability to represent complex preferences.

### Reward is identifiable primarily through preferences

Selections constrain relative rewards, not absolute reward values.

Different reward functions may reproduce the same observed choices.

### The CNN can learn incidental correlations

The encoder may discover image characteristics correlated with choices without recovering the actual scientific logic of the decision-maker.

This risk is especially significant when training datasets are small or contain strong spatial correlations.

### GP uncertainty is approximate

The random-Fourier-feature Laplace approximation does not fully account for uncertainty in the CNN representation or kernel parameters.

### Spatial validation may be affected by candidate availability

The observed path reflects both the agent's preferences and the candidate patches available at each step.

A reward-derived global probability field does not necessarily reproduce that sampling process.

### The model does not explicitly represent experimental history

The inferred reward depends on the current image patch.

Novelty, previous selections, changing scientific objectives, and exploration-exploitation strategies are not explicitly modeled.

---

## 17. Conclusions and Next Steps

This experiment introduces a two-dimensional preference-based Deep Kernel Learning approximation operating directly on microscopy image patches.

The approach replaces manually engineered image descriptors with a convolutional neural network that learns a compact latent representation.

A nonlinear kernel-based reward function is learned jointly with this representation using the recorded selections.

The resulting model provides several complementary outputs:

**B — Reward distributions:** Whether the learned reward ranks selected patches above rejected alternatives.

**C — Sequential reward evolution:** Whether the inferred reward reproduces the ordering of selections along the trace.

**D — Spatial reward landscapes:** Which regions of the original AFM image are assigned high reward.

**E — Two-dimensional latent space:** How the network organizes patches according to the learned preference structure.

**F — Spatial probability comparison:** Whether the inferred reward predicts the spatial concentration of actual selections.

The critical measure of success is held-out prediction of observed decisions, supported by spatial agreement between the reconstructed reward distribution and actual selection locations.

The next methodological step is to compare the DKL representation against the previous linear, GAM, and GP-RFF descriptor models using identical training/test splits.

If the image-based DKL model performs better, this would suggest that the learned neural representation captures decision-relevant image information absent from the manually specified descriptors.

Finally, the latent coordinates can be examined for physical interpretability, and the approximate kernel reward layer can be replaced with a more complete Gaussian-process preference model.

This would establish a pathway from observed human or agent microscopy decisions to autonomous experimental optimization using learned image representations and inferred scientific preferences.


In [ ]:
# ATHENA Day 20: image-patch CNN -> 2D latent -> approximate GP preference reward.
# Self-contained cell; run after ORIGINAL notebook loads `traces` and `img`.
# Requires torch, scipy, sklearn, matplotlib, pandas.
import copy, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import softmax
from scipy.ndimage import gaussian_filter
from scipy.spatial.distance import cdist
from sklearn.metrics import pairwise_distances
import torch
from torch import nn
import torch.nn.functional as F

# ---------------------- CONFIG ----------------------
DKL_TRACE_IDS = sorted(traces) #[6]           # Change to sorted(traces) to analyze ALL traces
DKL_SEED = 42
DKL_TEST_FRAC = 0.20
DKL_IMAGE_SIZE = 32           # resize candidate patches consistently
DKL_EPOCHS = 150
DKL_LR = 0.001
DKL_WEIGHT_DECAY = 1e-3
DKL_PATIENCE = 25
DKL_RFF = 96                  # random Fourier features approximating GP RBF kernel
DKL_LENGTHSCALE = 1.2
DKL_RFF_PRIOR = 1.0           # Gaussian prior variance on GP-feature weights
DKL_REWARD_TEMPERATURE = 1.0 # listwise selection softmax temperature
DKL_GRID_STRIDE = 16         # sparse spatial grid; set 4 for dense
DKL_KDE_SIGMA = 12           # pixels
DKL_POLICY_TEMP = 1.0
DKL_PATH_EXCLUSION = 12      # pixels
DKL_DRAW_ALL = True
DKL_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DKL_OUT = Path('day20_patch_DKL_2D')
DKL_OUT.mkdir(exist_ok=True)

assert 'traces' in globals() and len(traces), 'Run original notebook trace-loading cells first.'
assert 'img' in globals(), 'Run original notebook image-loading cell (img) first.'
IMG = np.asarray(img, dtype=float).squeeze()
assert IMG.ndim == 2

random.seed(DKL_SEED); np.random.seed(DKL_SEED); torch.manual_seed(DKL_SEED)

def patch_tensor(patches):
    a = np.stack([np.asarray(p, dtype=np.float32).squeeze() for p in patches])
    if a.ndim != 3:
        raise ValueError(f'Expected (n,h,w) patches; got {a.shape}')
    x = torch.from_numpy(a[:,None]).float()
    x = F.interpolate(x, size=(DKL_IMAGE_SIZE,DKL_IMAGE_SIZE),mode='bilinear',align_corners=False)
    # Global original-image intensity scaling to preserve intensity cues across patches.
    return x

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),nn.GELU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.GELU(),nn.MaxPool2d(2),
            nn.Conv2d(32,32,3,padding=1),nn.GELU(),nn.AdaptiveAvgPool2d((4,4)),
            nn.Flatten(),nn.Linear(32*4*4,64),nn.GELU(),nn.Linear(64,2)
        )
    def forward(self,x):
        return self.features(x)

class PatchDKL(nn.Module):
    """Trainable CNN 2D latent + fixed RFF GP kernel basis + Bayesian linear head MAP.
    Approximate DKL; not an exact end-to-end GP posterior.
    """
    def __init__(self,seed):
        super().__init__()
        self.encoder=Encoder()
        rng=np.random.default_rng(seed)
        self.register_buffer('omega',torch.tensor(rng.normal(size=(2,DKL_RFF))/DKL_LENGTHSCALE,dtype=torch.float32))
        self.register_buffer('phase',torch.tensor(rng.uniform(0,2*np.pi,size=(DKL_RFF,)),dtype=torch.float32))
        self.beta=nn.Parameter(torch.zeros(DKL_RFF))
    def phi(self,z):
        return (2/DKL_RFF)**0.5*torch.cos(z@self.omega+self.phase)
    def forward(self,x):
        z=self.encoder(x)
        phi=self.phi(z)
        return phi@self.beta,z,phi

def load_trace(trace):
    steps=sorted(trace.keys()) if isinstance(trace,dict) else list(range(len(trace)))
    entries=[trace[s] for s in steps]
    lengths=[len(e['patches']) for e in entries]
    if min(lengths)<2: raise ValueError('Each decision needs at least 2 candidates')
    tensors=[patch_tensor(e['patches']) for e in entries]
    labels=np.array([int(e['selected_candidate']) for e in entries])
    if any(y<0 or y>=k for y,k in zip(labels,lengths)):raise ValueError('Invalid selected_candidate')
    coords=[]
    for e in entries:
        p=np.asarray(e.get('position', [np.nan,np.nan])).ravel()
        try: coords.append([float(p[0]),float(p[1])])
        except Exception:coords.append([np.nan,np.nan])
    return steps,entries,tensors,labels,np.asarray(coords)

def fit_dkl(trace_id):
    steps, entries, tensors, labels, coords=load_trace(traces[trace_id]); T=len(tensors)
    assert T>=8, 'Too few decisions to split into training/validation/test'
    rng=np.random.default_rng(DKL_SEED+int(trace_id))
    ix=rng.permutation(T)
    ntest=max(1,round(DKL_TEST_FRAC*T))
    test=np.sort(ix[:ntest]); train=np.sort(ix[ntest:]);
    # Training-only validation for early stopping, preserves final test untouched.
    sub=rng.permutation(train)
    nval=max(1,round(0.15*len(train))) if len(train)>=12 else 0
    val=sub[:nval]; fit=sub[nval:]
    tensors=[x.to(DKL_DEVICE) for x in tensors]
    model=PatchDKL(DKL_SEED+int(trace_id)).to(DKL_DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=DKL_LR,weight_decay=DKL_WEIGHT_DECAY)
    best=np.inf; best_state=None; no_improve=0; logs=[]
    def loss_over(indices):
        losses=[]
        for t in indices:
            reward,_,_=model(tensors[int(t)])
            loss=F.cross_entropy((reward/DKL_REWARD_TEMPERATURE)[None,:],
                                  torch.tensor([labels[int(t)]],device=DKL_DEVICE))
            losses.append(loss)
        if not losses:return None
        return torch.stack(losses).mean()
    for epoch in range(DKL_EPOCHS):
        model.train();opt.zero_grad()
        loss=loss_over(fit)
        # Prior penalty on GP-feature weights, unlike arbitrary reward labels.
        loss=loss+0.5*model.beta.square().sum()/DKL_RFF_PRIOR/len(fit)
        loss.backward();nn.utils.clip_grad_norm_(model.parameters(),2.0);opt.step()
        model.eval()
        with torch.no_grad():
            score=float(loss_over(val if len(val) else fit).item())
            logs.append([epoch+1,float(loss.item()),score])
        if score<best-1e-5:
            best=score;best_state=copy.deepcopy(model.state_dict());no_improve=0
        else:
            no_improve+=1
            if no_improve>=DKL_PATIENCE:break
    model.load_state_dict(best_state);model.eval()
    # GP finite-feature Laplace uncertainty conditional on learned encoder.
    # curvature of listwise choice likelihood over all training decisions
    with torch.no_grad():
        H=np.eye(DKL_RFF,dtype=float)/DKL_RFF_PRIOR
        for t in train:
            _,_,phi=model(tensors[int(t)])
            a=phi.cpu().numpy(); prob=softmax((a@model.beta.detach().cpu().numpy())/DKL_REWARD_TEMPERATURE)
            H+=(a.T@(np.diag(prob)-np.outer(prob,prob))@a)/DKL_REWARD_TEMPERATURE**2
        cov=np.linalg.inv(H+1e-7*np.eye(DKL_RFF))
    return model,cov,(steps,entries,tensors,labels,coords,train,test),pd.DataFrame(logs,columns=['epoch','training_loss','early_stop_loss'])

def predict_patches(model,cov,tensor):
    with torch.no_grad():
        r,z,phi=model(tensor.to(DKL_DEVICE))
        a=phi.cpu().numpy();mean=r.cpu().numpy();z=z.cpu().numpy()
    sd=np.sqrt(np.maximum(np.einsum('ij,jk,ik->i',a,cov,a),0))
    return mean,z,sd

def spatial_grid(model,cov,patch_hw):
    ph,pw=patch_hw; H,W=IMG.shape
    if ph>H or pw>W: raise ValueError('Patch larger than original image')
    ys=np.unique(np.r_[np.arange(0,H-ph+1,DKL_GRID_STRIDE),H-ph]);xs=np.unique(np.r_[np.arange(0,W-pw+1,DKL_GRID_STRIDE),W-pw]);
    xy=np.array([(x+pw/2,y+ph/2) for y in ys for x in xs]);r=[];sd=[];lat=[]
    # Stream batches, avoid memory expansion of dense grids
    batch=[]
    def flush():
        if batch:
            u,z,s=predict_patches(model,cov,patch_tensor(batch));r.extend(u);sd.extend(s);lat.extend(z);batch.clear()
    for y in ys:
        for x in xs:
            batch.append(IMG[y:y+ph,x:x+pw])
            if len(batch)>=128:flush()
    flush()
    return xy,np.asarray(r),np.asarray(sd),np.asarray(lat),(len(ys),len(xs)),(xs,ys)

def path_indices(xy,reward,count,exclusion):
    eligible=np.ones(len(reward),bool);out=[]
    for _ in range(min(count,len(reward))):
        choices=np.where(eligible,reward,-np.inf);j=int(np.argmax(choices))
        if not np.isfinite(choices[j]):break
        out.append(j)
        eligible &= (np.sum((xy-xy[j])**2,axis=1)>exclusion**2)
    return np.array(out,dtype=int)

def spatial_kde(coords,xy,shape):
    """Gaussian KDE on sparse grid with common Gaussian bandwidth, normalized."""
    coords=np.asarray(coords,float);coords=coords[np.isfinite(coords).all(axis=1)]
    if not len(coords):return None
    d=cdist(xy,coords,'sqeuclidean')
    dens=np.exp(-0.5*d/(DKL_KDE_SIGMA**2)).mean(axis=1)
    return dens/(dens.sum()+1e-12)

summary=[]
for tid in DKL_TRACE_IDS:
    print(f'\n=== Trace {tid} | device={DKL_DEVICE} ===')
    model,cov,data,history=fit_dkl(tid)
    steps,entries,tensors,labels,coords,train,test=data;T=len(steps)
    rows=[];patch_rows=[];all_lat=[]
    for t in range(T):
        r,z,sd=predict_patches(model,cov,tensors[t]);all_lat.append(z)
        y=labels[t];alt=np.delete(r,y);p=softmax(r/DKL_REWARD_TEMPERATURE)
        rows.append(dict(trace=tid,step=steps[t],position=t,split='test' if t in test else 'train',selected=y,predicted=int(r.argmax()),accuracy=int(r.argmax()==y),rank=int(1+(r>r[y]).sum()),concordance=float(((r[y]>alt)+.5*(r[y]==alt)).mean()),nll=float(-np.log(max(p[y],1e-12))),margin=float(r[y]-alt.max()),chosen_reward=float(r[y])))
        for j in range(len(r)):
            patch_rows.append(dict(trace=tid,step=steps[t],position=t,candidate=j,chosen=int(j==y),reward=r[j],reward_sd=sd[j],z1=z[j,0],z2=z[j,1]))
    df=pd.DataFrame(rows);pf=pd.DataFrame(patch_rows)
    held=df[df.split=='test'];summary.append(dict(trace=tid,n_decisions=T,n_test=len(test),**{k:held[k].mean() for k in ['accuracy','rank','concordance','nll','margin']}))
    print('Held-out:',{k:round(float(held[k].mean()),3) for k in ['accuracy','rank','concordance','nll','margin']})
    base=DKL_OUT/f'trace_{tid}';base.mkdir(exist_ok=True)
    df.to_csv(base/'decisions.csv',index=False);pf.to_csv(base/'patch_rewards.csv',index=False);history.to_csv(base/'training.csv',index=False)
    # B: selected red vs rejected blue, all decisions, held-out squares
    fig,ax=plt.subplots(figsize=(max(14,T*.17),5))
    vv=[pf[(pf.position==t)&(pf.chosen==0)].reward.to_numpy() for t in range(T)]
    vp=ax.violinplot(vv,positions=np.arange(T),widths=.85,showextrema=False)
    for body in vp['bodies']:body.set_facecolor('royalblue');body.set_alpha(.22)
    ax.plot(np.arange(T),df.chosen_reward,color='red',lw=1,marker='.',ms=4)
    ax.scatter(test,df.iloc[test].chosen_reward,marker='s',facecolors='white',edgecolors='black',s=35,zorder=5)
    ax.set(xlabel='Decision index (black squares = held-out)',ylabel='DKL reward',title=f'Trace {tid}: B — chosen vs rejected')
    ax.grid(alpha=.12);fig.tight_layout();fig.savefig(base/'B_violin.png',dpi=160);plt.show()
    # C: choice reward and monotonicity (descriptive only)
    rr=df.chosen_reward.to_numpy();jump=np.r_[0,np.maximum(np.diff(rr),0)];env=np.minimum.accumulate(rr)
    fig,axs=plt.subplots(2,1,figsize=(13,7),sharex=True)
    axs[0].plot(rr,'r-',label='Chosen reward');axs[0].plot(env,'k--',label='Running minimum');axs[0].scatter(test,rr[test],facecolors='white',edgecolors='black',marker='s',s=38,zorder=5);axs[0].legend();axs[0].set_ylabel('Reward')
    axs[1].bar(np.arange(T),jump,color='firebrick',alpha=.65);axs[1].set(xlabel='Decision index',ylabel='Upward reward jump')
    fig.suptitle(f'Trace {tid}: C — temporal reward ordering (not enforced)');fig.tight_layout();fig.savefig(base/'C_monotonicity.png',dpi=160);plt.show()
    # 2D latent scatter, red selected, blue rejected. Heldout selections outlined.
    lat=np.vstack(all_lat);is_chosen=pf.chosen.to_numpy().astype(bool)
    fig,axs=plt.subplots(1,2,figsize=(12,5))
    axs[0].scatter(lat[~is_chosen,0],lat[~is_chosen,1],s=8,alpha=.14,c='royalblue',label='Rejected')
    axs[0].scatter(lat[is_chosen,0],lat[is_chosen,1],s=20,alpha=.8,c='red',label='Chosen')
    xt=np.array([all_lat[t][labels[t]] for t in test]);axs[0].scatter(xt[:,0],xt[:,1],facecolors='none',edgecolors='black',s=65,label='Held-out selected')
    axs[0].set(xlabel='Latent 1',ylabel='Latent 2',title='Learned 2D patch embedding');axs[0].legend(fontsize=8)
    sc=axs[1].scatter(lat[:,0],lat[:,1],c=pf.reward.to_numpy(),s=10,cmap='viridis');fig.colorbar(sc,ax=axs[1],label='Reward');axs[1].set(xlabel='Latent 1',ylabel='Latent 2',title='Reward in 2D latent space')
    fig.tight_layout();fig.savefig(base/'latent_2d.png',dpi=160);plt.show()
    # D: sparse image-space reward, uncertainty, proposed global greedy path
    patch_hw=np.asarray(entries[0]['patches'][0]).squeeze().shape
    xy,gr,gs,gz,shape,(xs,ys)=spatial_grid(model,cov,patch_hw)
    chosen_coord=coords[np.isfinite(coords).all(axis=1)]
    path=path_indices(xy,gr,min(T,len(xy)),DKL_PATH_EXCLUSION)
    grid_extent=(xs.min(),xs.max(),ys.max(),ys.min())
    fig,axs=plt.subplots(1,4,figsize=(20,5))
    axs[0].imshow(IMG,cmap='gray',origin='upper');
    if len(chosen_coord):axs[0].plot(chosen_coord[:,0],chosen_coord[:,1],'.-',ms=3,lw=.7,c='red')
    axs[0].set_title('Actual selected path')
    scale=(np.percentile(gr,2),np.percentile(gr,98))
    p1=axs[1].imshow(gr.reshape(shape),origin='upper',extent=grid_extent,cmap='viridis',vmin=scale[0],vmax=scale[1]);axs[1].set_title('DKL reward field');fig.colorbar(p1,ax=axs[1],shrink=.7)
    p2=axs[2].imshow(gs.reshape(shape),origin='upper',extent=grid_extent,cmap='magma');axs[2].set_title('Approx. reward uncertainty');fig.colorbar(p2,ax=axs[2],shrink=.7)
    axs[3].imshow(gr.reshape(shape),origin='upper',extent=grid_extent,cmap='viridis',vmin=scale[0],vmax=scale[1]);
    axs[3].plot(xy[path,0],xy[path,1],'.-',ms=3,lw=.7,c='white');axs[3].set_title('Global greedy path (different policy)')
    for a in axs:a.set_xlabel('x pixels');a.set_ylabel('y pixels')
    fig.suptitle(f'Trace {tid}: D — spatial reward and paths');fig.tight_layout();fig.savefig(base/'D_spatial.png',dpi=160);plt.show()
    # KDE of actual selected positions vs softmax global reward distribution.
    actual_kde=spatial_kde(chosen_coord,xy,shape)
    if actual_kde is not None:
        prob=softmax(gr/DKL_POLICY_TEMP)
        js_m=.5*(actual_kde+prob);eps=1e-12
        js=.5*np.sum(actual_kde*np.log((actual_kde+eps)/(js_m+eps)))+.5*np.sum(prob*np.log((prob+eps)/(js_m+eps)))
        corr=np.corrcoef(actual_kde,prob)[0,1] if np.std(actual_kde)>0 and np.std(prob)>0 else np.nan
        fig,axs=plt.subplots(1,3,figsize=(15,4.5))
        axs[0].imshow(actual_kde.reshape(shape),extent=grid_extent,origin='upper',cmap='viridis');axs[0].set_title('Actual selection KDE')
        axs[1].imshow(prob.reshape(shape),extent=grid_extent,origin='upper',cmap='viridis');axs[1].set_title('Softmax(reward / temperature)')
        axs[2].imshow((actual_kde-prob).reshape(shape),extent=grid_extent,origin='upper',cmap='coolwarm');axs[2].set_title(f'Difference | JS={js:.3f}, corr={corr:.2f}')
        fig.tight_layout();fig.savefig(base/'KDE_comparison.png',dpi=160);plt.show()
        print(f'Trace {tid}: global KDE JS={js:.4f}, correlation={corr:.3f} (candidate availability uncorrected)')
    pd.DataFrame({'x':xy[:,0],'y':xy[:,1],'reward':gr,'reward_sd':gs,'z1':gz[:,0],'z2':gz[:,1]}).to_csv(base/'spatial_grid.csv',index=False)
    pd.DataFrame({'x':xy[path,0],'y':xy[path,1],'reward':gr[path]}).to_csv(base/'greedy_path.csv',index=False)

summary_df=pd.DataFrame(summary)
summary_df.to_csv(DKL_OUT/'all_trace_heldout_summary.csv',index=False)
print('\n=== HELD-OUT SUMMARY ===\n',summary_df.to_string(index=False))
print('Saved outputs:',DKL_OUT.resolve())
